In [ ]:
# 1.1  Requirements check. Missing analysis packages are installed into THIS kernel when AUTO_INSTALL is True.
#      PyTorch is never installed automatically (it needs the CUDA build that matches your driver, see pytorch.org).
import importlib
import importlib.util
import subprocess
import sys

AUTO_INSTALL = True
REQUIRED = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "statsmodels": "statsmodels",
    "tqdm": "tqdm",
    "PIL": "pillow",
}
OPTIONAL = {
    "torch": "training, embeddings, Grad-CAM, complexity benchmark",
    "timm": "model definitions (DenseNet121, ConvNeXt-T, ViT-S)",
    "pydicom": "only to rebuild the VinBigData cache from DICOM",
    "jinja2": "LaTeX export of tables",
}

missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing and AUTO_INSTALL:
    print("installing:", " ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
    importlib.invalidate_caches()
still = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]
assert not still, f"install these packages first: {still}"
for mod, why in OPTIONAL.items():
    print(f"{'found  ' if importlib.util.find_spec(mod) else 'MISSING'} {mod:<8} ({why})")

In [ ]:
# 1.2  Imports and the single configuration object. EDIT ONLY THE `Config` DEFAULTS (or set the CXR_* environment
#      variables).
import contextlib
import datetime
import gc
import glob
import hashlib
import itertools
import json
import logging
import math
import os
import platform
import random
import re
import shutil
import sys
import time
import warnings
import zlib
from dataclasses import dataclass, field, asdict, replace
from pathlib import Path
from collections import OrderedDict
from typing import Optional

import numpy as np
import pandas as pd
import scipy
from scipy import stats
from scipy.special import expit
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.isotonic import IsotonicRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestRegressor
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, confusion_matrix, balanced_accuracy_score
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, GroupShuffleSplit
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# expected numerical warnings (empty-slice means for absent findings) and library deprecation notices are silenced
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="IProgress not found")

try:
    from tqdm.auto import tqdm
except ImportError:  # tqdm is optional: plain iteration without a progress bar

    def tqdm(it, **kw):
        return it


try:
    import statsmodels.formula.api as smf
    from statsmodels.stats.anova import AnovaRM, anova_lm

    HAS_SM = True
except ImportError:
    HAS_SM = False
try:
    import timm
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.precision", 4)

# "full" = numbers for the paper | "fast" = quick look | "smoke" = pipeline test
PROFILE = os.environ.get("CXR_PROFILE", "full").lower()


def _env_flag(name: str, default: bool) -> bool:
    v = os.environ.get(name)
    return default if v is None else v.strip().lower() in ("1", "true", "yes", "on")


@dataclass
class Config:
    # ---------------- locations ----------------
    # nih_split.csv, chexpert_final_clean.csv, 224-px caches
    data_root: Path = Path(os.environ.get("CXR_DATA_ROOT", "G:/"))
    # every earlier run (fair_ens, vit, convnext_dg, dg, controls, stage3, ...)
    runs: Path = Path(os.environ.get("CXR_RUNS", "G:/runs"))
    out_name: str = os.environ.get("CXR_OUT_NAME", "master_q1")  # everything this notebook writes goes to runs/out_name
    # ---------------- experimental grid (identical to the earlier runs) ----------------
    seeds: tuple = (0, 1, 2)
    epochs: int = 7
    batch_size: int = 64
    image_size: int = 224
    # ---------------- inference ----------------
    n_boot: int = 2000  # paired cluster-bootstrap replicates shared by every model (paired design)
    boot_chunk: int = 100  # replicates processed per vectorised block (memory/speed trade-off)
    boot_seed: int = 20261012
    resample_seeds: bool = True  # two-level bootstrap: patients AND training seeds are resampled
    alpha: float = 0.05
    equivalence_margin: float = 0.01  # smallest AUROC difference of practical interest (pre-specified)
    secondary_boot_metrics: bool = True  # also bootstrap AUPRC and sensitivity at 90% specificity
    # ---------------- repeated K-fold cross-validation ----------------
    cv_k: int = 5
    cv_r: int = 10
    recal_budgets: tuple = (25, 50, 100, 250, 500, 1000, 2500)
    recal_cv_r: int = 3
    # ---------------- feature engineering and classical baselines ----------------
    n_feat_per_site: int = 6000  # test images per site used for the acquisition-shift analysis
    n_classical_train: int = 20000  # training images per source for the classical (non-deep) baselines
    classical_cv_r: int = 2
    classical_pca: int = 128
    # ---------------- switches ----------------
    run_training: bool = _env_flag("CXR_RUN_TRAINING", False)  # train configurations whose predictions are missing
    # optional repeated grouped K-fold retraining of the deep models
    run_kfold_retrain: bool = _env_flag("CXR_RUN_KFOLD", False)
    # embeddings, Grad-CAM, border masking (skipped without torch/checkpoints)
    run_gpu_analyses: bool = _env_flag("CXR_RUN_GPU", True)
    run_complexity_benchmark: bool = _env_flag("CXR_RUN_BENCH", True)
    pretrained: bool = _env_flag("CXR_PRETRAINED", True)  # ImageNet initialisation for any NEW training run
    cache_results: bool = True
    master_seed: int = 2026
    n_workers: int = 8

    def __post_init__(self):
        self.data_root, self.runs = Path(self.data_root), Path(self.runs)
        self.out = self.runs / self.out_name


CFG = Config()
if PROFILE == "fast":
    CFG = replace(CFG, n_boot=500, cv_r=3, recal_cv_r=2, n_feat_per_site=2000, n_classical_train=8000, classical_cv_r=1)
elif PROFILE == "smoke":
    CFG = replace(
        CFG,
        n_boot=120,
        boot_chunk=40,
        cv_k=3,
        cv_r=2,
        recal_cv_r=1,
        recal_budgets=(25, 100),
        n_feat_per_site=150,
        n_classical_train=300,
        classical_cv_r=1,
        classical_pca=16,
        epochs=1,
        batch_size=8,
        n_workers=2,
    )

OUT = CFG.out
TAB, FIG, CACHE, NEWRUNS, LOGS = OUT / "tables", OUT / "figures", OUT / "cache", OUT / "runs", OUT / "logs"
for d_ in (OUT, TAB, FIG, CACHE, NEWRUNS, LOGS):
    d_.mkdir(parents=True, exist_ok=True)

D, R = CFG.data_root, CFG.runs
P = dict(  # every input file of the study, in one place
    nih_split=D / "nih_split.csv",
    nih_clean=D / "nih_clean.csv",
    nih_u8=D / "nih_224.u8",
    nih_raw=D / "NIH",
    cx_csv=D / "chexpert_final_clean.csv",
    cx_meta=D / "chexpert_images" / "chexpert_all.csv",
    cx_trainval_u8=D / "cx_trainval_224.u8",
    cx_test_npy=D / "cx_test_224.npy",
    vb_raw=D / "VinBig" / "vinbigdata",
    vb_labels=R / "vinbig" / "vinbig_image_labels.csv",
    vb_u8=R / "vinbig" / "vb_train_224.u8",
    vb_progress=R / "vinbig" / "vb_train_progress.json",
    vb_split=R / "fair_ens" / "vinbig_split.csv",
    fair_ens=R / "fair_ens",
    vit=R / "vit",
    convnext_dg=R / "convnext_dg",
    dg=R / "dg",
    part34=R / "part34",
    nih_train=R / "nih_train",
    nih_eval=R / "nih_eval",
    controls=R / "controls",
    stage2=R / "stage2",
    stage3=R / "stage3",
    vinbig=R / "vinbig",
    c_source=R / "c_source",
    master_v1=R / "master",
    gdro_dn=R / "gdro_7ep_3seeds",  # DenseNet121 GroupDRO on A+C and B+C (separate notebook, normalised hardness)
)
print(f"profile = {PROFILE} | data root = {D} | runs = {R} | outputs -> {OUT}")
print(
    f"bootstrap B = {CFG.n_boot} | repeated CV = {CFG.cv_r} x {CFG.cv_k}-fold "
    f"| statsmodels: {HAS_SM} | torch/timm: {HAS_TORCH}"
)

In [ ]:
# 1.3  Reproducibility: global seeding, named random streams, environment record, section timers
def seed_everything(seed: int, deterministic: bool = False) -> None:
    """Seed Python, NumPy and PyTorch. deterministic=True also forces deterministic cuDNN kernels (slower)."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    if HAS_TORCH:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        if deterministic:
            os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
            torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = True, False
            torch.use_deterministic_algorithms(True, warn_only=True)


def rng_for(*keys) -> np.random.Generator:
    """Independent, reproducible random stream for any analysis, keyed by names (e.g. rng_for('bootstrap', 'A')).
    Every analysis draws from its own stream, so adding or removing one analysis never changes the numbers of another.
    """
    ints = [CFG.master_seed] + [k if isinstance(k, (int, np.integer)) else zlib.crc32(str(k).encode()) for k in keys]
    return np.random.default_rng([int(i) for i in ints])


def environment_report() -> dict:
    rep = dict(
        timestamp=datetime.datetime.now().isoformat(timespec="seconds"),
        profile=PROFILE,
        python=sys.version.split()[0],
        platform=platform.platform(),
        processor=platform.processor(),
        cpu_count=os.cpu_count(),
        numpy=np.__version__,
        pandas=pd.__version__,
        scipy=scipy.__version__,
        sklearn=sklearn.__version__,
        matplotlib=mpl.__version__,
        seaborn=sns.__version__,
    )
    if HAS_SM:
        import statsmodels

        rep["statsmodels"] = statsmodels.__version__
    if HAS_TORCH:
        rep.update(
            torch=torch.__version__,
            timm=timm.__version__,
            cuda=torch.version.cuda,
            cudnn=torch.backends.cudnn.version(),
            gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
        )
    return rep


TIMER = OrderedDict()


@contextlib.contextmanager
def timed(name: str):
    """Wall-clock time of every analysis block; reported in the computational-cost section."""
    t0 = time.perf_counter()
    try:
        yield
    finally:
        TIMER[name] = TIMER.get(name, 0.0) + time.perf_counter() - t0


def file_digest(path, n_bytes: int = 1 << 20) -> str:
    """Short content hash (first MiB + size) used to record exactly which artifact produced each number."""
    path = Path(path)
    h = hashlib.sha1(str(path.stat().st_size).encode())
    with open(path, "rb") as fh:
        h.update(fh.read(n_bytes))
    return h.hexdigest()[:12]


def write_json(obj, path, **kw) -> None:
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(obj, fh, indent=2, **kw)


def read_json(path):
    with open(path, encoding="utf-8") as fh:
        return json.load(fh)


seed_everything(CFG.master_seed)
ENV = environment_report()
write_json(
    dict(environment=ENV, config={k: str(v) for k, v in asdict(CFG).items()}), LOGS / "environment_and_config.json"
)
display(pd.Series(ENV, name="value").to_frame())

In [ ]:
# 1.4  Publication style (IEEE two-column geometry, embedded TrueType fonts, 600-dpi raster + vector PDF) and table
#      export
sns.set_theme(context="paper", style="ticks", palette="colorblind")
mpl.rcParams.update(
    {
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "Nimbus Roman", "DejaVu Serif"],
        "mathtext.fontset": "stix",
        "font.size": 8,
        "axes.labelsize": 8,
        "axes.titlesize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
        "legend.title_fontsize": 7,
        "axes.linewidth": 0.6,
        "lines.linewidth": 1.0,
        "grid.linewidth": 0.4,
        "xtick.major.width": 0.6,
        "ytick.major.width": 0.6,
        "figure.dpi": 150,
        "savefig.dpi": 600,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "legend.frameon": False,
        "axes.titleweight": "regular",
    }
)
COL1, COL2 = 3.5, 7.16  # single / double column width in inches
SHOW_FIGURES = os.environ.get("CXR_SHOW_FIGURES", "1") == "1"
FIGURE_INDEX, TABLE_INDEX = [], []


def save_fig(fig, name: str, caption: str = "") -> None:
    """Vector PDF + 600-dpi PNG; every figure is listed in the figure index with its caption."""
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")
    fig.savefig(FIG / f"{name}.png", dpi=600, bbox_inches="tight")
    FIGURE_INDEX.append(dict(figure=name, caption=caption))
    if SHOW_FIGURES:
        plt.show()
    plt.close(fig)


def save_tab(
    df: pd.DataFrame, name: str, caption: str = "", index: bool = True, show: bool = True, floatfmt: str = "%.4f"
) -> pd.DataFrame:
    """CSV (full precision) + LaTeX (booktabs) copy of a table; listed in the table index."""
    df.to_csv(TAB / f"{name}.csv", index=index)
    try:
        df.to_latex(
            TAB / f"{name}.tex",
            index=index,
            float_format=lambda x: floatfmt % x,
            escape=True,
            caption=caption or None,
            label=f"tab:{name}",
        )
    except Exception:
        pass
    TABLE_INDEX.append(dict(table=name, caption=caption, rows=len(df)))
    if show:
        display(df)
    return df


def fmt_ci(est, lo, hi, d: int = 3) -> str:
    return f"{est:.{d}f} [{lo:.{d}f}, {hi:.{d}f}]"


def fmt_pm(x, d: int = 3) -> str:
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return "n/a"
    return f"{x.mean():.{d}f} ± {x.std(ddof=1):.{d}f}" if len(x) > 1 else f"{x.mean():.{d}f}"


def panel_label(ax, s: str) -> None:
    ax.text(-0.14, 1.04, s, transform=ax.transAxes, fontsize=9, fontweight="bold", va="bottom", ha="left")

In [ ]:
# 1.5  Core metric library (pure NumPy; every function is unit-tested against scikit-learn in the next cell)
def sigmoid(z):
    return expit(np.asarray(z, np.float64))


def logit(p, eps: float = 1e-6):
    p = np.clip(np.asarray(p, np.float64), eps, 1 - eps)
    return np.log(p / (1 - p))


def auroc(y, s) -> float:
    """Mann-Whitney AUROC with mid-ranks for ties (identical to sklearn.roc_auc_score, about 5x faster)."""
    y = np.asarray(y).astype(bool)
    n1 = int(y.sum())
    n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return np.nan
    r = stats.rankdata(s)
    return float((r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


def sens_at_spec(y, s, spec: float = 0.90) -> float:
    """Highest sensitivity among thresholds whose specificity is >= spec (operating point used clinically)."""
    fpr, tpr, _ = roc_curve(y, s, drop_intermediate=False)
    ok = fpr <= 1 - spec + 1e-12
    return float(tpr[ok].max()) if ok.any() else 0.0


THR_GRID = np.linspace(0.02, 0.95, 94)  # identical grid to every earlier notebook


def best_f1_threshold(y, p, grid=THR_GRID) -> float:
    """Threshold maximising F1 on (y, p); used ONLY on source-hospital data, never on the target."""
    y1 = (np.asarray(y) == 1)[:, None]
    pr = np.asarray(p)[:, None] >= grid[None, :]
    tp = (pr & y1).sum(0)
    fp = pr.sum(0) - tp
    fn = y1.sum() - tp
    return float(grid[np.argmax(2 * tp / np.maximum(2 * tp + fp + fn, 1))])


def youden_threshold(y, p) -> float:
    fpr, tpr, thr = roc_curve(y, p)
    return float(np.clip(thr[np.argmax(tpr - fpr)], 0, 1))


def ece_score(y, p, n_bins: int = 10, strategy: str = "uniform") -> float:
    """Expected calibration error. strategy='uniform' (equal-width, Guo et al. 2017) or 'quantile' (equal-mass, Nixon
    et al. 2019)."""
    y = np.asarray(y, float)
    p = np.asarray(p, float)
    if strategy == "uniform":
        b = np.clip((p * n_bins).astype(int), 0, n_bins - 1)
    else:
        edges = np.unique(np.quantile(p, np.linspace(0, 1, n_bins + 1)))
        b = np.clip(np.searchsorted(edges, p, side="right") - 1, 0, max(len(edges) - 2, 0))
    sp = np.bincount(b, weights=p, minlength=n_bins)
    sy = np.bincount(b, weights=y, minlength=n_bins)
    return float(np.abs(sp - sy).sum() / len(p))


def logistic_recalibration(y, p, fit_slope: bool = True, iters: int = 100):
    """Fits logit P(y=1) = a + b * logit(p) by damped Newton-Raphson. fit_slope=False fixes b = 1 (offset), so `a` is
    the calibration intercept / calibration-in-the-large (Van Calster et al. 2019). fit_slope=True returns the
    calibration slope b (b < 1: predictions too extreme; b > 1: too modest). Returns NaN when the estimate does not
    exist: (near-)constant predictions, a single outcome class, or (quasi-)separation."""
    z = logit(p)
    y = np.asarray(y, float)
    a, b = 0.0, 1.0
    if len(y) == 0 or y.min() == y.max() or (fit_slope and np.std(z) < 1e-6):
        return np.nan, np.nan
    converged = False
    for _ in range(iters):
        q = expit(a + b * z)
        w = q * (1 - q) + 1e-12
        if fit_slope:
            X = np.c_[np.ones_like(z), z]
            step = np.linalg.solve((X * w[:, None]).T @ X + 1e-9 * np.eye(2), X.T @ (y - q))
        else:
            step = np.array([(y - q).sum() / w.sum()])
        step = step * min(1.0, 5.0 / max(np.abs(step).max(), 1e-12))  # damping: no step larger than 5 logit units
        a += step[0]
        if fit_slope:
            b += step[1]
        if np.abs(step).max() < 1e-9:
            converged = True
            break
    if not converged or not (np.isfinite(a) and np.isfinite(b)) or abs(a) > 50 or abs(b) > 50:
        return np.nan, np.nan
    return float(a), float(b)


def binary_report(y, p, thr: float) -> dict:
    """Threshold-dependent metrics at a fixed operating point (thr chosen on source data)."""
    y = np.asarray(y).astype(int)
    yhat = (np.asarray(p) >= thr).astype(int)
    tp = int(((yhat == 1) & (y == 1)).sum())
    fp = int(((yhat == 1) & (y == 0)).sum())
    fn = int(((yhat == 0) & (y == 1)).sum())
    tn = int(((yhat == 0) & (y == 0)).sum())
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    spec = tn / (tn + fp) if tn + fp else 0.0
    f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0
    den = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return dict(
        tp=tp,
        fp=fp,
        fn=fn,
        tn=tn,
        precision=prec,
        recall=rec,
        specificity=spec,
        f1=f1,
        accuracy=(tp + tn) / len(y),
        balanced_accuracy=(rec + spec) / 2,
        mcc=((tp * tn - fp * fn) / den) if den else 0.0,
        positive_rate=float(yhat.mean()),
    )


def probability_metrics(y, p) -> dict:
    """Threshold-free metrics: discrimination, probabilistic accuracy (regression-type errors) and calibration."""
    y = np.asarray(y, float)
    p = np.clip(np.asarray(p, float), 1e-7, 1 - 1e-7)
    e = p - y
    a_int, _ = logistic_recalibration(y, p, fit_slope=False)
    _, slope = logistic_recalibration(y, p, fit_slope=True)
    prev = y.mean()
    brier = float(np.mean(e**2))
    brier_ref = prev * (1 - prev)
    return dict(
        auroc=auroc(y, p),
        auprc=float(average_precision_score(y, p)),
        sens_at_90spec=sens_at_spec(y, p),
        brier=brier,
        brier_skill=1 - brier / brier_ref if brier_ref > 0 else np.nan,
        rmse=float(np.sqrt(brier)),
        mae=float(np.mean(np.abs(e))),
        log_loss=float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))),
        ece=ece_score(y, p),
        ace=ece_score(y, p, strategy="quantile"),
        cal_intercept=a_int,
        cal_slope=slope,
        citl=float(p.mean() - prev),
        prevalence=float(prev),
        n=int(len(y)),
        n_pos=int(y.sum()),
    )


def holm(p) -> np.ndarray:
    """Holm (1979) step-down family-wise error correction."""
    p = np.asarray(p, float)
    out = np.full(len(p), np.nan)
    ok = np.isfinite(p)
    q = p[ok]
    o = np.argsort(q)
    m = len(q)
    adj = np.empty(m)
    run = 0.0
    for r_, i in enumerate(o):
        run = max(run, min(1.0, (m - r_) * q[i]))
        adj[i] = run
    out[ok] = adj
    return out


def bh_fdr(p) -> np.ndarray:
    """Benjamini-Hochberg (1995) false-discovery-rate adjusted p-values (q-values)."""
    p = np.asarray(p, float)
    out = np.full(len(p), np.nan)
    ok = np.isfinite(p)
    q = p[ok]
    m = len(q)
    o = np.argsort(q)
    r_ = np.arange(1, m + 1)
    adj = np.minimum.accumulate((q[o] * m / r_)[::-1])[::-1]
    res = np.empty(m)
    res[o] = np.minimum(adj, 1.0)
    out[ok] = res
    return out

In [ ]:
# 1.6  Paired, two-level cluster-bootstrap engine
#   Level 1: patients are resampled with replacement (images of one patient move together; VinBigData has no patient
#            id, so each image is its own cluster). The SAME replicate weights are shared by every model on a test site
#   -> all contrasts are paired. Level 2: training seeds are resampled with replacement inside every replicate, so
#   intervals cover training randomness too. AUROC, AUPRC and sensitivity@90%specificity are computed exactly (ties get
#   half credit) from integer weights; a score vector is sorted once and each block of replicates is evaluated with
#   vectorised group sums.


class ClusterBootstrap:
    def __init__(self, site_groups: dict, n_boot: int, chunk: int):
        self.B, self.chunk, self.W, self._seed_idx = n_boot, chunk, {}, {}
        for site, g in site_groups.items():
            g = np.asarray(g, np.int64)
            G = int(g.max()) + 1
            rng = rng_for("cluster-bootstrap", CFG.boot_seed, site)
            cnt = np.empty((n_boot, G), np.uint8)
            for i in range(0, n_boot, 250):
                n = min(250, n_boot - i)
                cnt[i : i + n] = np.minimum(rng.multinomial(G, np.full(G, 1.0 / G), size=n), 255)
            self.W[site] = cnt[:, g]  # (B, N): weight of an image = draws of its patient

    def seed_index(self, S: int) -> np.ndarray:
        if S not in self._seed_idx:
            rng = rng_for("seed-resampling", CFG.boot_seed, S)
            self._seed_idx[S] = (
                rng.integers(0, S, (self.B, S))
                if (CFG.resample_seeds and S > 1)
                else np.tile(np.arange(S), (self.B, 1))
            )
        return self._seed_idx[S]


class ScoreIndex:
    """Sorted view of one score vector: order, labels in that order and tie-group boundaries."""

    __slots__ = ("order", "ys", "starts", "tied")

    def __init__(self, y, s):
        s = np.asarray(s, np.float64)
        order = np.argsort(s, kind="mergesort")
        ss = s[order]
        self.order = order
        self.ys = np.asarray(y, np.float32)[order]
        self.starts = np.flatnonzero(np.r_[True, ss[1:] != ss[:-1]])
        self.tied = len(self.starts) < len(ss)

    def group_sums(self, W):
        Ws = W[:, self.order]
        pos = Ws * self.ys
        neg = Ws - pos
        if self.tied:
            pos = np.add.reduceat(pos, self.starts, axis=1)
            neg = np.add.reduceat(neg, self.starts, axis=1)
        return pos.astype(np.float64), neg.astype(np.float64)


def weighted_rank_metrics(pos, neg, which=("auroc",), spec: float = 0.90) -> dict:
    """pos/neg: (b, G) weight of positives/negatives per tie group, groups in ascending score order."""
    P = pos.sum(1)
    Nn = neg.sum(1)
    cn = np.cumsum(neg, 1) - neg
    out = {}
    with np.errstate(invalid="ignore", divide="ignore"):
        if "auroc" in which:
            out["auroc"] = (np.einsum("bg,bg->b", pos, cn) + 0.5 * np.einsum("bg,bg->b", pos, neg)) / (P * Nn)
        if "auprc" in which or "sens90" in which:
            tp = P[:, None] - (np.cumsum(pos, 1) - pos)  # positive weight scored >= the group
            if "auprc" in which:
                out["auprc"] = (pos * (tp / np.maximum(tp + (Nn[:, None] - cn), 1e-12))).sum(1) / P
            if "sens90" in which:
                ok = cn >= spec * Nn[:, None] - 1e-9
                g0 = ok.argmax(1)
                out["sens90"] = np.where(ok.any(1), tp[np.arange(len(g0)), g0] / P, 0.0)
    return out


@dataclass
class BootResult:
    key: tuple
    site: str
    seeds: list
    findings: list  # finding indices that were evaluated
    point: dict  # metric -> (S, L) per-seed point estimates (NaN where not evaluated)
    boot: dict  # metric -> (B, S, L) float32 replicates


def _boot_cache_path(key, site, metrics, probs_by_seed, mask, tag):
    h = hashlib.sha1(repr((key, site, tuple(metrics), CFG.n_boot, CFG.boot_seed, tag)).encode())
    for s in sorted(probs_by_seed):
        a = np.ascontiguousarray(probs_by_seed[s], dtype=np.float32)
        h.update(a[:: max(1, len(a) // 4096)].tobytes())
        h.update(str(a.shape).encode())
    if mask is not None:
        h.update(np.packbits(np.asarray(mask, bool)).tobytes())
    return CACHE / "boot" / f"{h.hexdigest()[:20]}.npz"


def bootstrap_cell(
    key, site: str, probs_by_seed: dict, metrics=("auroc",), mask=None, min_events: int = 5, tag: str = "", allowed=None
) -> BootResult:
    """Point estimates and paired cluster-bootstrap replicates for one model family on one test site.
    probs_by_seed: {seed: (N, L) probabilities}. mask: optional boolean (N,) subgroup selector (weights outside are 0).
    allowed: finding indices the model was trained on (e.g. a VinBigData-only model has no Edema output)."""
    Y = YT[site]
    valid = VALID[site]
    seeds = sorted(probs_by_seed)
    S = len(seeds)
    B = BOOT.B
    N = len(Y)
    m_ = np.ones(N, bool) if mask is None else np.asarray(mask, bool)
    allowed = range(L) if allowed is None else allowed
    js = [
        j
        for j in range(L)
        if valid[j]
        and j in allowed
        and min(Y[m_, j].sum(), m_.sum() - Y[m_, j].sum()) >= min_events
        and all(np.isfinite(probs_by_seed[s][:, j]).all() for s in seeds)
    ]
    cpath = _boot_cache_path(key, site, metrics, probs_by_seed, mask, tag + str(list(js)))
    if CFG.cache_results and cpath.exists():
        z = np.load(cpath, allow_pickle=False)
        return BootResult(key, site, seeds, js, {m: z[f"pt_{m}"] for m in metrics}, {m: z[f"bs_{m}"] for m in metrics})
    idx = {(si, j): ScoreIndex(Y[:, j], probs_by_seed[s][:, j]) for si, s in enumerate(seeds) for j in js}
    point = {m: np.full((S, L), np.nan) for m in metrics}
    boot = {m: np.full((B, S, L), np.nan, np.float32) for m in metrics}
    w1 = m_.astype(np.float32)[None, :]
    for (si, j), ix in idx.items():
        r = weighted_rank_metrics(*ix.group_sums(w1), which=metrics)
        for m in metrics:
            point[m][si, j] = r[m][0]
    for i in range(0, B, BOOT.chunk):
        W = BOOT.W[site][i : i + BOOT.chunk].astype(np.float32)
        if mask is not None:
            W *= w1
        for (si, j), ix in idx.items():
            r = weighted_rank_metrics(*ix.group_sums(W), which=metrics)
            for m in metrics:
                boot[m][i : i + BOOT.chunk, si, j] = r[m]
    if CFG.cache_results:
        cpath.parent.mkdir(parents=True, exist_ok=True)
        np.savez(cpath, **{f"pt_{m}": point[m] for m in metrics}, **{f"bs_{m}": boot[m] for m in metrics})
    return BootResult(key, site, seeds, js, point, boot)


def macro_draws(res: BootResult, metric: str = "auroc", findings=None) -> np.ndarray:
    """(B,) bootstrap distribution of the macro metric: mean over findings of the mean over (resampled) seeds."""
    f = res.findings if findings is None else [j for j in findings if j in res.findings]
    A = res.boot[metric][:, :, f]
    if A.shape[1] > 1:
        A = np.take_along_axis(A, BOOT.seed_index(A.shape[1])[:, :, None], axis=1)
    return np.nanmean(np.nanmean(A, axis=1), axis=1)


def macro_point(res: BootResult, metric: str = "auroc", findings=None) -> float:
    f = res.findings if findings is None else [j for j in findings if j in res.findings]
    return float(np.nanmean(np.nanmean(res.point[metric][:, f], axis=0)))


def decide(est: float, p: float, p_equiv: float, delta: float = None) -> str:
    """Verdict for a contrast X - Y from a (multiplicity-adjusted) two-sided p and a TOST equivalence p at margin +/-
    delta. 'positive'/'negative' = X - Y differs from 0; the bracket says whether the estimate also reaches the
    margin; 'equivalent' = both one-sided tests reject |X - Y| >= delta; otherwise 'inconclusive' (absence of
    evidence, not of effect)."""
    delta = CFG.equivalence_margin if delta is None else delta
    if not np.isfinite(p):
        return "not estimable"
    if p < CFG.alpha:
        return f"{'positive' if est > 0 else 'negative'} ({'≥' if abs(est) >= delta else '<'} margin)"
    if np.isfinite(p_equiv) and p_equiv < CFG.alpha:
        return "equivalent (TOST)"
    return "inconclusive"


def summarize_draws(est: float, d: np.ndarray, delta: float = None) -> dict:
    """Effect estimate, 95% percentile CI, bootstrap-SE Wald p (as pROC's bootstrap test), percentile p, TOST
    equivalence p at +/- delta, probability of superiority, minimal detectable effect, and an UNADJUSTED verdict
    (finalize_family replaces it with the Holm-adjusted verdict)."""
    delta = CFG.equivalence_margin if delta is None else delta
    d = np.asarray(d, float)
    d = d[np.isfinite(d)]
    B_ = len(d)
    se = float(d.std(ddof=1)) if B_ > 1 else np.nan
    lo, hi = np.percentile(d, [2.5, 97.5]) if B_ else (np.nan, np.nan)
    if np.isfinite(se) and se > 1e-12:
        z = est / se
        p_wald = float(2 * stats.norm.sf(abs(z)))
        p_eq = float(max(stats.norm.sf((est + delta) / se), stats.norm.sf((delta - est) / se)))
    else:  # no sampling variation: a test is not defined
        z = p_wald = p_eq = np.nan
    p_boot = (
        float(min(1.0, 2 * min((1 + (d <= 0).sum()) / (B_ + 1), (1 + (d >= 0).sum()) / (B_ + 1)))) if B_ else np.nan
    )
    return dict(
        estimate=float(est),
        se=se,
        ci_low=float(lo),
        ci_high=float(hi),
        z=float(z),
        p=p_wald,
        p_boot=p_boot,
        p_equiv=p_eq,
        prob_superiority=float((d > 0).mean()) if B_ else np.nan,
        mde_80=float(2.8 * se),
        verdict=decide(est, p_wald, p_eq, delta),
    )


def contrast(rA: BootResult, rB: BootResult, metric: str = "auroc", findings=None, **info) -> dict:
    """Paired contrast A - B on the macro metric over the findings both families evaluate."""
    f = sorted(set(rA.findings) & set(rB.findings) & set(range(L) if findings is None else findings))
    est = macro_point(rA, metric, f) - macro_point(rB, metric, f)
    return dict(
        **info, n_findings=len(f), **summarize_draws(est, macro_draws(rA, metric, f) - macro_draws(rB, metric, f))
    )


def finalize_family(rows: list, family_col: str = "family") -> pd.DataFrame:
    """Holm within each pre-specified family (difference tests and TOST equivalence tests separately), Benjamini-
    Hochberg across everything in the table; the verdict is recomputed from the Holm-adjusted p-values."""
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df["p_holm"] = df.groupby(family_col)["p"].transform(lambda x: holm(x.values))
    df["p_equiv_holm"] = (
        df.groupby(family_col)["p_equiv"].transform(lambda x: holm(x.values)) if "p_equiv" in df else np.nan
    )
    df["q_bh"] = bh_fdr(df["p"].values)
    df["significant_holm"] = df["p_holm"] < CFG.alpha
    df["verdict"] = [decide(e, p_, q_) for e, p_, q_ in zip(df.estimate, df.p_holm, df.p_equiv_holm)]
    return df

In [ ]:
# 1.7  Hypothesis tests and effect sizes
# ---------- DeLong (independent images) and Obuchowski (images clustered within patients) tests for correlated AUROCs
# ----------
def _structural_components(y, scores):
    """Per-unit DeLong placement values (Sun & Xu 2014 fast algorithm). scores: (k, N)."""
    y = np.asarray(y).astype(bool)
    X = scores[:, y]
    Yn = scores[:, ~y]
    m, n = X.shape[1], Yn.shape[1]
    tx = stats.rankdata(X, axis=1)
    ty = stats.rankdata(Yn, axis=1)
    tz = stats.rankdata(np.concatenate([X, Yn], 1), axis=1)
    aucs = (tz[:, :m].sum(1) - m * (m + 1) / 2) / (m * n)
    v10 = (tz[:, :m] - tx) / n  # each positive: share of negatives scored below it (ties 1/2)
    v01 = 1.0 - (tz[:, m:] - ty) / m  # each negative: share of positives scored above it
    return aucs, v10, v01, m, n


def delong_test(y, s1, s2, clusters=None) -> dict:
    """Paired test of AUROC(s1) - AUROC(s2) on the same images.
    clusters=None: DeLong et al. (1988). clusters given: Obuchowski (1997) clustered-data variance (patients with
    several images).
    """
    y = np.asarray(y).astype(bool)
    aucs, v10, v01, m, n = _structural_components(y, np.vstack([s1, s2]).astype(np.float64))
    if clusters is None:
        S = np.cov(v10) / m + np.cov(v01) / n
    else:
        c = pd.factorize(np.asarray(clusters))[0]
        n_cl = c.max() + 1
        cp, cn_ = c[y], c[~y]
        Xc = np.stack([np.bincount(cp, weights=v10[k], minlength=n_cl) for k in range(2)])
        mi = np.bincount(cp, minlength=n_cl)
        Yc = np.stack([np.bincount(cn_, weights=v01[k], minlength=n_cl) for k in range(2)])
        ni = np.bincount(cn_, minlength=n_cl)
        dX = Xc - mi[None] * aucs[:, None]
        dY = Yc - ni[None] * aucs[:, None]
        I10, I01 = int((mi > 0).sum()), int((ni > 0).sum())
        S10 = I10 / ((I10 - 1) * m) * (dX @ dX.T)
        S01 = I01 / ((I01 - 1) * n) * (dY @ dY.T)
        S11 = n_cl / (n_cl - 1) * (dX @ dY.T)
        S = S10 / m + S01 / n + (S11 + S11.T) / (m * n)
    var = S[0, 0] + S[1, 1] - 2 * S[0, 1]
    diff = aucs[0] - aucs[1]
    z = diff / np.sqrt(var) if var > 0 else np.nan
    return dict(
        auc1=float(aucs[0]),
        auc2=float(aucs[1]),
        diff=float(diff),
        se=float(np.sqrt(max(var, 0))),
        z=float(z),
        p=float(2 * stats.norm.sf(abs(z))) if np.isfinite(z) else np.nan,
    )


# ---------- paired comparisons of two models over many tasks ----------
def paired_effects(x, y) -> dict:
    """x, y: paired scores over tasks. Paired t-test (Cohen's d_z, Hedges' g_av), Wilcoxon signed-rank (matched-pairs
    rank-biserial r, Kerby 2014), Shapiro-Wilk on the differences (guides parametric vs non-parametric reading)."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    d = x - y
    n = len(d)
    t, p_t = stats.ttest_rel(x, y)
    sd = d.std(ddof=1)
    dz = d.mean() / sd if sd > 0 else np.nan
    sav = (x.std(ddof=1) + y.std(ddof=1)) / 2
    gav = (d.mean() / sav) * (1 - 3 / (4 * (n - 1) - 1)) if sav > 0 and n > 2 else np.nan
    nz = d[d != 0]
    if len(nz) >= 1:
        w = stats.wilcoxon(nz, zero_method="wilcox")
        rk = stats.rankdata(np.abs(nz))
        rbc = (rk[nz > 0].sum() - rk[nz < 0].sum()) / rk.sum()
        w_stat, p_w = float(w.statistic), float(w.pvalue)
    else:
        w_stat, p_w, rbc = np.nan, 1.0, 0.0
    p_sw = float(stats.shapiro(d).pvalue) if 3 <= n <= 5000 else np.nan
    return dict(
        n_tasks=n,
        mean_diff=float(d.mean()),
        median_diff=float(np.median(d)),
        t=float(t),
        p_ttest=float(p_t),
        cohens_dz=float(dz),
        hedges_gav=float(gav),
        wilcoxon_W=w_stat,
        p_wilcoxon=p_w,
        rank_biserial=float(rbc),
        shapiro_p_diff=p_sw,
        wins=int((d > 0).sum()),
        losses=int((d < 0).sum()),
        ties=int((d == 0).sum()),
    )


def friedman_nemenyi(scores: pd.DataFrame, alpha: float = 0.05) -> dict:
    """scores: rows = tasks (blocks), columns = methods, higher = better (Demsar 2006).
    Friedman chi2, Iman-Davenport F, Kendall's W, mean ranks (1 = best) and the Nemenyi critical difference."""
    data = scores.dropna()
    N, k = data.shape
    ranks = data.rank(axis=1, ascending=False)
    R = ranks.mean().sort_values()
    chi2, p = stats.friedmanchisquare(*[data[c].values for c in data.columns])
    ff = (N - 1) * chi2 / (N * (k - 1) - chi2) if N * (k - 1) - chi2 > 0 else np.inf
    q = stats.studentized_range.ppf(1 - alpha, k, np.inf) / np.sqrt(2)
    return dict(
        N=N,
        k=k,
        chi2=float(chi2),
        p_friedman=float(p),
        F_iman_davenport=float(ff),
        p_iman_davenport=float(stats.f.sf(ff, k - 1, (k - 1) * (N - 1))),
        kendalls_W=float(chi2 / (N * (k - 1))),
        mean_ranks=R,
        cd_nemenyi=float(q * np.sqrt(k * (k + 1) / (6 * N))),
    )


def pairwise_wilcoxon_holm(scores: pd.DataFrame) -> pd.DataFrame:
    data = scores.dropna()
    rows = []
    for a, b in itertools.combinations(data.columns, 2):
        rows.append(dict(A=a, B=b, **paired_effects(data[a], data[b])))
    out = pd.DataFrame(rows)
    out["p_wilcoxon_holm"] = holm(out.p_wilcoxon.values)
    out["p_ttest_holm"] = holm(out.p_ttest.values)
    return out


def rm_anova(scores: pd.DataFrame) -> dict:
    """One-way repeated-measures ANOVA (methods within tasks) with partial eta squared."""
    if not HAS_SM:
        return {}
    data = scores.dropna()
    long = (
        data.reset_index(drop=True)
        .rename_axis("task")
        .reset_index()
        .melt(id_vars="task", var_name="method", value_name="score")
    )
    res = AnovaRM(long, depvar="score", subject="task", within=["method"]).fit().anova_table.iloc[0]
    F_, d1, d2 = float(res["F Value"]), float(res["Num DF"]), float(res["Den DF"])
    return dict(F=F_, df1=d1, df2=d2, p=float(res["Pr > F"]), partial_eta2=F_ * d1 / (F_ * d1 + d2))


# ---------- independent groups ----------
def kruskal_effect(groups: dict) -> dict:
    """Kruskal-Wallis H with epsilon-squared effect size (Tomczak & Tomczak 2014)."""
    vals = [np.asarray(v, float)[np.isfinite(v)] for v in groups.values()]
    n = sum(len(v) for v in vals)
    H, p = stats.kruskal(*vals)
    return dict(H=float(H), p=float(p), epsilon2=float(H * (n + 1) / (n**2 - 1)), n=n, k=len(vals))


def dunn_holm(groups: dict) -> pd.DataFrame:
    """Dunn's post-hoc test (tie-corrected) with Holm adjustment."""
    names = list(groups)
    vals = [np.asarray(groups[k], float) for k in names]
    allv = np.concatenate(vals)
    n = len(allv)
    rk = stats.rankdata(allv)
    _, tie_counts = np.unique(allv, return_counts=True)
    tie = (tie_counts**3 - tie_counts).sum() / (12 * (n - 1))
    sizes = [len(v) for v in vals]
    idx = np.cumsum([0] + sizes)
    mr = [rk[idx[i] : idx[i + 1]].mean() for i in range(len(vals))]
    rows = []
    for i, j in itertools.combinations(range(len(vals)), 2):
        se = np.sqrt((n * (n + 1) / 12 - tie) * (1 / sizes[i] + 1 / sizes[j]))
        z = (mr[i] - mr[j]) / se
        rows.append(
            dict(
                A=names[i],
                B=names[j],
                z=float(z),
                p=float(2 * stats.norm.sf(abs(z))),
                cliffs_delta=cliffs_delta(vals[i], vals[j]),
            )
        )
    out = pd.DataFrame(rows)
    out["p_holm"] = holm(out.p.values)
    return out


def cliffs_delta(x, y) -> float:
    """Cliff's delta = P(X > Y) - P(X < Y), from the Mann-Whitney U statistic."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    if len(x) == 0 or len(y) == 0:
        return np.nan
    U = stats.mannwhitneyu(x, y, alternative="two-sided").statistic
    return float(2 * U / (len(x) * len(y)) - 1)


def anova_effects(table: pd.DataFrame) -> pd.DataFrame:
    """Adds partial eta squared and omega squared to a statsmodels anova_lm table."""
    t = table.copy()
    ss_res = float(t.loc["Residual", "sum_sq"])
    df_res = float(t.loc["Residual", "df"])
    ms_res = ss_res / df_res
    ss_tot = float(t["sum_sq"].sum())
    t["partial_eta2"] = t["sum_sq"] / (t["sum_sq"] + ss_res)
    t["omega2"] = (t["sum_sq"] - t["df"] * ms_res) / (ss_tot + ms_res)
    t.loc["Residual", ["partial_eta2", "omega2"]] = np.nan
    return t


def cd_diagram(ax, mean_ranks: pd.Series, cd: float, title: str = "") -> None:
    """Critical-difference diagram (Demsar 2006): methods whose mean ranks differ by less than CD are joined by a
    bar."""
    R = mean_ranks.sort_values()
    k = len(R)
    lo, hi = 1, k
    ax.set_xlim(lo - 0.3, hi + 0.3)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.plot([lo, hi], [0.8, 0.8], color="k", lw=0.8)
    for t_ in range(lo, hi + 1):
        ax.plot([t_, t_], [0.8, 0.83], color="k", lw=0.8)
        ax.text(t_, 0.86, str(t_), ha="center", va="bottom", fontsize=7)
    half = (k + 1) // 2
    for i, (name, r) in enumerate(R.items()):
        left = i < half
        y = 0.62 - 0.11 * (i if left else k - 1 - i)
        xt = lo - 0.25 if left else hi + 0.25
        ax.plot([r, r, xt], [0.8, y, y], color="k", lw=0.6)
        ax.text(
            xt + (-0.03 if left else 0.03),
            y,
            f"{name} ({r:.2f})",
            ha="right" if left else "left",
            va="center",
            fontsize=6.5,
        )
    vals = R.values
    cliques = []
    for i in range(k):
        j = i
        while j + 1 < k and vals[j + 1] - vals[i] < cd:
            j += 1
        if j > i and not any(a <= i and j <= b for a, b in cliques):
            cliques.append((i, j))
    for c_, (i, j) in enumerate(cliques):
        yb = 0.74 - 0.035 * c_
        ax.plot([vals[i] - 0.03, vals[j] + 0.03], [yb, yb], color="k", lw=2.0, solid_capstyle="round")
    ax.plot([lo, lo + cd], [0.97, 0.97], color="k", lw=1.0)
    ax.text(lo + cd / 2, 0.99, f"CD = {cd:.2f}", ha="center", va="bottom", fontsize=6.5)
    ax.set_ylim(0.62 - 0.11 * (half - 1) - 0.12, 1.0)  # trim empty space below the lowest label
    if title:
        ax.set_title(title, fontsize=6.5, pad=10)

In [ ]:
# 1.8  Repeated K-fold cross-validation utilities
#   Folds are grouped by patient (no patient in both the fitting and the held-out part) and stratified on the outcome.
#   Repeated K-fold estimates are correlated (overlapping training sets), so intervals and paired tests use the
#   Nadeau & Bengio (2003) corrected resampled t statistic: Var_corrected = (1/J + n_test/n_train) * s^2, J = K x R.
def repeated_group_kfold(y_strat, groups, k: int, r: int, key: str):
    """Yields (repeat, fold, fit_idx, held_out_idx). Falls back to plain stratified folds if a class has fewer than k
    groups."""
    y_strat = np.asarray(y_strat)
    groups = np.asarray(groups)
    for rep in range(r):
        rs = int(rng_for("cv", key, rep).integers(0, 2**31 - 1))
        try:
            splitter = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=rs)
            splits = list(splitter.split(np.zeros(len(y_strat)), y_strat, groups))
        except ValueError:
            splitter = StratifiedKFold(n_splits=k, shuffle=True, random_state=rs)
            splits = list(splitter.split(np.zeros(len(y_strat)), y_strat))
        for fold, (a, b) in enumerate(splits):
            yield rep, fold, a, b


def corrected_t_ci(scores, n_train: float, n_test: float, alpha: float = 0.05) -> dict:
    """Mean of J = K x R fold scores with the Nadeau-Bengio corrected (1 - alpha) interval."""
    x = np.asarray(scores, float)
    x = x[np.isfinite(x)]
    J = len(x)
    if J < 2:
        return dict(mean=float(x.mean()) if J else np.nan, ci_low=np.nan, ci_high=np.nan, sd=np.nan, J=J)
    se = np.sqrt((1 / J + n_test / n_train) * x.var(ddof=1))
    t = stats.t.ppf(1 - alpha / 2, J - 1)
    return dict(
        mean=float(x.mean()),
        ci_low=float(x.mean() - t * se),
        ci_high=float(x.mean() + t * se),
        sd=float(x.std(ddof=1)),
        J=J,
    )


def corrected_paired_t(diffs, n_train: float, n_test: float) -> dict:
    """Nadeau-Bengio corrected resampled paired t-test on per-fold differences, with Cohen's d_z of the fold
    differences."""
    d = np.asarray(diffs, float)
    d = d[np.isfinite(d)]
    J = len(d)
    if J < 2 or d.std(ddof=1) == 0:
        return dict(mean_diff=float(d.mean()) if J else np.nan, t=np.nan, p=np.nan, cohens_dz=np.nan, J=J)
    se = np.sqrt((1 / J + n_test / n_train) * d.var(ddof=1))
    t = d.mean() / se
    return dict(
        mean_diff=float(d.mean()),
        t=float(t),
        p=float(2 * stats.t.sf(abs(t), J - 1)),
        cohens_dz=float(d.mean() / d.std(ddof=1)),
        J=J,
    )

In [ ]:
# 1.9  Library self-tests against independent reference implementations (run on synthetic data; they must all pass)
def _selftests():
    rng = rng_for("selftest")
    n = 3000
    y = (rng.random(n) < 0.15).astype(int)
    s = np.round(rng.normal(y * 0.9, 1.0), 2)  # rounding creates many ties
    ix = ScoreIndex(y, s)
    r = weighted_rank_metrics(*ix.group_sums(np.ones((1, n), np.float32)), which=("auroc", "auprc", "sens90"))
    assert abs(r["auroc"][0] - roc_auc_score(y, s)) < 1e-9 and abs(auroc(y, s) - roc_auc_score(y, s)) < 1e-12
    assert abs(r["auprc"][0] - average_precision_score(y, s)) < 1e-9, (r["auprc"][0], average_precision_score(y, s))
    assert abs(r["sens90"][0] - sens_at_spec(y, s)) < 1e-9
    w = rng.integers(0, 4, n)
    rep = np.repeat(np.arange(n), w)  # integer weights == row duplication
    r2 = weighted_rank_metrics(*ix.group_sums(w[None].astype(np.float32)), which=("auroc", "auprc"))
    assert (
        abs(r2["auroc"][0] - roc_auc_score(y[rep], s[rep])) < 1e-9
        and abs(r2["auprc"][0] - average_precision_score(y[rep], s[rep])) < 1e-9
    )
    # DeLong: equals the Obuchowski estimator when every patient has one image; close to the cluster-bootstrap variance
    # otherwise
    s2 = s + rng.normal(0, 0.5, n)
    d1 = delong_test(y, s, s2)
    d2 = delong_test(y, s, s2, clusters=np.arange(n))
    assert abs(d1["se"] - d2["se"]) < 1e-6 * max(1, d1["se"]) * 1e3, (d1, d2)
    g = np.repeat(np.arange(n // 3), 3)
    u = rng.normal(0, 1, n // 3)[g]
    yc = (rng.random(n) < 0.2).astype(int)
    sa, sb = yc + u + rng.normal(0, 1, n), yc + 0.7 * u + rng.normal(0, 1.2, n)
    dc = delong_test(yc, sa, sb, clusters=g)
    dd = []
    for _ in range(400):
        pick = rng.integers(0, n // 3, n // 3)
        ii = np.concatenate([np.arange(3 * k, 3 * k + 3) for k in pick])
        dd.append(roc_auc_score(yc[ii], sa[ii]) - roc_auc_score(yc[ii], sb[ii]))
    assert 0.75 < dc["se"] / np.std(dd) < 1.33, (dc["se"], np.std(dd))
    pv = rng.random(12)
    if HAS_SM:
        from statsmodels.stats.multitest import multipletests

        assert np.allclose(holm(pv), multipletests(pv, method="holm")[1]) and np.allclose(
            bh_fdr(pv), multipletests(pv, method="fdr_bh")[1]
        )
    pc = rng.random(20000)
    yc2 = (rng.random(20000) < pc).astype(int)
    a_, b_ = logistic_recalibration(yc2, pc)
    assert abs(b_ - 1) < 0.1 and abs(a_) < 0.1 and ece_score(yc2, pc) < 0.02
    ci = corrected_t_ci(rng.normal(0.8, 0.01, 50), 800, 200)
    assert ci["ci_low"] < 0.8 < ci["ci_high"]
    fr = friedman_nemenyi(pd.DataFrame(rng.normal(0, 1, (30, 4)) + np.arange(4)))
    assert list(fr["mean_ranks"].index)[0] == 3  # highest scores rank first
    return (
        "all library self-tests passed (AUROC/AUPRC/sensitivity with ties and weights, DeLong/Obuchowski, Holm/BH, "
        "calibration, CV CI, Friedman)"
    )


print(_selftests())

In [ ]:
# 2.1  Study constants: findings, sites, training sets, architectures, methods
LABELS = ["Cardiomegaly", "Edema", "Consolidation", "Atelectasis", "Pneumothorax", "Pleural Effusion"]
L = len(LABELS)
FIND5 = [f_ for f_ in LABELS if f_ != "Edema"]
IDX5 = [LABELS.index(f_) for f_ in FIND5]  # VinBigData has no Edema
SITES = {"A": "NIH ChestX-ray14", "B": "CheXpert", "C": "VinBigData"}
SHORT = {"A": "NIH", "B": "CheXpert", "C": "VinBigData"}
TRAIN_SETS = ["A", "B", "C", "A+B", "A+C", "B+C"]
POOLED = ["A+B", "A+C", "B+C"]
UNSEEN = {"A+B": "C", "A+C": "B", "B+C": "A"}
SOURCES_FOR = {"A": ("B", "C"), "B": ("A", "C"), "C": ("A", "B")}
POOL_FOR = {"A": "B+C", "B": "A+C", "C": "A+B"}
ARCHS = ["DenseNet121", "ConvNeXt-T", "ViT-S"]
CLASSICAL = "LogReg (pixels+acq.)"
NAT, BAL = "ERM (natural)", "ERM (balanced)"
METHODS = [NAT, BAL, "DANN", "CORAL", "GroupDRO"]
METHOD_TAG = {NAT: "erm_nat", BAL: "erm_bal", "DANN": "dann", "CORAL": "coral", "GroupDRO": "gdro"}
ARCH_SPEC = {  # recipes used by every earlier run (reported as such; no per-architecture search was run)
    "DenseNet121": dict(timm="densenet121", short="densenet", lr=1e-4, wd=1e-4, warmup=0, clip=None, drop_path=0.0),
    "ConvNeXt-T": dict(
        timm="convnext_tiny.fb_in1k", short="convnext", lr=1e-4, wd=1e-4, warmup=0, clip=None, drop_path=0.0
    ),
    "ViT-S": dict(
        timm="vit_small_patch16_224.augreg_in21k_ft_in1k",
        short="vit",
        lr=5e-5,
        wd=0.05,
        warmup=300,
        clip=1.0,
        drop_path=0.1,
    ),
}
DG_DEFAULTS = dict(dann_w=0.1, coral_w=1.0, dro_eta=0.01)  # the settings of every DG run in the archive
SEEDS = list(CFG.seeds)
ARCHIVE_EPOCHS = 7  # epochs of every archived run (part of its file names); CFG.epochs is for NEW runs


def is_seen(ts: str, T: str) -> bool:
    """True when test site T is one of the training sources of training set ts (in-distribution evaluation)."""
    return T in ts.split("+")


def tsp(ts: str) -> str:
    """File-name form of a training set ('A+B' -> 'ApB')."""
    return ts.replace("+", "p")


def dname(ts: str, T: str) -> str:
    """Direction label 'training set→test site'."""
    return f"{ts}→{T}"


_cb = sns.color_palette("colorblind", 10)
ARCH_COL = {"DenseNet121": _cb[0], "ConvNeXt-T": _cb[1], "ViT-S": _cb[2], CLASSICAL: (0.45, 0.45, 0.45)}
ARCH_MK = {"DenseNet121": "o", "ConvNeXt-T": "^", "ViT-S": "s", CLASSICAL: "D"}
METHOD_COL = dict(zip(METHODS, [_cb[7], _cb[4], _cb[3], _cb[8], _cb[9]]))
SITE_COL = {"A": _cb[0], "B": _cb[3], "C": _cb[2]}
EXT_DIRS = [(ts, T) for ts in TRAIN_SETS for T in "ABC" if not is_seen(ts, T)]  # 9 external (unseen-site) directions
SEEN_DIRS = [
    (ts, T) for ts in TRAIN_SETS for T in "AB" if is_seen(ts, T)
]  # in-distribution cells with a clean held-out test set
print(
    f"{L} findings | {len(TRAIN_SETS)} training sets | {len(EXT_DIRS)} external "
    f"+ {len(SEEN_DIRS)} in-distribution directions | archs: {ARCHS}"
)

In [ ]:
# 2.2  Data builders. They reproduce the original preparation steps EXACTLY and run only when an output file is missing
#      (the archive already contains all of them, so on your machine this cell only checks that they exist).
from concurrent.futures import ThreadPoolExecutor

NIH_MAP = {
    "Cardiomegaly": "Cardiomegaly",
    "Edema": "Edema",
    "Consolidation": "Consolidation",
    "Atelectasis": "Atelectasis",
    "Pneumothorax": "Pneumothorax",
    "Effusion": "Pleural Effusion",
}
VB_MAP = {
    "Cardiomegaly": "Cardiomegaly",
    "Consolidation": "Consolidation",
    "Atelectasis": "Atelectasis",
    "Pneumothorax": "Pneumothorax",
    "Pleural Effusion": "Pleural effusion",
}
CX_PRESENT = 3  # CheXpert mirror coding: 0 unlabeled, 1 uncertain, 2 absent, 3 present -> U-zeros policy


def load_gray224(path, size: int = 224) -> np.ndarray:
    """8-bit PNG/JPEG -> grayscale -> direct bilinear resize to size x size (identical to every cache in the
    archive)."""
    from PIL import Image

    with Image.open(path) as im:
        return np.asarray(im.convert("L").resize((size, size), Image.BILINEAR))


def read_dicom_gray8(path, scale: str = "minmax") -> np.ndarray:
    """VinBigData DICOM -> VOI LUT -> MONOCHROME1 inverted -> 8-bit (min-max), as in the VinBigData preparation
    notebook."""
    import pydicom

    try:
        from pydicom.pixels import apply_voi_lut
    except ImportError:
        from pydicom.pixel_data_handlers.util import apply_voi_lut
    ds = pydicom.dcmread(path)
    a = ds.pixel_array
    try:
        a = apply_voi_lut(a, ds)
    except Exception:
        pass
    a = a.astype(np.float32)
    if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1":
        a = a.max() - a
    lo, hi = (a.min(), a.max()) if scale == "minmax" else np.percentile(a, [0.5, 99.5])
    return np.clip((a - lo) / max(hi - lo, 1e-6) * 255, 0, 255).astype(np.uint8)


def build_memmap_cache(paths, out_path: Path, reader, size: int = 224, block: int = 1000) -> list:
    """Resumable uint8 memmap cache (N, size, size). Progress is stored next to the cache; unreadable files are
    reported."""
    from PIL import Image

    prog = Path(str(out_path) + ".progress.json")
    st = read_json(prog) if prog.exists() else dict(next=0, bad=[])
    mm = np.memmap(out_path, dtype=np.uint8, mode="r+" if out_path.exists() else "w+", shape=(len(paths), size, size))

    def put(i):
        try:
            a = reader(paths[i])
            mm[i] = (
                a if a.shape == (size, size) else np.asarray(Image.fromarray(a).resize((size, size), Image.BILINEAR))
            )
        except Exception as e:
            st["bad"].append(int(i))
            print("unreadable:", paths[i], type(e).__name__)

    for s0 in tqdm(range(st["next"], len(paths), block), desc=f"caching {out_path.name}"):
        with ThreadPoolExecutor(CFG.n_workers) as ex:
            list(ex.map(put, range(s0, min(s0 + block, len(paths)))))
        mm.flush()
        st["next"] = min(s0 + block, len(paths))
        write_json(st, prog)
    return st["bad"]


def build_nih_tables() -> None:
    """NIH ChestX-ray14: six findings from 'Finding Labels'; official test list; patient-level 90/10 train/val split of
    the official train_val part (GroupShuffleSplit, random_state=42) -> nih_split.csv (columns split_rev, official)."""
    raw = P["nih_raw"]
    paths = {}
    for root, _, files in os.walk(raw):
        for n in files:
            if n.lower().endswith(".png"):
                paths[n] = os.path.join(root, n)
    d = pd.read_csv(sorted(raw.glob("Data_Entry*.csv"))[0]).rename(
        columns={
            "Image Index": "name",
            "Finding Labels": "finding",
            "Patient ID": "pid",
            "Patient Age": "Age",
            "Patient Gender": "Sex",
            "View Position": "view",
            "Follow-up #": "followup",
        }
    )
    d = d[["name", "finding", "pid", "Age", "Sex", "view", "followup"]].copy()
    d["patient_id"] = "nih" + d["pid"].astype(str)
    d["Age"] = pd.to_numeric(d["Age"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
    d.loc[d["Age"] > 100, "Age"] = np.nan
    d["Sex"] = d["Sex"].map({"M": "Male", "F": "Female"})
    for k, v in NIH_MAP.items():
        d[v] = d["finding"].apply(lambda s, k=k: int(k in s.split("|")))
    d["img_path"] = d["name"].map(paths)
    d = d.dropna(subset=["img_path"]).reset_index(drop=True)
    test_names = set((raw / "test_list.txt").read_text().split())
    d["official"] = np.where(d["name"].isin(test_names), "test", "train_val")
    d.to_csv(P["nih_clean"], index=False)
    tv = np.flatnonzero(d["official"].values == "train_val")
    te = np.flatnonzero(d["official"].values == "test")
    a, b = next(GroupShuffleSplit(1, test_size=0.1, random_state=42).split(tv, groups=d["patient_id"].values[tv]))
    d["split_rev"] = "train"
    d.loc[np.sort(tv[b]), "split_rev"] = "val"
    d.loc[te, "split_rev"] = "test"
    d.to_csv(P["nih_split"], index=False)


def build_chexpert_tables() -> None:
    """CheXpert (frontal images of the Hugging Face mirror): U-zeros (only 'present' = 1), patient-level 80/10/10 split
    (two GroupShuffleSplits, random_state=42); the 202 expert-labelled 'valid' images are kept apart."""
    meta = pd.read_csv(P["cx_meta"])
    for lab in LABELS + ["Support Devices"]:
        meta[lab + "_raw"] = meta[lab]
        meta[lab] = (meta[lab] == CX_PRESENT).astype(int)
    meta["Sex"] = meta["Sex"].map({0: "Male", 1: "Female"})
    meta["origin"] = meta["Path"].str.extract(r"/(train|valid)/")[0]
    tr_all = meta[meta["origin"] == "train"].copy()
    ex = meta[meta["origin"] == "valid"].copy()
    a, b = next(
        GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(tr_all, groups=tr_all["patient_id"])
    )
    tmp = tr_all.iloc[b]
    va, te = next(GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42).split(tmp, groups=tmp["patient_id"]))
    tr_all["split"] = "train"
    tr_all.loc[tmp.index[va], "split"] = "val"
    tr_all.loc[tmp.index[te], "split"] = "test"
    ex["split"] = "expert_valid"
    final = pd.concat([tr_all, ex], ignore_index=True)
    assert (final[final.split != "expert_valid"].groupby("patient_id")["split"].nunique() == 1).all(), "patient leakage"
    final.to_csv(P["cx_csv"], index=False)


def build_vinbig_labels() -> None:
    """VinBigData: image positive for a finding when >= 2 of the 3 radiologists marked it (sensitivity rules kept: any
    rater, rater count)."""
    d = pd.read_csv(P["vb_raw"] / "train.csv")
    ids_all = d["image_id"].unique()
    pos = d[d["class_name"] != "No finding"].drop_duplicates(["image_id", "rad_id", "class_name"])
    cnt = (
        pos.groupby(["image_id", "class_name"])["rad_id"]
        .nunique()
        .unstack(fill_value=0)
        .reindex(ids_all, fill_value=0)
        .rename_axis("image_id")
    )
    lab = pd.DataFrame(index=cnt.index)
    for name, col in VB_MAP.items():
        n = cnt[col] if col in cnt else 0
        lab[name] = (n >= 2).astype(int)
        lab[name + "_any"] = (n >= 1).astype(int)
        lab[name + "_n"] = n
    files = {Path(f).stem: f for f in glob.glob(str(P["vb_raw"] / "train" / "*"))}
    lab = lab.reset_index()
    lab = lab[lab["image_id"].isin(files)].reset_index(drop=True)
    P["vb_labels"].parent.mkdir(parents=True, exist_ok=True)
    lab.to_csv(P["vb_labels"], index=False)


def ensure_data() -> None:
    """Builds whatever derived file is missing (only possible when the raw data are present) and fails with a clear
    list otherwise."""
    if not P["nih_split"].exists() and P["nih_raw"].exists():
        build_nih_tables()
    if not P["cx_csv"].exists() and P["cx_meta"].exists():
        build_chexpert_tables()
    if not P["vb_labels"].exists() and P["vb_raw"].exists():
        build_vinbig_labels()
    S_ = CFG.image_size
    if not P["nih_u8"].exists() and P["nih_split"].exists():
        build_memmap_cache(pd.read_csv(P["nih_split"])["img_path"].tolist(), P["nih_u8"], load_gray224)
    if P["cx_csv"].exists():
        cx_ = pd.read_csv(P["cx_csv"])
        if not P["cx_trainval_u8"].exists():
            build_memmap_cache(
                cx_[cx_["split"].isin(["train", "val"])]["img_path"].tolist(), P["cx_trainval_u8"], load_gray224
            )
        if not P["cx_test_npy"].exists():
            tmp = P["cx_test_npy"].with_suffix(".u8")
            te_paths = cx_[cx_["split"] == "test"]["img_path"].tolist()
            build_memmap_cache(te_paths, tmp, load_gray224)
            np.save(
                P["cx_test_npy"], np.asarray(np.memmap(tmp, dtype=np.uint8, mode="r", shape=(len(te_paths), S_, S_)))
            )
    if not P["vb_u8"].exists() and P["vb_labels"].exists() and P["vb_raw"].exists():
        files = {Path(f).stem: f for f in glob.glob(str(P["vb_raw"] / "train" / "*"))}
        bad = build_memmap_cache(
            [files[i] for i in pd.read_csv(P["vb_labels"])["image_id"]], P["vb_u8"], read_dicom_gray8
        )
        write_json(dict(next=len(files), bad=bad), P["vb_progress"])
    need = {k: P[k] for k in ["nih_split", "nih_u8", "cx_csv", "cx_trainval_u8", "cx_test_npy", "vb_labels", "vb_u8"]}
    missing = {k: str(v) for k, v in need.items() if not v.exists()}
    if missing:
        raise FileNotFoundError(
            "missing inputs (set CXR_DATA_ROOT / CXR_RUNS or edit Config):\n" + json.dumps(missing, indent=2)
        )
    print("all derived data files present:", ", ".join(need))

In [ ]:
# 2.3  Load the three sites, assemble training pools / test sets, and verify integrity (patient leakage, cache shapes,
#      label coding)
with timed("2 data loading"):
    ensure_data()
    S_ = CFG.image_size
    # ---- A: NIH ChestX-ray14 (official test list; patient-level train/val split) ----
    nih = pd.read_csv(P["nih_split"])
    off = nih["official"].values == "test"
    te_i = np.flatnonzero(off)
    tr_i = np.flatnonzero(nih["split_rev"].values == "train")
    va_i = np.flatnonzero(nih["split_rev"].values == "val")
    NIH_A = np.memmap(P["nih_u8"], dtype=np.uint8, mode="r", shape=(len(nih), S_, S_))
    Yn = nih[LABELS].values.astype(np.int8)
    # ---- B: CheXpert (patient-level 80/10/10 split; U-zeros) ----
    cx = pd.read_csv(P["cx_csv"])
    cxt = cx[cx["split"] == "test"].reset_index(drop=True)
    cxm = cx[cx["split"].isin(["train", "val"])].reset_index(drop=True)
    cx_tr = np.flatnonzero(cxm["split"].values == "train")
    cx_va = np.flatnonzero(cxm["split"].values == "val")
    CX_A = np.memmap(P["cx_trainval_u8"], dtype=np.uint8, mode="r", shape=(len(cxm), S_, S_))
    CX_T = np.load(P["cx_test_npy"], mmap_mode="r")
    Ycm = cxm[LABELS].values.astype(np.int8)
    Yc = cxt[LABELS].values.astype(np.int8)
    # ---- C: VinBigData (>= 2 of 3 radiologists; no Edema; no patient identifier) ----
    vb = pd.read_csv(P["vb_labels"])
    NVB = len(vb)
    bad = read_json(P["vb_progress"])["bad"] if P["vb_progress"].exists() else []
    assert len(bad) == 0, f"{len(bad)} unreadable VinBigData images: cache rows would not match the labels"
    VB_A = np.memmap(P["vb_u8"], dtype=np.uint8, mode="r", shape=(NVB, S_, S_))
    Yvb = np.zeros((NVB, L), np.int8)
    for f_ in FIND5:
        Yvb[:, LABELS.index(f_)] = vb[f_].values
    if P["vb_split"].exists():  # same 80/20 image split as the archived C-trained runs
        s_ = pd.read_csv(P["vb_split"])
        vb_va = np.flatnonzero(s_["split"].values == "val")
        vb_tr = np.flatnonzero(s_["split"].values == "train")
    else:
        perm = np.random.default_rng(0).permutation(NVB)
        vb_va = np.sort(perm[: NVB // 5])
        vb_tr = np.sort(perm[NVB // 5 :])
    VALID_C = np.array([lab != "Edema" for lab in LABELS])

SRC = {
    "A": dict(arr=NIH_A, Y=Yn, tr=tr_i, va=va_i, valid=np.ones(L, bool), pid=nih["patient_id"].values),
    "B": dict(arr=CX_A, Y=Ycm, tr=cx_tr, va=cx_va, valid=np.ones(L, bool), pid=cxm["patient_id"].values),
    "C": dict(arr=VB_A, Y=Yvb, tr=vb_tr, va=vb_va, valid=VALID_C, pid=vb["image_id"].values),
}
TEST = {
    "A": (NIH_A, te_i),
    "B": (CX_T, np.arange(len(CX_T))),
    "C": (VB_A, np.arange(NVB)),
}  # C is a test site only for models not trained on it
YT = {"A": Yn[te_i], "B": Yc, "C": Yvb}
VALID = {"A": np.ones(L, bool), "B": np.ones(L, bool), "C": VALID_C}
GID = {
    "A": pd.factorize(nih["patient_id"].values[off])[0],
    "B": pd.factorize(cxt["patient_id"].values)[0],
    "C": np.arange(NVB),
}
VALY = {"A": Yn[va_i], "B": Ycm[cx_va], "C": Yvb[vb_va]}
FIND_SITE = {T: [j for j in range(L) if VALID[T][j]] for T in "ABC"}
CX_VIEW_CODE = {0: "AP", 1: "PA"}  # verified on the CheXpert mirror (che44-nih77 notebook)
META = {
    "A": pd.DataFrame(
        dict(
            sex=nih["Sex"].values[off],
            age=pd.to_numeric(nih["Age"], errors="coerce").values[off],
            view=nih["view"].values[off],
        )
    ),
    "B": pd.DataFrame(
        dict(
            sex=cxt["Sex"].values,
            age=pd.to_numeric(cxt["Age"], errors="coerce").values,
            view=pd.Series(cxt["AP/PA"].values).map(CX_VIEW_CODE).fillna("other").values,
        )
    ),
    "C": pd.DataFrame(dict(sex=np.full(NVB, np.nan), age=np.full(NVB, np.nan), view=np.full(NVB, "PA"))),
}  # VinBigData labels carry no demographics

# ---------------- integrity checks (fail loudly) ----------------
pa = {k: set(nih["patient_id"].values[i]) for k, i in [("train", tr_i), ("val", va_i), ("test", te_i)]}
pb = {k: set(cx.loc[cx["split"] == k, "patient_id"]) for k in ["train", "val", "test"]}
for nm, ps in [("NIH", pa), ("CheXpert", pb)]:
    for a_, b_ in itertools.combinations(ps, 2):
        assert not (ps[a_] & ps[b_]), f"patient leakage in {nm}: {a_} / {b_}"
assert NIH_A.shape[0] == len(nih) and CX_A.shape[0] == len(cxm) and CX_T.shape[0] == len(cxt) and VB_A.shape[0] == NVB
assert set(np.unique(Yn)) <= {0, 1} and set(np.unique(Yc)) <= {0, 1} and set(np.unique(Yvb)) <= {0, 1}
assert not set(vb_tr) & set(vb_va)
display(
    pd.DataFrame(
        {
            "train images": [len(tr_i), len(cx_tr), len(vb_tr)],
            "train patients": [len(pa["train"]), len(pb["train"]), len(vb_tr)],
            "validation images": [len(va_i), len(cx_va), len(vb_va)],
            "test images": [len(te_i), len(cxt), NVB],
            "test patients": [len(pa["test"]), len(pb["test"]), NVB],
        },
        index=[f"{SITES[t]} ({t})" for t in "ABC"],
    )
)
print("integrity checks passed: no patient appears in two partitions; caches and label tables are aligned")

In [ ]:
# 2.4  Table 1 (cohort characteristics) and Table 2 (between-site differences with effect sizes)
def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    den = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / den
    h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / den
    return (c - h, c + h)


def cramers_v(table) -> float:
    table = np.asarray(table, float)
    chi2 = stats.chi2_contingency(table, correction=False)[0]
    return float(np.sqrt(chi2 / (table.sum() * (min(table.shape) - 1))))


with timed("2 cohort tables"):
    rows = []
    for T in "ABC":
        Y = YT[T]
        m = META[T]
        ipp = pd.Series(GID[T]).value_counts()
        r = {
            "site": f"{SITES[T]} ({T})",
            "test images": len(Y),
            "test patients": int(GID[T].max() + 1),
            "images per patient (mean)": f"{ipp.mean():.2f}",
            "female, %": f"{100 * (m['sex'] == 'Female').mean():.1f}" if m["sex"].notna().any() else "not recorded",
            "age, mean ± SD": fmt_pm(m["age"].dropna(), 1) if m["age"].notna().any() else "not recorded",
            "AP view, %": f"{100 * (m['view'] == 'AP').mean():.1f}" if T != "C" else "not recorded",
            "training images": len(SRC[T]["tr"]),
        }
        for j, f_ in enumerate(LABELS):
            k = int(Y[:, j].sum())
            lo, hi = wilson(k, len(Y))
            r[f"{f_}, % [95% CI]"] = (
                f"{100 * k / len(Y):.1f} [{100 * lo:.1f}, {100 * hi:.1f}]" if VALID[T][j] else "not labelled"
            )
        rows.append(r)
    T1 = save_tab(
        pd.DataFrame(rows).set_index("site").T,
        "table01_cohort_characteristics",
        "Cohort characteristics of the three test sites (prevalence with Wilson 95% CI).",
    )

    rows = []  # between-site differences (are the test sets exchangeable?)
    for j, f_ in enumerate(LABELS):
        sites = [T for T in "ABC" if VALID[T][j]]
        tab = np.array([[YT[T][:, j].sum(), len(YT[T]) - YT[T][:, j].sum()] for T in sites])
        chi2, p, dof, _ = stats.chi2_contingency(tab, correction=False)
        rows.append(
            dict(
                variable=f"prevalence: {f_}",
                sites="+".join(sites),
                test="chi-square",
                statistic=chi2,
                df=dof,
                p=p,
                effect="Cramér's V",
                effect_size=cramers_v(tab),
            )
        )
    a_age, b_age = META["A"]["age"].dropna().values, META["B"]["age"].dropna().values
    U = stats.mannwhitneyu(a_age, b_age)
    rows.append(
        dict(
            variable="age",
            sites="A+B",
            test="Mann-Whitney U",
            statistic=float(U.statistic),
            df=np.nan,
            p=float(U.pvalue),
            effect="Cliff's delta (A vs B)",
            effect_size=cliffs_delta(a_age, b_age),
        )
    )
    for var, levels in [("sex", ["Male", "Female"]), ("view", ["AP", "PA"])]:
        tab = np.array([[(META[T][var] == lv).sum() for lv in levels] for T in "AB"])
        chi2, p, dof, _ = stats.chi2_contingency(tab, correction=False)
        rows.append(
            dict(
                variable=var,
                sites="A+B",
                test="chi-square",
                statistic=chi2,
                df=dof,
                p=p,
                effect="Cramér's V",
                effect_size=cramers_v(tab),
            )
        )
    T2 = pd.DataFrame(rows)
    T2["p_holm"] = holm(T2.p.values)
    save_tab(
        T2,
        "table02_between_site_differences",
        "Between-site differences of the test sets (Holm-adjusted).",
        index=False,
    )

    fig, ax = plt.subplots(2, 2, figsize=(COL2, 4.6))
    prev = pd.DataFrame(
        [
            dict(
                site=SHORT[T],
                finding=f_,
                prevalence=YT[T][:, j].mean(),
                lo=wilson(YT[T][:, j].sum(), len(YT[T]))[0],
                hi=wilson(YT[T][:, j].sum(), len(YT[T]))[1],
            )
            for T in "ABC"
            for j, f_ in enumerate(LABELS)
            if VALID[T][j]
        ]
    )
    w = 0.27
    for i, T in enumerate("ABC"):
        g = prev[prev.site == SHORT[T]].set_index("finding").reindex(LABELS)
        ax[0, 0].bar(
            np.arange(L) + (i - 1) * w,
            g.prevalence,
            w,
            yerr=[g.prevalence - g.lo, g.hi - g.prevalence],
            capsize=1.2,
            color=SITE_COL[T],
            label=SHORT[T],
            error_kw=dict(lw=0.6),
        )
    ax[0, 0].set_xticks(range(L))
    ax[0, 0].set_xticklabels(LABELS, rotation=30, ha="right")
    ax[0, 0].set_ylabel("Prevalence (test set)")
    ax[0, 0].legend()
    panel_label(ax[0, 0], "a")
    ages = pd.concat([pd.DataFrame(dict(age=META[T]["age"].dropna(), site=SHORT[T])) for T in "AB"])
    sns.kdeplot(
        data=ages,
        x="age",
        hue="site",
        common_norm=False,
        fill=True,
        alpha=0.3,
        ax=ax[0, 1],
        palette=[SITE_COL["A"], SITE_COL["B"]],
        lw=0.8,
    )
    ax[0, 1].set(xlabel="Age (years)", ylabel="Density")
    panel_label(ax[0, 1], "b")
    vm = (
        pd.DataFrame({SHORT[T]: META[T]["view"].value_counts(normalize=True) for T in "AB"})
        .reindex(["PA", "AP", "other"])
        .fillna(0)
    )
    bottom = np.zeros(2)
    for k_, c_ in zip(vm.index, ["0.3", "0.65", "0.9"]):
        ax[1, 0].bar(range(2), vm.loc[k_].values, 0.55, bottom=bottom, color=c_, edgecolor="k", lw=0.4, label=k_)
        bottom += vm.loc[k_].values
    ax[1, 0].set_xticks(range(2))
    ax[1, 0].set_xticklabels(vm.columns)
    ax[1, 0].set_ylabel("Share of test images")
    ax[1, 0].legend(title="view")
    panel_label(ax[1, 0], "c")
    for T in "AB":
        c_ = np.bincount(np.bincount(GID[T]))
        ax[1, 1].plot(np.arange(len(c_))[1:], c_[1:], marker="o", ms=2.5, color=SITE_COL[T], label=SHORT[T])
    ax[1, 1].set(xscale="log", yscale="log", xlabel="Test images per patient", ylabel="Patients")
    ax[1, 1].legend()
    panel_label(ax[1, 1], "d")
    fig.tight_layout()
    save_fig(
        fig,
        "fig01_cohorts",
        "Cohort characteristics: (a) prevalence with Wilson 95% CI, (b) age, "
        "(c) view position, (d) images per patient (the reason for the patient-level bootstrap).",
    )

In [ ]:
# 2.5  Pre-processing check: after identical resizing, do the three caches have the same intensity statistics?
#      (Bones bright / air dark in every site; per-image mean and contrast compared with two-sample KS and Wasserstein
#      distances.)
with timed("2 preprocessing check"):
    n_int = min(1000, *[len(TEST[T][1]) for T in "ABC"])
    SAMP_INT = {T: np.sort(rng_for("intensity", T).choice(TEST[T][1], n_int, replace=False)) for T in "ABC"}
    INT = {}
    for T in "ABC":
        x = np.asarray(TEST[T][0][SAMP_INT[T]], np.float32).reshape(n_int, -1)
        INT[T] = pd.DataFrame(dict(mean=x.mean(1), contrast=x.std(1), dark=(x < 5).mean(1), bright=(x > 250).mean(1)))
    rows = []
    for a_, b_ in itertools.combinations("ABC", 2):
        for v in ["mean", "contrast"]:
            ks = stats.ks_2samp(INT[a_][v], INT[b_][v])
            rows.append(
                dict(
                    pair=f"{SHORT[a_]} vs {SHORT[b_]}",
                    statistic=v,
                    KS_D=ks.statistic,
                    p=ks.pvalue,
                    wasserstein=stats.wasserstein_distance(INT[a_][v], INT[b_][v]),
                    cliffs_delta=cliffs_delta(INT[a_][v], INT[b_][v]),
                )
            )
    IT = pd.DataFrame(rows)
    IT["p_holm"] = holm(IT.p.values)
    save_tab(
        IT,
        "table03_intensity_shift",
        "Per-image intensity statistics between sites (two-sample KS, Wasserstein distance, Cliff's delta).",
        index=False,
    )
    display(pd.concat({SHORT[T]: INT[T].describe().loc[["mean", "std"]] for T in "ABC"}).round(3))

    fig = plt.figure(figsize=(COL2, 4.0))
    gs = fig.add_gridspec(2, 6, height_ratios=[1.1, 1])
    for r_, T in enumerate("ABC"):
        for c_ in range(2):
            a_ = fig.add_subplot(gs[0, r_ * 2 + c_])
            a_.imshow(TEST[T][0][SAMP_INT[T][c_ * 7]], cmap="gray", vmin=0, vmax=255)
            a_.axis("off")
            if c_ == 0:
                a_.set_title(SHORT[T], loc="left", fontsize=7)
    long = pd.concat([INT[T].assign(site=SHORT[T]) for T in "ABC"])
    for k_, (v, lab) in enumerate([("mean", "Per-image mean intensity"), ("contrast", "Per-image contrast (SD)")]):
        a_ = fig.add_subplot(gs[1, k_ * 3 : (k_ + 1) * 3])
        sns.kdeplot(
            data=long,
            x=v,
            hue="site",
            common_norm=False,
            fill=True,
            alpha=0.25,
            lw=0.8,
            ax=a_,
            palette=[SITE_COL[t] for t in "ABC"],
        )
        a_.set(xlabel=lab, ylabel="Density" if k_ == 0 else "")
        if k_:
            a_.get_legend().remove()
    fig.tight_layout()
    save_fig(
        fig,
        "fig02_preprocessing_check",
        "Pre-processing check: example test images per site and per-image intensity distributions.",
    )

In [ ]:
# 3.1  Engineered features
#   (i)  32 interpretable ACQUISITION descriptors (intensity distribution, border/collimation, texture, geometry): they
#        describe how a hospital acquires and processes radiographs, not the disease, and are used to fingerprint
#   covariate shift. (ii) 32 x 32 block-mean PIXEL features (7 x 7 blocks of the 224-px cache): input of the classical
#   (non-deep) baseline.
ACQ_GROUPS = {
    "intensity": [
        "int_mean",
        "int_std",
        "int_skew",
        "int_kurt",
        "p01",
        "p05",
        "p25",
        "p50",
        "p75",
        "p95",
        "p99",
        "frac_dark",
        "frac_bright",
        "hist_entropy",
    ],
    "border": [
        "border_top",
        "border_bottom",
        "border_left",
        "border_right",
        "border_center_ratio",
        "corner_mean",
        "border_std",
    ],
    "texture": ["grad_mean", "grad_p90", "lap_var", "edge_density", "noise_hf"],
    "geometry": ["lr_asym", "tb_diff", "center_mean", "col_profile_std", "row_profile_std", "fov_fraction"],
}
ACQ_NAMES = [f for g in ACQ_GROUPS.values() for f in g]


def acquisition_features(X: np.ndarray) -> np.ndarray:
    """X: (n, 224, 224) uint8 -> (n, 32) float32 acquisition descriptors (vectorised; about 2 ms per image)."""
    x = X.astype(np.float32) / 255.0
    n, H, W = x.shape
    f = x.reshape(n, -1)
    b = max(H // 18, 4)
    mu = f.mean(1)
    sd = f.std(1) + 1e-6
    zc = (f - mu[:, None]) / sd[:, None]
    q = np.percentile(f, [1, 5, 25, 50, 75, 95, 99], axis=1)
    hist = np.stack([np.bincount((r * 63).astype(np.int64), minlength=64) for r in f]).astype(np.float32) / f.shape[1]
    ent = -(hist * np.log(hist + 1e-12)).sum(1)
    top, bot, lef, rig = (
        x[:, :b].mean((1, 2)),
        x[:, -b:].mean((1, 2)),
        x[:, :, :b].mean((1, 2)),
        x[:, :, -b:].mean((1, 2)),
    )
    c0, c1 = H // 4, 3 * H // 4
    center = x[:, c0:c1, c0:c1].mean((1, 2))
    corners = np.stack([x[:, :b, :b], x[:, :b, -b:], x[:, -b:, :b], x[:, -b:, -b:]], 1).mean((1, 2, 3))
    frame = np.concatenate(
        [
            x[:, :b].reshape(n, -1),
            x[:, -b:].reshape(n, -1),
            x[:, b:-b, :b].reshape(n, -1),
            x[:, b:-b, -b:].reshape(n, -1),
        ],
        1,
    )
    gy, gx = np.gradient(x, axis=(1, 2))
    gm = np.sqrt(gx**2 + gy**2).reshape(n, -1)
    lap = (x[:, 1:-1, 1:-1] * 4 - x[:, :-2, 1:-1] - x[:, 2:, 1:-1] - x[:, 1:-1, :-2] - x[:, 1:-1, 2:]).reshape(n, -1)
    blur = (x[:, ::2, ::2] + x[:, 1::2, ::2] + x[:, ::2, 1::2] + x[:, 1::2, 1::2]) / 4
    hf = np.abs(x[:, ::2, ::2] - blur).mean((1, 2))
    feats = np.column_stack(
        [
            mu,
            sd,
            (zc**3).mean(1),
            (zc**4).mean(1) - 3,
            *q,
            (f < 0.02).mean(1),
            (f > 0.98).mean(1),
            ent,
            top,
            bot,
            lef,
            rig,
            (top + bot + lef + rig) / 4 / (center + 1e-3),
            corners,
            frame.std(1),
            gm.mean(1),
            np.percentile(gm, 90, axis=1),
            lap.var(1),
            (gm > 0.08).mean(1),
            hf,
            np.abs(x[:, :, : W // 2] - x[:, :, W // 2 :][:, :, ::-1]).mean((1, 2)),
            x[:, : H // 3].mean((1, 2)) - x[:, -H // 3 :].mean((1, 2)),
            center,
            x.mean(1).std(1),
            x.mean(2).std(1),
            (f > 0.08).mean(1),
        ]
    )
    return feats.astype(np.float32)


def pixel_features(X: np.ndarray, grid: int = 32) -> np.ndarray:
    """(n, 224, 224) uint8 -> (n, grid*grid) block means in [0, 1]."""
    n, H, W = X.shape
    k = H // grid
    return (X[:, : grid * k, : grid * k].astype(np.float32).reshape(n, grid, k, grid, k).mean((2, 4)) / 255.0).reshape(
        n, -1
    )


def featurize(A, idx, block: int = 256, pixels: bool = True, desc: str = ""):
    """Streams images from a cache in blocks -> (acquisition features, pixel features)."""
    acq, pix = [], []
    for i in tqdm(range(0, len(idx), block), desc=desc or "features", leave=False):
        Xb = np.asarray(A[np.asarray(idx[i : i + block])])
        acq.append(acquisition_features(Xb))
        if pixels:
            pix.append(pixel_features(Xb).astype(np.float16))
    return np.concatenate(acq), (np.concatenate(pix) if pixels else None)


FEATURE_VERSION = "acq32+pix32x32|v2"  # change when the descriptor definitions change: invalidates the cache


def cached_features(name: str, A, idx, pixels: bool = True):
    key = zlib.crc32(np.asarray(idx, np.int64).tobytes()) ^ zlib.crc32(FEATURE_VERSION.encode())
    f = CACHE / f"features_{name}_{len(idx)}_{key:08x}.npz"
    if f.exists():
        z = np.load(f)
        return z["acq"], (z["pix"] if pixels and "pix" in z else None)
    acq, pix = featurize(A, idx, pixels=pixels, desc=name)
    np.savez(f, acq=acq, **({"pix": pix} if pix is not None else {}))
    return acq, pix


assert len(ACQ_NAMES) == acquisition_features(np.zeros((2, CFG.image_size, CFG.image_size), np.uint8)).shape[1] == 32
print("acquisition descriptors:", {g: len(v) for g, v in ACQ_GROUPS.items()}, "| pixel features: 32 x 32 = 1024")

In [ ]:
# 3.2  Acquisition-shift fingerprint: which engineered descriptors differ between hospitals, and by how much?
with timed("3 feature extraction"):
    n_fs = min(CFG.n_feat_per_site, *[len(YT[T]) for T in "ABC"])
    FS_POS = {
        T: np.sort(rng_for("feature-sample", T).choice(len(YT[T]), n_fs, replace=False)) for T in "ABC"
    }  # positions in each test set
    FS = {T: cached_features(f"site{T}", TEST[T][0], TEST[T][1][FS_POS[T]]) for T in "ABC"}
    rows = []
    for a_, b_ in itertools.combinations("ABC", 2):
        Xa, Xb = FS[a_][0], FS[b_][0]
        for k_, nm in enumerate(ACQ_NAMES):
            sp = np.sqrt((Xa[:, k_].var(ddof=1) + Xb[:, k_].var(ddof=1)) / 2)
            rows.append(
                dict(
                    pair=f"{SHORT[a_]} vs {SHORT[b_]}",
                    feature=nm,
                    group=next(g for g, v in ACQ_GROUPS.items() if nm in v),
                    smd=(Xa[:, k_].mean() - Xb[:, k_].mean()) / sp if sp > 0 else 0.0,
                    ks=stats.ks_2samp(Xa[:, k_], Xb[:, k_]).statistic,
                )
            )
    SHIFT = pd.DataFrame(rows)
    smd_wide = SHIFT.pivot(index="feature", columns="pair", values="smd").reindex(ACQ_NAMES)
    save_tab(
        SHIFT.groupby(["pair", "group"])[["smd", "ks"]].agg(lambda v: np.median(np.abs(v))).round(3),
        "table04_shift_by_feature_group",
        "Median absolute standardised mean difference (SMD, Cohen's d with "
        "pooled SD) and median KS distance of the acquisition "
        "descriptors, by descriptor group and site pair (medians: near-constant descriptors give very large d).",
    )
    fig, ax = plt.subplots(1, 1, figsize=(COL1 * 1.25, 4.6))
    v = min(5.0, float(np.nanmax(np.abs(smd_wide.values))))  # colour scale clipped at |d| = 5 (values beyond saturate)
    sns.heatmap(
        smd_wide.clip(-v, v),
        cmap="RdBu_r",
        center=0,
        vmin=-v,
        vmax=v,
        ax=ax,
        linewidths=0.2,
        linecolor="w",
        cbar_kws=dict(label=f"SMD (Cohen's d), clipped at ±{v:.0f}", shrink=0.6),
    )
    yt = 0
    for g, names in ACQ_GROUPS.items():
        yt += len(names)
        ax.axhline(yt, color="k", lw=0.6)
    ax.set(xlabel="", ylabel="")
    ax.tick_params(axis="y", labelsize=5.5)
    ax.tick_params(axis="x", labelsize=6, rotation=20)
    fig.tight_layout()
    save_fig(
        fig,
        "fig03_acquisition_shift_heatmap",
        "Standardised mean differences of 32 acquisition descriptors between sites (|d| > 0.8 = large).",
    )
print(
    "descriptors with |SMD| > 0.8 per pair:",
    SHIFT.assign(large=SHIFT.smd.abs() > 0.8).groupby("pair").large.sum().to_dict(),
)

In [ ]:
# 3.3  Site probe with repeated grouped K-fold CV: can engineered features alone identify the hospital (chance = 1/3)?
#      Three learners are compared on the SAME folds (paired): logistic regression and gradient boosting on the 32
#      acquisition descriptors, logistic regression on 64 PCA components of the 32 x 32 pixels. Importance = drop in
#      balanced accuracy when a descriptor (or a whole descriptor group) is permuted on the held-out fold (Breiman
#      2001; Fisher et al. 2019).
def _probe_models():
    return {
        "LR (acquisition)": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=1.0)),
        "GBM (acquisition)": lambda: HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, random_state=0),
        "LR (pixels, PCA-64)": lambda: make_pipeline(
            StandardScaler(),
            PCA(n_components=min(64, n_fs - 1), random_state=0),
            LogisticRegression(max_iter=2000, C=1.0),
        ),
    }


def _perm_drop(model, X, y, cols, rng, base, n_rep: int = 3):
    out = []
    for _ in range(n_rep):
        Xp = X.copy()
        perm = rng.permutation(len(X))
        Xp[:, cols] = X[perm][:, cols]
        out.append(base - balanced_accuracy_score(y, model.predict(Xp)))
    return float(np.mean(out))


with timed("3 site probe (repeated CV)"):
    Xacq = np.concatenate([FS[T][0] for T in "ABC"])
    Xpix = np.concatenate([FS[T][1] for T in "ABC"]).astype(np.float32)
    ysite = np.repeat(np.arange(3), n_fs)
    gsite = np.concatenate([[f"{T}{g}" for g in GID[T][FS_POS[T]]] for T in "ABC"])
    rows, imp_rows = [], []
    for rep, fold, a, b in tqdm(
        list(repeated_group_kfold(ysite, gsite, CFG.cv_k, CFG.cv_r, "site-probe")), desc="site probe CV"
    ):
        for nm, mk in _probe_models().items():
            X_ = Xpix if "pixels" in nm else Xacq
            m = mk().fit(X_[a], ysite[a])
            pr = m.predict_proba(X_[b])
            bacc = balanced_accuracy_score(ysite[b], pr.argmax(1))
            rows.append(
                dict(
                    rep=rep,
                    fold=fold,
                    model=nm,
                    balanced_accuracy=bacc,
                    macro_auroc=roc_auc_score(ysite[b], pr, multi_class="ovr"),
                    n_fit=len(a),
                    n_test=len(b),
                )
            )
            if nm == "LR (acquisition)":
                rng_ = rng_for("perm", rep, fold)
                for k_, f_ in enumerate(ACQ_NAMES):
                    imp_rows.append(
                        dict(
                            rep=rep,
                            fold=fold,
                            kind="feature",
                            name=f_,
                            importance=_perm_drop(m, Xacq[b], ysite[b], [k_], rng_, bacc),
                        )
                    )
                for g, names in ACQ_GROUPS.items():
                    imp_rows.append(
                        dict(
                            rep=rep,
                            fold=fold,
                            kind="group",
                            name=g,
                            importance=_perm_drop(
                                m, Xacq[b], ysite[b], [ACQ_NAMES.index(x) for x in names], rng_, bacc
                            ),
                        )
                    )
    PROBE = pd.DataFrame(rows)
    IMP = pd.DataFrame(imp_rows)
    ntr, nte = PROBE.n_fit.mean(), PROBE.n_test.mean()
    summ = []
    for nm, g in PROBE.groupby("model", sort=False):
        for met in ["balanced_accuracy", "macro_auroc"]:
            summ.append(dict(model=nm, metric=met, **corrected_t_ci(g[met].values, ntr, nte)))
    PROBE_SUM = save_tab(
        pd.DataFrame(summ),
        "table05_site_probe_cv",
        f"Site identification from engineered features, {CFG.cv_r} x {CFG.cv_k}-fold "
        "patient-grouped CV (Nadeau-Bengio corrected 95% CI; chance balanced accuracy = 0.333).",
        index=False,
    )
    piv = PROBE.pivot_table(index=["rep", "fold"], columns="model", values="balanced_accuracy")
    cmp_rows = [
        dict(A=a_, B=b_, **corrected_paired_t(piv[a_] - piv[b_], ntr, nte))
        for a_, b_ in itertools.combinations(piv.columns, 2)
    ]
    PROBE_CMP = pd.DataFrame(cmp_rows)
    PROBE_CMP["p_holm"] = holm(PROBE_CMP.p.values)
    save_tab(
        PROBE_CMP,
        "table06_site_probe_learner_comparison",
        "Corrected resampled paired t-tests between probe learners (balanced accuracy).",
        index=False,
    )
    IMPS = (
        IMP.groupby(["kind", "name"])["importance"]
        .apply(lambda v: pd.Series(corrected_t_ci(v.values, ntr, nte)))
        .unstack()
        .reset_index()
        .sort_values("mean", ascending=False)
    )
    save_tab(
        IMPS[IMPS.kind == "group"],
        "table07_site_probe_group_importance",
        "Permutation importance of descriptor groups (drop in balanced accuracy).",
        index=False,
    )

    fig, ax = plt.subplots(1, 2, figsize=(COL2, 3.2), gridspec_kw=dict(width_ratios=[1.6, 1]))
    top = IMPS[IMPS.kind == "feature"].head(15).iloc[::-1]
    grp_of = {f_: g for g, v in ACQ_GROUPS.items() for f_ in v}
    gcol = dict(zip(ACQ_GROUPS, sns.color_palette("colorblind", 4)))
    ax[0].barh(
        range(len(top)),
        top["mean"],
        xerr=[top["mean"] - top["ci_low"], top["ci_high"] - top["mean"]],
        color=[gcol[grp_of[n]] for n in top.name],
        error_kw=dict(lw=0.6, capsize=1.5),
    )
    ax[0].set_yticks(range(len(top)))
    ax[0].set_yticklabels(top.name, fontsize=6)
    ax[0].set_xlabel("Drop in balanced accuracy (95% corrected CI)")
    ax[0].axvline(0, color="k", lw=0.5)
    panel_label(ax[0], "a")
    ax[0].legend(
        handles=[mpl.patches.Patch(color=c_, label=g) for g, c_ in gcol.items()], loc="lower right", fontsize=6
    )
    sns.boxplot(data=PROBE, x="model", y="balanced_accuracy", ax=ax[1], color="0.85", fliersize=1.5, linewidth=0.6)
    ax[1].axhline(1 / 3, color="r", ls="--", lw=0.7, label="chance")
    ax[1].set(xlabel="", ylabel=f"Balanced accuracy ({CFG.cv_r} x {CFG.cv_k} folds)")
    ax[1].tick_params(axis="x", rotation=25, labelsize=6)
    ax[1].legend()
    panel_label(ax[1], "b")
    fig.tight_layout()
    save_fig(
        fig,
        "fig04_site_probe_importance",
        "Site probe: (a) permutation importance of acquisition descriptors (top 15, coloured by group); "
        "(b) balanced accuracy of three probe learners across repeated grouped CV folds.",
    )

In [ ]:
# 4.1  Deep-learning utilities shared by training, inference, embeddings and Grad-CAM (skipped cleanly without PyTorch)
if HAS_TORCH:
    DEV = "cuda" if torch.cuda.is_available() else "cpu"
    AMP = DEV == "cuda"
    _NORM = {}

    def arch_norm(arch: str):
        """Input normalisation of the pretrained weights (ImageNet statistics for DenseNet/ConvNeXt, 0.5/0.5 for the
        AugReg ViT)."""
        if arch not in _NORM:
            try:
                pc = timm.create_model(ARCH_SPEC[arch]["timm"], pretrained=False, num_classes=L).pretrained_cfg
                mu, sd = pc["mean"], pc["std"]
            except Exception:
                mu, sd = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
            _NORM[arch] = (torch.tensor(mu, device=DEV).view(1, 3, 1, 1), torch.tensor(sd, device=DEV).view(1, 3, 1, 1))
        return _NORM[arch]

    def make_model(arch: str, pretrained: bool = False):
        sp = ARCH_SPEC[arch]
        kw = dict(drop_path_rate=sp["drop_path"]) if sp["drop_path"] > 0 else {}
        return timm.create_model(sp["timm"], pretrained=pretrained, num_classes=L, **kw).to(DEV)

    def load_checkpoint(arch: str, path):
        m = make_model(arch, False)
        m.load_state_dict(torch.load(path, map_location=DEV, weights_only=True))
        return m.eval()

    def to_tensor(A, idx, arch: str, flip: bool = False):
        """uint8 cache rows -> normalised 3-channel tensor; flip = random horizontal flip (the only augmentation of
        every run)."""
        x = torch.from_numpy(np.ascontiguousarray(A[np.asarray(idx)])).to(DEV).float().div(255).unsqueeze(1)
        if flip:
            f = torch.rand(len(idx), device=DEV) < 0.5
            x = torch.where(f.view(-1, 1, 1, 1), x.flip(-1), x)
        mu, sd = arch_norm(arch)
        return (x.repeat(1, 3, 1, 1) - mu) / sd

    def autocast():
        return torch.autocast(device_type="cuda", enabled=AMP) if DEV == "cuda" else contextlib.nullcontext()

    @torch.inference_mode()
    def predict_logits(m, A, idx, arch: str, bs: int = 128) -> np.ndarray:
        m.eval()
        out = []
        for i in range(0, len(idx), bs):
            with autocast():
                out.append(m(to_tensor(A, idx[i : i + bs], arch)).float().cpu())
        return torch.cat(out).numpy()

    @torch.inference_mode()
    def embed(m, A, idx, arch: str, bs: int = 100) -> np.ndarray:
        """Penultimate (pre-logit) representation."""
        m.eval()
        out = []
        for i in range(0, len(idx), bs):
            with autocast():
                out.append(
                    m.forward_head(m.forward_features(to_tensor(A, idx[i : i + bs], arch)), pre_logits=True)
                    .float()
                    .cpu()
                )
        return torch.cat(out).numpy()

    class GRL(torch.autograd.Function):
        """Gradient reversal layer of DANN (Ganin et al. 2016)."""

        @staticmethod
        def forward(ctx, x, lam):
            ctx.lam = lam
            return x.view_as(x)

        @staticmethod
        def backward(ctx, g):
            return -ctx.lam * g, None

    def coral_loss(fa, fb):
        """Deep CORAL (Sun & Saenko 2016): squared Frobenius distance of the feature covariances / (4 d^2)."""
        d = fa.size(1)
        return ((torch.cov(fa.T) - torch.cov(fb.T)) ** 2).sum() / (4 * d * d)

    class GroupDRO:
        """Online GroupDRO (Sagawa et al. 2020). hardness='raw' uses the group loss (original); 'relative' divides it by
        the prior-only loss of that group so a group with a higher label prevalence is not automatically 'harder'
        (Stage 3 fix)."""

        def __init__(self, n_groups: int, eta: float, hardness: str = "raw", prior_loss=None):
            self.q = torch.ones(n_groups, device=DEV) / n_groups
            self.eta = eta
            self.hardness = hardness
            self.H = torch.tensor(
                prior_loss if prior_loss is not None else np.ones(n_groups), device=DEV, dtype=torch.float32
            )

        def __call__(self, ls, g):
            gl = torch.zeros_like(self.q)
            pres = torch.zeros_like(self.q)
            for k in range(len(self.q)):
                mk = g == k
                if mk.any():
                    gl[k] = ls[mk].mean()
                    pres[k] = 1.0
            hard = gl.detach() / self.H if self.hardness == "relative" else gl.detach()
            self.q = self.q * torch.exp(self.eta * hard * pres)
            self.q = self.q / self.q.sum()
            w = self.q * pres
            return (w * gl).sum() / w.sum()

    print(
        f"PyTorch {torch.__version__} | timm {timm.__version__} | device: {DEV}"
        + (f" ({torch.cuda.get_device_name(0)})" if DEV == "cuda" else "")
    )
else:
    DEV, AMP = "cpu", False
    print(
        "PyTorch/timm not installed: training, embeddings, Grad-CAM and the "
        "runtime benchmark are skipped; every analysis of saved predictions runs."
    )

In [ ]:
# 4.2  One resumable trainer for EVERY configuration of the study (architecture x training set x method x seed,
#      size-matched subsets, source mixtures, K-fold retraining). Output format = the archive's ViT/ConvNeXt format
#      ({T}_test_P, {k}_val_P), so new runs are picked up by the prediction registry exactly like archived ones. Model
#      selection uses SOURCE validation data only.
@dataclass
class TrainSpec:
    arch: str
    train_set: str
    method: str = NAT
    seed: int = 0
    epochs: Optional[int] = None
    lr: Optional[float] = None
    wd: Optional[float] = None
    sampler: Optional[str] = None  # natural | balanced (default: natural for ERM (natural), balanced otherwise)
    dann_w: float = DG_DEFAULTS["dann_w"]
    coral_w: float = DG_DEFAULTS["coral_w"]
    dro_eta: float = DG_DEFAULTS["dro_eta"]
    dro_hardness: str = "raw"
    subset: Optional[dict] = None  # {source: n_images}: random patient-level subset (size control, mixtures)
    train_idx: Optional[dict] = None  # {source: explicit training rows} (K-fold retraining)
    eval_limit: Optional[int] = None  # predict only the first n test/validation rows (trainer self-test only)
    tag: str = ""

    def stem(self) -> Path:
        name = f"{ARCH_SPEC[self.arch]['short']}_{tsp(self.train_set)}_{METHOD_TAG[self.method]}_seed{self.seed}" + (
            f"_{self.tag}" if self.tag else ""
        )
        return NEWRUNS / ARCH_SPEC[self.arch]["short"] / name


def patient_subset(pool_idx, pids, n_target: int, rng) -> np.ndarray:
    """Random subset of exactly n_target images drawn patient by patient (as in the size-control and mixture runs)."""
    if n_target >= len(pool_idx):
        return np.sort(pool_idx)
    up = np.unique(pids)
    rng.shuffle(up)
    cnt = pd.Series(pids).value_counts().to_dict()
    chosen, tot = [], 0
    for p_ in up:
        chosen.append(p_)
        tot += cnt[p_]
        if tot >= n_target:
            break
    idx = pool_idx[np.isin(pids, chosen)]
    return np.sort(rng.permutation(idx)[:n_target])


def prior_only_loss(Y, valid) -> float:
    """Binary cross-entropy of a model that only knows each finding's prevalence (GroupDRO 'relative hardness'
    denominator)."""
    p = np.clip(np.asarray(Y, float).mean(0), 1e-6, 1 - 1e-6)[np.asarray(valid, bool)]
    return float(np.mean(-(p * np.log(p) + (1 - p) * np.log(1 - p))))


def batch_plan(pools: dict, mode: str, seed: int, ep: int, bs: int):
    """natural: random pooled batches (hospital mix follows pool sizes); balanced: bs/len(pools) images from every
    hospital per step."""
    srcs = list(pools)
    rng = np.random.default_rng(seed * 100 + ep)
    spe = math.ceil(sum(len(v) for v in pools.values()) / bs)
    if mode == "natural" or len(srcs) == 1:
        dom = np.concatenate([np.full(len(pools[k]), i) for i, k in enumerate(srcs)])
        pos = np.concatenate([pools[k] for k in srcs])
        perm = rng.permutation(len(pos))
        for k in range(spe):
            s = perm[k * bs : (k + 1) * bs]
            yield {srcs[i]: np.sort(pos[s][dom[s] == i]) for i in range(len(srcs)) if (dom[s] == i).any()}
    else:
        h = bs // len(srcs)
        for _ in range(spe):
            yield {k: np.sort(rng.choice(pools[k], h, replace=False)) for k in srcs}


def source_val_auroc(m, arch: str, srcs) -> dict:
    out = {}
    for k in srcs:
        s_ = SRC[k]
        Pv = predict_logits(m, s_["arr"], s_["va"], arch)
        Yv = s_["Y"][s_["va"]]
        out[k] = float(
            np.nanmean(
                [auroc(Yv[:, j], Pv[:, j]) for j in range(L) if s_["valid"][j] and 5 <= Yv[:, j].sum() <= len(Yv) - 5]
            )
        )
    return out


def train_run(spec: TrainSpec) -> Path:
    """Trains (or resumes) one model and writes <stem>.pt, <stem>_history.csv and <stem>_preds.npz. Returns the
    predictions path."""
    assert HAS_TORCH, "PyTorch/timm are required for training"
    sp = ARCH_SPEC[spec.arch]
    srcs = spec.train_set.split("+")
    stem = spec.stem()
    stem.parent.mkdir(parents=True, exist_ok=True)
    BEST, LAST, HIST, PRD = (Path(str(stem) + s) for s in (".pt", "_last.pt", "_history.csv", "_preds.npz"))
    if PRD.exists():
        return PRD
    E = spec.epochs or CFG.epochs
    lr = spec.lr or sp["lr"]
    wd = spec.wd or sp["wd"]
    bs = CFG.batch_size
    mode = spec.sampler or ("natural" if spec.method == NAT else "balanced")
    assert spec.method in (NAT, BAL) or len(srcs) == 2, "DANN, CORAL and GroupDRO need two source hospitals"
    rng = np.random.default_rng([spec.seed, 9000])
    pools = {}
    for k in srcs:
        base = spec.train_idx[k] if spec.train_idx else SRC[k]["tr"]
        pools[k] = (
            patient_subset(base, SRC[k]["pid"][base], spec.subset[k], rng)
            if spec.subset and k in spec.subset
            else np.sort(base)
        )
    pools = {k: v for k, v in pools.items() if len(v)}
    seed_everything(spec.seed)
    m = make_model(spec.arch, pretrained=CFG.pretrained)
    disc = (
        nn.Sequential(nn.Linear(m.num_features, 256), nn.ReLU(), nn.Linear(256, len(srcs))).to(DEV)
        if spec.method == "DANN"
        else None
    )
    prior = [
        prior_only_loss(SRC[k]["Y"][pools[k]], SRC[k]["valid"]) for k in srcs
    ]  # loss of a prevalence-only model per hospital
    dro = GroupDRO(len(srcs), spec.dro_eta, spec.dro_hardness, prior) if spec.method == "GroupDRO" else None
    params = list(m.parameters()) + (list(disc.parameters()) if disc else [])
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=wd)
    spe = math.ceil(sum(len(v) for v in pools.values()) / bs)
    total = E * spe
    warm = min(sp["warmup"], spe) if sp["warmup"] else 0
    if warm:
        sch = torch.optim.lr_scheduler.LambdaLR(
            opt,
            lambda t: (t + 1) / warm if t < warm else 0.5 * (1 + math.cos(math.pi * (t - warm) / max(1, total - warm))),
        )
    else:
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    hist, best, start = [], -1.0, 0
    if LAST.exists():
        ck = torch.load(LAST, map_location=DEV, weights_only=False)
        m.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        sch.load_state_dict(ck["sched"])
        scaler.load_state_dict(ck["scaler"])
        hist, best, start = ck["hist"], ck["best"], ck["ep"] + 1
        if disc is not None:
            disc.load_state_dict(ck["disc"])
        if dro is not None:
            dro.q = ck["q"].to(DEV)
    valid_t = {k: torch.tensor(SRC[k]["valid"], dtype=torch.float32, device=DEV) for k in srcs}
    for ep in range(start, E):
        m.train()
        t0 = time.time()
        tl = 0.0
        nb = 0
        for step, batch in enumerate(
            tqdm(
                batch_plan(pools, mode, spec.seed, ep, bs), total=spe, leave=False, desc=f"{stem.name} ep {ep + 1}/{E}"
            )
        ):
            ks = list(batch)
            x = torch.cat([to_tensor(SRC[k]["arr"], batch[k], spec.arch, flip=True) for k in ks])
            y = torch.from_numpy(np.concatenate([SRC[k]["Y"][batch[k]] for k in ks]).astype(np.float32)).to(DEV)
            msk = torch.cat([valid_t[k].expand(len(batch[k]), L) for k in ks])
            dom = torch.cat([torch.full((len(batch[k]),), srcs.index(k), device=DEV, dtype=torch.long) for k in ks])
            with autocast():
                f = m.forward_head(m.forward_features(x), pre_logits=True)
                z = m.get_classifier()(f)
            f, z = f.float(), z.float()
            # masked: no Edema for C
            ls = (F.binary_cross_entropy_with_logits(z, y, reduction="none") * msk).sum(1) / msk.sum(1).clamp(min=1)
            if dro is not None:
                loss = dro(ls, dom)
            else:
                loss = ls.mean()
                if spec.method == "CORAL" and (dom == 0).sum() > 1 and (dom == 1).sum() > 1:
                    loss = loss + spec.coral_w * coral_loss(f[dom == 0], f[dom == 1])
                if spec.method == "DANN":
                    lam = 2 / (1 + math.exp(-10 * (ep * spe + step) / total)) - 1
                    loss = loss + spec.dann_w * F.cross_entropy(disc(GRL.apply(f, lam)), dom)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            if sp["clip"]:
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(params, sp["clip"])
            scaler.step(opt)
            scaler.update()
            sch.step()
            tl += float(loss.detach())
            nb += 1
        vs = source_val_auroc(m, spec.arch, srcs)
        vm = float(np.mean(list(vs.values())))
        hist.append(
            dict(
                epoch=ep + 1,
                train_loss=tl / max(nb, 1),
                val_mean=vm,
                **{f"val_{k}": v for k, v in vs.items()},
                sec=time.time() - t0,
                q=" ".join(f"{v:.4f}" for v in dro.q.cpu().numpy()) if dro is not None else "",
            )
        )
        pd.DataFrame(hist).to_csv(HIST, index=False)
        print(
            f"{stem.name} ep {ep + 1}/{E} | loss {hist[-1]['train_loss']:.4f} "
            f"| source-val AUROC {vs} | {hist[-1]['sec'] / 60:.1f} min"
        )
        if vm > best:
            best = vm
            torch.save(m.state_dict(), BEST)
        torch.save(
            dict(
                model=m.state_dict(),
                opt=opt.state_dict(),
                sched=sch.state_dict(),
                scaler=scaler.state_dict(),
                hist=hist,
                best=best,
                ep=ep,
                disc=disc.state_dict() if disc is not None else None,
                q=dro.q.cpu() if dro is not None else None,
            ),
            LAST,
        )
    m.load_state_dict(torch.load(BEST, map_location=DEV, weights_only=True))
    z = {}
    lim = (lambda ix: ix[: spec.eval_limit]) if spec.eval_limit else (lambda ix: ix)
    for T in "ABC":
        if not (T == "C" and "C" in srcs):  # C is a clean test site only if it was not trained on
            z[f"{T}_test_P"] = sigmoid(predict_logits(m, TEST[T][0], lim(TEST[T][1]), spec.arch)).astype(np.float32)
    for k in sorted(set(["A", "B"]) | set(srcs)):
        z[f"{k}_val_P"] = sigmoid(predict_logits(m, SRC[k]["arr"], lim(SRC[k]["va"]), spec.arch)).astype(np.float32)
    if spec.train_idx:  # K-fold retraining: predictions on the held-out source fold
        for k in srcs:
            ho = np.setdiff1d(SRC[k]["tr"], spec.train_idx[k])
            z[f"{k}_heldout_idx"] = ho
            z[f"{k}_heldout_P"] = sigmoid(predict_logits(m, SRC[k]["arr"], ho, spec.arch)).astype(np.float32)
    np.savez(PRD, **z)
    del m, opt, sch, scaler, disc
    gc.collect()
    if DEV == "cuda":
        torch.cuda.empty_cache()
    return PRD

In [ ]:
# 4.3  Classical (non-deep) baseline with repeated patient-grouped K-fold cross-validation
#      Logistic regression (one per finding) on 32 acquisition descriptors + PCA of the 32 x 32 pixel grid. It anchors
#      the benchmark from below, tests whether the cross-hospital asymmetry also appears in a shallow model, and, being
#      cheap, allows a true repeated K-fold CV with retraining in every fold (internal AUROC with corrected CI;
#      external AUROC of every fold model).
class ClassicalModel:
    """Standardise -> PCA(pixels) -> standardise -> winsorise at +/- `clip` SD -> L2 logistic regression per finding.
    Winsorising matters only for external hospitals: their descriptors can lie tens of training SDs away, which would
    push every logit to +/- infinity, make all probabilities identical in float32 and the external AUROC a
    meaningless 0.5."""

    def __init__(self, n_pca: int = 128, C: float = 1.0, clip: float = 5.0):
        self.n_pca, self.C, self.clip = n_pca, C, clip

    def _z(self, Xa, Xp):
        Z = np.hstack([self.sa.transform(Xa), self.pca.transform(np.asarray(Xp, np.float32))])
        return np.clip(self.sz.transform(Z), -self.clip, self.clip) if hasattr(self, "sz") else Z

    def fit(self, Xa, Xp, Y, valid):
        self.sa = StandardScaler().fit(Xa)
        self.pca = PCA(n_components=min(self.n_pca, len(Xp) - 1, Xp.shape[1]), random_state=0).fit(
            np.asarray(Xp, np.float32)
        )
        self.__dict__.pop("sz", None)
        self.sz = StandardScaler().fit(self._z(Xa, Xp))
        Z = self._z(Xa, Xp)
        self.models = {
            j: LogisticRegression(C=self.C, max_iter=2000).fit(Z, Y[:, j])
            for j in range(L)
            if valid[j] and 0 < Y[:, j].sum() < len(Y)
        }
        return self

    def decision(self, Xa, Xp) -> np.ndarray:
        Z = self._z(Xa, Xp)
        D = np.full((len(Z), L), np.nan)
        for j, m in self.models.items():
            D[:, j] = m.decision_function(Z)
        return D

    def predict_proba(self, Xa, Xp):
        return sigmoid(self.decision(Xa, Xp)).astype(np.float32)


def source_union_valid(ts: str) -> np.ndarray:
    """Findings a model trained on `ts` has labels for (a VinBigData-only model never saw Edema)."""
    return np.logical_or.reduce([SRC[k]["valid"] for k in ts.split("+")])


CLASSICAL_SETS = ["A", "B", "C", "A+B"]
CLASSICAL_VERSION = (
    f"winsorised-z5|pca{CFG.classical_pca}|n{CFG.n_classical_train}"  # stale prediction files are rebuilt
)
with timed("4 classical baseline (repeated CV)"):
    FEAT_TEST = {T: cached_features(f"test{T}", TEST[T][0], TEST[T][1]) for T in "ABC"}
    FEAT_VAL = {k: cached_features(f"val{k}", SRC[k]["arr"], SRC[k]["va"]) for k in "ABC"}
    CL_SAMPLE, FEAT_TR = {}, {}
    for k in "ABC":
        n_ = min(CFG.n_classical_train, len(SRC[k]["tr"]))
        CL_SAMPLE[k] = np.sort(rng_for("classical-sample", k).choice(SRC[k]["tr"], n_, replace=False))
        FEAT_TR[k] = cached_features(f"train{k}", SRC[k]["arr"], CL_SAMPLE[k])

    def _stack(ts):
        ks = ts.split("+")
        Xa = np.concatenate([FEAT_TR[k][0] for k in ks])
        Xp = np.concatenate([FEAT_TR[k][1] for k in ks])
        Y = np.concatenate([SRC[k]["Y"][CL_SAMPLE[k]] for k in ks])
        G = np.concatenate([[f"{k}{p_}" for p_ in SRC[k]["pid"][CL_SAMPLE[k]]] for k in ks])
        return Xa, Xp, Y, G

    cv_rows, imp_rows = [], []
    PIX_RING = np.zeros((32, 32), bool)
    PIX_RING[:4] = PIX_RING[-4:] = True
    PIX_RING[:, :4] = PIX_RING[:, -4:] = True
    PIX_RING = PIX_RING.ravel()
    for ts in CLASSICAL_SETS:
        Xa, Xp, Y, G = _stack(ts)
        valid = source_union_valid(ts)
        ystrat = np.minimum(Y[:, valid].sum(1), 2)
        for rep, fold, a, b in tqdm(
            list(repeated_group_kfold(ystrat, G, CFG.cv_k, CFG.classical_cv_r, f"classical-{ts}")),
            desc=f"classical CV {ts}",
            leave=False,
        ):
            mdl = ClassicalModel(CFG.classical_pca).fit(Xa[a], Xp[a], Y[a], valid)
            Pb = mdl.predict_proba(Xa[b], Xp[b])
            fj = [j for j in range(L) if valid[j] and j in mdl.models and 0 < Y[b, j].sum() < len(b)]
            row = dict(
                train=ts,
                rep=rep,
                fold=fold,
                n_fit=len(a),
                n_test=len(b),
                internal=np.mean([auroc(Y[b, j], Pb[:, j]) for j in fj]),
            )
            for T in "ABC":
                if not is_seen(ts, T):
                    Pt = mdl.predict_proba(*FEAT_TEST[T])
                    fT = [j for j in mdl.models if VALID[T][j]]
                    row[f"external_{T}"] = np.mean([auroc(YT[T][:, j], Pt[:, j]) for j in fT])
            cv_rows.append(row)
            if ts in ("A", "B") and rep == 0:  # grouped permutation importance on the held-out fold
                base = row["internal"]
                rng_ = rng_for("classical-perm", ts, fold)
                groups = {
                    "pixels: border ring": ("p", np.flatnonzero(PIX_RING)),
                    "pixels: interior": ("p", np.flatnonzero(~PIX_RING)),
                    **{f"acquisition: {g}": ("a", [ACQ_NAMES.index(x) for x in v]) for g, v in ACQ_GROUPS.items()},
                }
                for gname, (kind, cols) in groups.items():
                    Xa_, Xp_ = Xa[b].copy(), np.asarray(Xp[b], np.float32).copy()
                    perm = rng_.permutation(len(b))
                    if kind == "p":
                        Xp_[:, cols] = Xp_[perm][:, cols]
                    else:
                        Xa_[:, cols] = Xa_[perm][:, cols]
                    Pp = mdl.predict_proba(Xa_, Xp_)
                    imp_rows.append(
                        dict(
                            train=ts,
                            fold=fold,
                            group=gname,
                            importance=base - np.mean([auroc(Y[b, j], Pp[:, j]) for j in fj]),
                        )
                    )
    CLCV = pd.DataFrame(cv_rows)
    summ = []
    for ts, g in CLCV.groupby("train", sort=False):
        summ.append(
            dict(
                train=ts,
                evaluation="internal (held-out folds)",
                **corrected_t_ci(g.internal, g.n_fit.mean(), g.n_test.mean()),
            )
        )
        for T in "ABC":
            c_ = f"external_{T}"
            if c_ in g and g[c_].notna().any():
                v = g[c_].dropna().values
                summ.append(
                    dict(
                        train=ts,
                        evaluation=f"external {SHORT[T]} (fold models)",
                        mean=v.mean(),
                        ci_low=np.percentile(v, 2.5),
                        ci_high=np.percentile(v, 97.5),
                        sd=v.std(ddof=1) if len(v) > 1 else np.nan,
                        J=len(v),
                    )
                )
    CL_SUM = save_tab(
        pd.DataFrame(summ),
        "table08_classical_baseline_cv",
        "Classical baseline: repeated patient-grouped K-fold CV. Internal: "
        "Nadeau-Bengio corrected 95% CI; external: 2.5-97.5 percentile range over fold models.",
        index=False,
    )
    CL_IMP = pd.DataFrame(imp_rows)

    # final classical models: three random training subsamples ('seeds') per training set, saved in the registry format
    for ts in CLASSICAL_SETS:
        Xa, Xp, Y, G = _stack(ts)
        valid = source_union_valid(ts)
        for s in SEEDS:
            f = NEWRUNS / "classical" / f"classical_{tsp(ts)}_erm_nat_seed{s}_preds.npz"
            if f.exists():
                with np.load(f) as z_old:
                    if "version" in z_old.files and str(z_old["version"]) == CLASSICAL_VERSION:
                        continue
            f.parent.mkdir(parents=True, exist_ok=True)
            sel = np.sort(rng_for("classical-final", ts, s).choice(len(Y), int(0.9 * len(Y)), replace=False))
            mdl = ClassicalModel(CFG.classical_pca).fit(Xa[sel], Xp[sel], Y[sel], valid)
            z = {f"{T}_test_P": mdl.predict_proba(*FEAT_TEST[T]) for T in "ABC" if not (T == "C" and "C" in ts)}
            z.update({f"{k}_val_P": mdl.predict_proba(*FEAT_VAL[k]) for k in "ABC"})
            np.savez(f, version=np.array(CLASSICAL_VERSION), **z)
print("classical baseline ready:", sorted(p_.name for p_ in (NEWRUNS / "classical").glob("*.npz"))[:3], "...")

In [ ]:
# 4.4  Post-hoc adaptation models fitted on k labelled TARGET images (few-label recalibration) or on no target labels
#      at all. All are monotone in the score, so AUROC cannot change; they repair calibration and the operating
#      threshold. Platt/temperature/bias act on the logit; isotonic regression is non-parametric (Zadrozny & Elkan
#      2002).
import scipy.optimize


def _fit_lr_1d(z, y, fit_slope=True, fit_bias=True):
    if fit_slope and fit_bias:
        a, b = logistic_recalibration(y, sigmoid(z), fit_slope=True)
        if np.isfinite(a) and np.isfinite(b):
            return float(np.clip(b, 0.05, 20.0)), a
        fit_slope = False  # slope not estimable (separation): fall back to the offset
    if fit_bias:
        a, _ = logistic_recalibration(y, sigmoid(z), fit_slope=False)
        return (1.0, a) if np.isfinite(a) else (1.0, 0.0)
    res = scipy.optimize.minimize_scalar(
        lambda T: np.mean(np.logaddexp(0, z / T) - y * z / T), bounds=(0.05, 20), method="bounded"
    )
    return 1.0 / res.x, 0.0  # temperature scaling (Guo et al. 2017): slope 1/T, no bias


RECAL_METHODS = ["none (source threshold)", "threshold only", "bias", "temperature", "Platt", "isotonic"]


def recalibrate(method: str, z_fit, y_fit, z_eval, thr_source: float, min_events: int = 5):
    """Returns (calibrated probabilities on the evaluation images, decision threshold). Falls back to the uncorrected
    model when the labelled pool has fewer than `min_events` positives or negatives (the fallback rate is reported)."""
    p_eval = sigmoid(z_eval)
    npos = int(np.sum(y_fit))
    nneg = len(y_fit) - npos
    ok = npos >= min_events and nneg >= min_events
    if method == "none (source threshold)" or not ok:
        return p_eval, thr_source, False
    if method == "threshold only":
        return p_eval, best_f1_threshold(y_fit, sigmoid(z_fit)), True
    if method == "isotonic":
        iso = IsotonicRegression(out_of_bounds="clip", y_min=1e-4, y_max=1 - 1e-4).fit(z_fit, y_fit)
        return iso.predict(z_eval), best_f1_threshold(y_fit, iso.predict(z_fit)), True
    a, b = _fit_lr_1d(z_fit, y_fit, fit_slope=(method == "Platt"), fit_bias=(method in ("Platt", "bias")))
    return sigmoid(a * z_eval + b), best_f1_threshold(y_fit, sigmoid(a * z_fit + b)), True


def saerens_em(p, prior_source: float, iters: int = 300, tol: float = 1e-7):
    """Label-free prior-shift correction (Saerens et al. 2002): EM estimate of the target prevalence and adjusted
    posteriors."""
    p = np.clip(p, 1e-6, 1 - 1e-6)
    pi = prior_source
    q = p
    for _ in range(iters):
        num = p * (pi / prior_source)
        q = num / (num + (1 - p) * ((1 - pi) / (1 - prior_source)))
        new = float(np.clip(q.mean(), 1e-4, 1 - 1e-4))
        if abs(new - pi) < tol:
            break
        pi = new
    return q, pi

In [ ]:
# 4.5  Artifact registry: where every archived prediction lives. One function returns, for any (architecture, training
#      set, method, seed), the test-site probabilities, the SOURCE-validation probabilities and the provenance of each
#      array. Archived files are preferred; a run trained by this notebook is used only when no archived file exists.
FE_NAME = {"DenseNet121": "DenseNet", "ConvNeXt-T": "ConvNeXt"}
VB_SLUG = {NAT: "ERM_natural_", BAL: "ERM_balanced_", "DANN": "DANN", "CORAL": "CORAL", "GroupDRO": "GroupDRO"}


@dataclass
class PredSet:
    # site -> (N_site, L) probabilities (only sites that are clean for this model)
    test: dict = field(default_factory=dict)
    val: dict = field(default_factory=dict)  # source/other-site VALIDATION probabilities, site -> (n_val, L)
    prov: list = field(default_factory=list)  # files used

    def ok(self):
        return bool(self.test)


def _npz(path):
    return np.load(path) if Path(path).exists() else None


def _f32(a):
    return np.asarray(a, np.float32)


def load_predset(arch: str, ts: str, method: str, s: int) -> PredSet:
    ps = PredSet()
    srcs = ts.split("+")
    tag = METHOD_TAG[method]
    if arch == "ViT-S":
        f = P["vit"] / (f"vit_{tsp(ts)}_seed{s}" + ("" if method == NAT else f"_{tag}") + "_preds.npz")
        z = _npz(f)
        if z is not None:
            ps.test = {
                T: _f32(z[f"{T}_test_P"]) for T in "ABC" if f"{T}_test_P" in z and not (T == "C" and "C" in srcs)
            }
            ps.val = {k: _f32(z[f"{k}_val_P"]) for k in "ABC" if f"{k}_val_P" in z}
            ps.prov.append(str(f))
    elif arch == "ConvNeXt-T" and method != NAT:
        f = P["convnext_dg"] / f"cnx_{tsp(ts)}_seed{s}_{tag}_preds.npz"
        z = _npz(f)
        if z is not None:
            ps.test = {
                T: _f32(z[f"{T}_test_P"]) for T in "ABC" if f"{T}_test_P" in z and not (T == "C" and "C" in srcs)
            }
            ps.val = {k: _f32(z[f"{k}_val_P"]) for k in "ABC" if f"{k}_val_P" in z}
            ps.prov.append(str(f))
    elif arch in FE_NAME:
        f = P["fair_ens"] / f"logits_{FE_NAME[arch]}_{tsp(ts)}_{tag}_seed{s}.npz"
        z = _npz(f)
        if z is not None:
            ps.test = {T: _f32(sigmoid(z[T])) for T in "ABC" if T in z and not (T == "C" and "C" in srcs)}
            if "C" in srcs and "C" in z:
                ps.val["C"] = _f32(sigmoid(z["C"])[vb_va])  # VinBigData validation split was never trained on
            ps.prov.append(str(f))
        if arch == "DenseNet121" and ts == "A+B":  # archived A+B DenseNet runs (dg folder) + their VinBigData logits
            d = _npz(P["dg"] / f"{tag}_e{ARCHIVE_EPOCHS}_seed{s}_preds.npz")
            if d is not None:
                ps.val.update(A=_f32(d["nih_val_P"]), B=_f32(d["cx_val_P"]))
                ps.prov.append(str(P["dg"] / f"{tag}_e{ARCHIVE_EPOCHS}_seed{s}_preds.npz"))
                if not ps.test:
                    ps.test = {"A": _f32(d["nih_test_P"]), "B": _f32(d["cx_test_P"])}
                    fv = P["vinbig"] / f"logit_{VB_SLUG[method]}_seed{s}.npy"
                    if fv.exists():
                        ps.test["C"] = _f32(sigmoid(np.load(fv)))
                        ps.prov.append(str(fv))
        if method == NAT and ts in ("A", "B"):  # single-hospital runs: their own validation predictions
            if arch == "DenseNet121":
                f2 = (
                    P["nih_train"] / f"densenet121_nih_e{ARCHIVE_EPOCHS}_seed{s}_preds.npz"
                    if ts == "A"
                    else R / f"densenet121_e{ARCHIVE_EPOCHS}_seed{s}_results" / "preds.npz"
                )
                d = _npz(f2)
                if d is not None:
                    ps.val[ts] = _f32(d["nih_val_P"] if ts == "A" else d["val_P"])
                    ps.prov.append(str(f2))
            else:
                f2 = P["part34"] / f"convnext_tiny_{'nih' if ts == 'A' else 'cx'}_e{ARCHIVE_EPOCHS}_seed{s}_preds.npz"
                d = _npz(f2)
                if d is not None:
                    ps.val[ts] = _f32(d["val_P"])
                    ps.prov.append(str(f2))
            other = "B" if ts == "A" else "A"  # target-site validation predictions (recalibration pools)
            arch_stub = "densenet121" if arch == "DenseNet121" else "convnext_tiny"
            site_stub = "nih" if ts == "A" else "cx"
            for f3, key in [
                (P["master_v1"] / "extra" / f"extra_{arch}_{ts}_seed{s}.npz", "tgt_val"),
                (P["part34"] / f"{arch_stub}_{site_stub}_e{ARCHIVE_EPOCHS}_seed{s}_pool.npz", "pool_P"),
            ]:
                d = _npz(f3)
                if d is not None and other not in ps.val:
                    ps.val[other] = _f32(d[key])
                    ps.prov.append(str(f3))
    elif arch == CLASSICAL:
        f = NEWRUNS / "classical" / f"classical_{tsp(ts)}_erm_nat_seed{s}_preds.npz"
        z = _npz(f)
        if z is not None:
            ps.test = {T: _f32(z[f"{T}_test_P"]) for T in "ABC" if f"{T}_test_P" in z}
            ps.val = {k: _f32(z[f"{k}_val_P"]) for k in "ABC" if f"{k}_val_P" in z}
            ps.prov.append(str(f))
    if not ps.ok() and arch in ARCH_SPEC:  # a run trained (or imported, see 4.5b) by this notebook
        f = Path(str(TrainSpec(arch, ts, method, s).stem()) + "_preds.npz")
        z = _npz(f)
        if z is not None:
            ps.test = {T: _f32(z[f"{T}_test_P"]) for T in "ABC" if f"{T}_test_P" in z}
            ps.val = {k: _f32(z[f"{k}_val_P"]) for k in "ABC" if f"{k}_val_P" in z}
            ps.prov.append(str(f))
            if "source" in z.files:  # imported run: original checkpoint or predictions file
                ps.prov.append(str(z["source"]))
    for T, v in ps.test.items():
        assert v.shape == (
            len(YT[T]),
            L,
        ), f"{arch} {ts} {method} seed {s}: {T} predictions have shape {v.shape}, labels {YT[T].shape}"
    return ps


def source_thresholds(ps: PredSet, ts: str):
    """Max-F1 threshold per finding from SOURCE data only: source validation predictions; otherwise the held-out
    (patient-disjoint) test set of a SEEN source. The target site is never touched. Returns (thresholds, rule)."""
    thr = np.full(L, np.nan)
    rule = "none"
    for j in range(L):
        ys, pp = [], []
        for k in ts.split("+"):
            if not SRC[k]["valid"][j]:
                continue
            if k in ps.val:
                ys.append(VALY[k][:, j])
                pp.append(ps.val[k][:, j])
                rule = "source validation"
            elif k in ps.test and k in "AB":
                ys.append(YT[k][:, j])
                pp.append(ps.test[k][:, j])
                rule = "held-out source test" if rule == "none" else rule
        if ys and np.isfinite(np.concatenate(pp)).all():
            thr[j] = best_f1_threshold(np.concatenate(ys), np.concatenate(pp))
    return thr, rule


def load_stage3() -> tuple:
    """Stage 3 equal-budget tuning trials: plan table, per-trial histories and logits on the three test sites."""
    plan_f = P["stage3"] / "trials_plan.csv"
    if not plan_f.exists():
        return None, {}
    TR = pd.read_csv(plan_f)
    LG = {}
    for tid in TR.trial_id:
        for s in SEEDS:
            z = _npz(P["stage3"] / "logits" / f"logits_{tid}_seed{s}.npz")
            if z is not None:
                LG[(tid, s)] = {T: _f32(sigmoid(z[T])) for T in "ABC"}
    return TR, LG


def load_controls() -> dict:
    """DenseNet121 size-control (8 random 77,353-image CheXpert subsets) and source-mixture runs (fixed 77,353-image
    budget)."""
    C = {}
    for f in sorted(P["controls"].glob(f"densenet121_cx77353_e{ARCHIVE_EPOCHS}_seed*_preds.npz")):
        s = int(re.search(r"seed(\d+)_preds", f.name).group(1))
        z = np.load(f)
        d = {"A": _f32(z["nih_test_P"]), "B": _f32(z["cx_test_P"])}
        fv = P["controls"] / f"densenet121_endpoint_f100_seed{s}_vb.npy"
        if fv.exists():
            d["C"] = _f32(np.load(fv))
        C[("CX77", s)] = d
    for frac in (25, 50, 75):
        for s in SEEDS:
            z = _npz(P["controls"] / f"densenet121_mix{frac:03d}cx_n77353_e{ARCHIVE_EPOCHS}_seed{s}_preds.npz")
            if z is not None:
                C[(f"MIX{frac}", s)] = {"A": _f32(z["nih_test_P"]), "B": _f32(z["cx_test_P"]), "C": _f32(z["vb_P"])}
    return C

In [ ]:
# 4.5b DenseNet121 GroupDRO on A+C and B+C, trained in a separate notebook (folder runs/gdro_7ep_3seeds):
#      7 epochs, seeds 0-2, batch 64, AdamW lr 1e-4 / wd 1e-4, cosine, balanced batches, eta = 0.01, hardness = group
#      loss / loss of a prevalence-only predictor ("relative" hardness), best epoch on SOURCE validation AUROC.
#      The checkpoints are converted once into the registry's prediction format
#      (runs/densenet/densenet_{ApC|BpC}_gdro_seed{s}_preds.npz) so that every later analysis treats them exactly like
#      the other runs. A predictions file written by that notebook's save_preds() is converted without a GPU.
GDRO_DN = {"A+C": "B", "B+C": "A"}  # training set -> unseen test site
# per-seed values printed by the GroupDRO notebook (used when result_*.json is not available)
GDRO_DN_PRINTED = {
    ("A+C", 0): dict(mean_AUROC=0.7016, q_A=0.9685, q_C=0.0315),
    ("A+C", 1): dict(mean_AUROC=0.7152, q_A=0.9926, q_C=0.0074),
    ("A+C", 2): dict(mean_AUROC=0.7045, q_A=0.9793, q_C=0.0207),
    ("B+C", 0): dict(mean_AUROC=0.7569, q_B=0.9978, q_C=0.0022),
    ("B+C", 1): dict(mean_AUROC=0.7888, q_B=0.9981, q_C=0.0019),
    ("B+C", 2): dict(mean_AUROC=0.7607, q_B=0.9969, q_C=0.0031),
}


def import_gdro_densenet(ts: str, s: int) -> str:
    """Writes the registry-format predictions of one DenseNet121 GroupDRO run; returns how they were obtained."""
    tag, srcs = tsp(ts), ts.split("+")
    dst = Path(str(TrainSpec("DenseNet121", ts, "GroupDRO", s).stem()) + "_preds.npz")
    if dst.exists():
        return "already imported"
    ck = P["gdro_dn"] / f"densenet121_{tag}_gdro_e{ARCHIVE_EPOCHS}_seed{s}.pt"
    pf = P["gdro_dn"] / f"densenet121_{tag}_gdro_e{ARCHIVE_EPOCHS}_seed{s}_preds.npz"
    z = {}
    if HAS_TORCH and ck.exists():  # full export: both test sites (seen and unseen) and all validation sets
        m = load_checkpoint("DenseNet121", ck)
        for T in "AB":
            z[f"{T}_test_P"] = sigmoid(predict_logits(m, TEST[T][0], TEST[T][1], "DenseNet121"))
        for k in sorted(set("AB") | set(srcs)):
            z[f"{k}_val_P"] = sigmoid(predict_logits(m, SRC[k]["arr"], SRC[k]["va"], "DenseNet121"))
        del m
        gc.collect()
        if DEV == "cuda":
            torch.cuda.empty_cache()
        source, how = ck, "checkpoint"
    elif pf.exists():  # save_preds() format: unseen-site test predictions and source validation predictions
        with np.load(pf) as d:
            z[f"{GDRO_DN[ts]}_test_P"] = d["test_P"]
            for k in srcs:
                z[f"{k}_val_P"] = d[f"{k}_val_P"]
        source, how = pf, "predictions file"
    else:
        return "checkpoint only: needs PyTorch" if ck.exists() else "not found"
    for k_, v_ in z.items():
        T_ = k_[0]
        n_ = len(YT[T_]) if k_.endswith("_test_P") else len(SRC[T_]["va"])
        assert np.shape(v_) == (n_, L), f"{pf.name}: {k_} has shape {np.shape(v_)}, expected {(n_, L)}"
    dst.parent.mkdir(parents=True, exist_ok=True)
    np.savez(dst, source=np.array(str(source)), **{k_: np.asarray(v_, np.float32) for k_, v_ in z.items()})
    return how


GDRO_DN_RUNS = []  # one row per run: provenance, final hospital weights q, printed AUROC
for ts, T in GDRO_DN.items():
    for s in SEEDS:
        how = import_gdro_densenet(ts, s)
        row = dict(train=ts, test=T, seed=s, imported_from=how, **GDRO_DN_PRINTED.get((ts, s), {}))
        rj = P["gdro_dn"] / f"result_{tsp(ts)}_seed{s}.json"
        if rj.exists():
            row.update({k_: v_ for k_, v_ in read_json(rj).items() if k_ == "mean_AUROC" or k_.startswith("q_")})
        hf = P["gdro_dn"] / f"hist_{tsp(ts)}_seed{s}.csv"
        if hf.exists():
            h_ = pd.read_csv(hf)
            row.update({c: float(h_[c].iloc[-1]) for c in h_.columns if c.startswith("q_")})
            row["best_epoch"] = int(h_.epoch.iloc[int(h_.val_mean.values.argmax())])
        GDRO_DN_RUNS.append(row)
GDRO_DN_RUNS = pd.DataFrame(GDRO_DN_RUNS)
display(GDRO_DN_RUNS)
if (~GDRO_DN_RUNS.imported_from.isin(["checkpoint", "predictions file", "already imported"])).any():
    print(
        f"some DenseNet121 GroupDRO runs could not be imported from {P['gdro_dn']} (see 'imported_from'): "
        "these cells stay empty in the grid."
    )

In [ ]:
# 4.6  The full experimental grid, what the archive already covers, and (optionally) training of whatever is missing.
GRID = [(a, ts, NAT, s) for a in ARCHS for ts in TRAIN_SETS for s in SEEDS] + [
    (a, ts, m_, s) for a in ARCHS for ts in POOLED for m_ in METHODS[1:] for s in SEEDS
]
with timed("4 registry scan"):
    PS = {}
    for key in tqdm(GRID, desc="scanning archived predictions"):
        ps_ = load_predset(*key)
        if ps_.ok():
            PS[key] = ps_
    for ts in CLASSICAL_SETS:
        for s in SEEDS:
            ps_ = load_predset(CLASSICAL, ts, NAT, s)
            if ps_.ok():
                PS[(CLASSICAL, ts, NAT, s)] = ps_
cov = pd.DataFrame([dict(arch=a, train=ts, method=m_, seed=s, found=(a, ts, m_, s) in PS) for a, ts, m_, s in GRID])
COVERAGE = cov.groupby(["train", "method", "arch"]).found.sum().unstack("arch").reindex(columns=ARCHS)
MISSING = cov[~cov.found][["arch", "train", "method", "seed"]]
print(
    f"archived/new predictions found for {int(cov.found.sum())} of {len(cov)} deep-model runs; missing: {len(MISSING)}"
)
if len(MISSING):
    display(MISSING.groupby(["arch", "train", "method"]).seed.apply(list).to_frame("missing seeds"))

if CFG.run_training and HAS_TORCH and len(MISSING):
    est_h = len(MISSING) * {"cuda": 0.6}.get(DEV, 20.0)
    print(f"training {len(MISSING)} missing runs (rough estimate {est_h:.0f} GPU-hours); every run is resumable")
    for _, r in MISSING.iterrows():
        try:
            train_run(TrainSpec(r.arch, r.train, r.method, int(r.seed)))
            PS[(r.arch, r.train, r.method, int(r.seed))] = load_predset(r.arch, r.train, r.method, int(r.seed))
        except torch.cuda.OutOfMemoryError:
            print(
                "GPU out of memory: lower Config.batch_size, restart the kernel and rerun (finished runs are skipped)."
            )
            break
elif len(MISSING):
    print(
        "CFG.run_training is False: missing configurations are reported, not "
        "trained; every analysis uses the available runs."
    )

if PROFILE == "smoke" and HAS_TORCH:  # trainer self-test: every method once, tiny subsets, 1 epoch
    for a_, ts_, m_ in [
        ("DenseNet121", "A", NAT),
        ("ConvNeXt-T", "A+B", "DANN"),
        ("ViT-S", "A+B", "CORAL"),
        ("DenseNet121", "A+C", "GroupDRO"),
    ]:
        f_ = train_run(
            TrainSpec(
                a_,
                ts_,
                m_,
                0,
                subset={k: 32 for k in ts_.split("+")},
                tag="selftest",
                dro_hardness="relative",
                eval_limit=48,
            )
        )
        z_ = np.load(f_)
        assert all(z_[k].shape[1] == L for k in z_.files), f_
    print("trainer self-test passed for ERM, DANN, CORAL and GroupDRO on all three architectures")

In [ ]:
# 4.7  OPTIONAL: repeated patient-grouped K-fold RETRAINING of the three deep architectures on NIH (A).
#      Off by default (CFG.run_kfold_retrain): K x R x 3 runs of KFOLD_EPOCHS epochs (about 2.5 GPU-hours per
#      repetition on an RTX 4070 SUPER with 3 epochs). When results exist, the internal held-out AUROC gets a
#      Nadeau-Bengio corrected CI and the architectures are compared with corrected resampled paired t-tests on the
#      SAME folds; external AUROC of every fold model is reported.
KFOLD_SOURCE = "A"
KFOLD_K, KFOLD_R, KFOLD_EPOCHS = (5, 2, 3) if PROFILE != "smoke" else (2, 1, 1)  # smoke: pipeline check only


def kfold_specs(arch: str):
    tr = SRC[KFOLD_SOURCE]["tr"]
    Y = SRC[KFOLD_SOURCE]["Y"][tr]
    pid = SRC[KFOLD_SOURCE]["pid"][tr]
    for rep, fold, a, b in repeated_group_kfold(
        np.minimum(Y.sum(1), 2), pid, KFOLD_K, KFOLD_R, f"kfold-retrain-{KFOLD_SOURCE}"
    ):
        yield rep, fold, TrainSpec(
            arch,
            KFOLD_SOURCE,
            NAT,
            seed=rep,
            epochs=KFOLD_EPOCHS,
            train_idx={KFOLD_SOURCE: np.sort(tr[a])},
            tag=f"kfold_r{rep}f{fold}",
        )


if CFG.run_kfold_retrain and HAS_TORCH:
    for a_ in ARCHS:
        for rep, fold, spec in kfold_specs(a_):
            train_run(spec)

KF_rows = []
for a_ in ARCHS:
    for rep, fold, spec in kfold_specs(a_):
        f_ = Path(str(spec.stem()) + "_preds.npz")
        if not f_.exists():
            continue
        z = np.load(f_)
        ho = z[f"{KFOLD_SOURCE}_heldout_idx"]
        Ph = z[f"{KFOLD_SOURCE}_heldout_P"]
        Yh = SRC[KFOLD_SOURCE]["Y"][ho]
        row = dict(
            arch=a_,
            rep=rep,
            fold=fold,
            n_fit=len(spec.train_idx[KFOLD_SOURCE]),
            n_test=len(ho),
            internal=np.nanmean([auroc(Yh[:, j], Ph[:, j]) for j in range(L) if 0 < Yh[:, j].sum() < len(Yh)]),
        )
        for T in "ABC":
            if not is_seen(KFOLD_SOURCE, T) and f"{T}_test_P" in z:
                row[f"external_{T}"] = np.nanmean([auroc(YT[T][:, j], z[f"{T}_test_P"][:, j]) for j in FIND_SITE[T]])
        KF_rows.append(row)
KFOLD = pd.DataFrame(KF_rows)
if len(KFOLD):
    summ = []
    for a_, g in KFOLD.groupby("arch"):
        summ.append(
            dict(
                arch=a_,
                evaluation="internal (held-out folds)",
                **corrected_t_ci(g.internal, g.n_fit.mean(), g.n_test.mean()),
            )
        )
        for c_ in [c for c in KFOLD.columns if c.startswith("external")]:
            v = g[c_].dropna().values
            summ.append(
                dict(
                    arch=a_,
                    evaluation=f"external {SHORT[c_[-1]]} (fold models)",
                    mean=v.mean(),
                    ci_low=np.percentile(v, 2.5),
                    ci_high=np.percentile(v, 97.5),
                    sd=v.std(ddof=1) if len(v) > 1 else np.nan,
                    J=len(v),
                )
            )
    save_tab(
        pd.DataFrame(summ),
        "table09_kfold_retraining",
        "Deep models, repeated patient-grouped K-fold retraining on NIH. Internal: Nadeau-Bengio corrected 95% CI; "
        "external: 2.5-97.5 percentile range over the fold models (fixed external test set).",
        index=False,
    )
    piv = KFOLD.pivot_table(index=["rep", "fold"], columns="arch", values="internal")
    cmp_ = [
        dict(A=a_, B=b_, **corrected_paired_t(piv[a_] - piv[b_], KFOLD.n_fit.mean(), KFOLD.n_test.mean()))
        for a_, b_ in itertools.combinations(piv.columns, 2)
    ]
    save_tab(
        pd.DataFrame(cmp_).assign(p_holm=lambda d: holm(d.p.values)),
        "table10_kfold_architecture_tests",
        "Corrected resampled paired t-tests between architectures (internal held-out AUROC).",
        index=False,
    )
else:
    print(
        "no K-fold retraining results yet (set CFG.run_kfold_retrain = True; "
        f"plan: {KFOLD_K} folds x {KFOLD_R} repeats x {len(ARCHS)} architectures x {KFOLD_EPOCHS} epochs)"
    )

In [ ]:
# 5.1  Coverage of the experimental grid and provenance of every prediction file
with timed("5 coverage"):
    prov = pd.DataFrame(
        [
            dict(
                arch=k[0],
                train=k[1],
                method=k[2],
                seed=k[3],
                sites=",".join(sorted(v.test)),
                val=",".join(sorted(v.val)),
                files=" | ".join(Path(p_).name for p_ in v.prov),
            )
            for k, v in PS.items()
        ]
    )
    prov.to_csv(TAB / "provenance_predictions.csv", index=False)
    fig, ax = plt.subplots(figsize=(COL2, 3.4))
    C_ = COVERAGE.copy()
    C_.index = [f"{t} | {m}" for t, m in C_.index]
    sns.heatmap(
        C_,
        annot=True,
        fmt=".0f",
        cmap="Greens",
        vmin=0,
        vmax=len(SEEDS),
        cbar=False,
        linewidths=0.3,
        linecolor="w",
        ax=ax,
        annot_kws=dict(fontsize=6),
    )
    ax.set(xlabel="", ylabel="")
    ax.tick_params(axis="y", labelsize=6)
    fig.tight_layout()
    save_fig(
        fig,
        "figS01_coverage",
        f"Seeds available per configuration (maximum {len(SEEDS)}); provenance "
        "of every array in provenance_predictions.csv.",
    )
KEYS = sorted(
    {k[:3] for k in PS}, key=lambda k: ([*ARCHS, CLASSICAL].index(k[0]), TRAIN_SETS.index(k[1]), METHODS.index(k[2]))
)
SEEDS_OF = {k: [s for s in SEEDS if (*k, s) in PS] for k in KEYS}
print(
    f"{len(KEYS)} model families (architecture x training set x method) with "
    f"{sum(map(len, SEEDS_OF.values()))} trained models"
)

In [ ]:
# 5.2  Per-model, per-site, per-finding metric table (all trained models). Thresholds come from SOURCE data only.
def model_findings(ts: str, T: str):
    """Findings evaluated for a model trained on `ts` at site T: labelled at T AND seen during training."""
    mv = source_union_valid(ts)
    return [
        j for j in range(L) if VALID[T][j] and mv[j] and min(YT[T][:, j].sum(), len(YT[T]) - YT[T][:, j].sum()) >= 5
    ]


with timed("5 metric table"):
    rows, THR = [], {}
    for (a, ts, m_, s), ps in tqdm(PS.items(), desc="metrics"):
        thr, rule = source_thresholds(ps, ts)
        THR[(a, ts, m_, s)] = (thr, rule)
        for T, Pm in ps.test.items():
            for j in model_findings(ts, T):
                y, p = YT[T][:, j], Pm[:, j]
                r = dict(
                    arch=a,
                    train=ts,
                    method=m_,
                    seed=s,
                    test=T,
                    seen=is_seen(ts, T),
                    finding=LABELS[j],
                    threshold=thr[j],
                    threshold_rule=rule,
                    **probability_metrics(y, p),
                )
                if np.isfinite(thr[j]):
                    r.update(binary_report(y, p, thr[j]))
                rows.append(r)
    MT = pd.DataFrame(rows)
    MT.to_csv(TAB / "metrics_per_model_site_finding.csv", index=False)
    METRIC_COLS = [
        "auroc",
        "auprc",
        "sens_at_90spec",
        "precision",
        "recall",
        "specificity",
        "f1",
        "balanced_accuracy",
        "mcc",
        "brier",
        "brier_skill",
        "rmse",
        "mae",
        "log_loss",
        "ece",
        "ace",
        "cal_intercept",
        "cal_slope",
    ]
    # macro over findings
    MS = (
        MT.groupby(["arch", "train", "method", "seed", "test", "seen"])[[c for c in METRIC_COLS if c in MT]]
        .mean()
        .reset_index()
    )
    MM = MS.groupby(["arch", "train", "method", "test", "seen"])[[c for c in METRIC_COLS if c in MS]].agg(
        ["mean", "std"]
    )
print(
    f"{len(MT)} rows: {MT.arch.nunique()} model types x {MT.train.nunique()} "
    f"training sets x {MT.method.nunique()} methods x seeds x sites x findings"
)
display(MT.threshold_rule.value_counts().to_frame("models x findings"))

In [ ]:
# 5.3  Main benchmark: transfer matrices (ERM natural) for the three deep architectures and the classical baseline,
#      and the full multi-metric table for the unseen-site directions.
ARCH_ALL = [a for a in [*ARCHS, CLASSICAL] if a in MS.arch.unique()]
with timed("5 transfer matrices"):
    nat = MS[MS.method == NAT]
    rows = []
    for (a, ts, T), g in nat.groupby(["arch", "train", "test"]):
        rows.append(
            dict(
                arch=a,
                train=ts,
                test=T,
                seen=is_seen(ts, T),
                auroc=g.auroc.mean(),
                auroc_sd=g.auroc.std(ddof=1) if len(g) > 1 else 0.0,
                auprc=g.auprc.mean(),
                brier=g.brier.mean(),
                ece=g.ece.mean(),
                seeds=len(g),
            )
        )
    TM = pd.DataFrame(rows)
    disp = TM.assign(cell=TM.apply(lambda r: f"{r.auroc:.3f} ± {r.auroc_sd:.3f}" + (" †" if r.seen else ""), axis=1))
    TAB_TM = disp.pivot_table(index="train", columns=["arch", "test"], values="cell", aggfunc="first").reindex(
        index=TRAIN_SETS
    )
    TAB_TM = TAB_TM.reindex(columns=[(a, t) for a in ARCH_ALL for t in "ABC" if (a, t) in TAB_TM.columns])
    save_tab(
        TAB_TM,
        "table11_transfer_matrix_auroc",
        "Macro AUROC (mean ± SD over seeds) of ERM (natural) models; rows = training set, "
        "columns = test site; † = site seen in training; VinBigData (C) column "
        "= 5 findings. Models trained with C are not tested on C.",
    )

    fig, axes = plt.subplots(1, len(ARCH_ALL), figsize=(COL2, 2.6), sharey=True)
    axes = np.atleast_1d(axes)
    vmin, vmax = TM.auroc.min() - 0.01, TM.auroc.max() + 0.005
    for ax, a in zip(axes, ARCH_ALL):
        M_ = (
            TM[TM.arch == a]
            .pivot(index="train", columns="test", values="auroc")
            .reindex(index=TRAIN_SETS, columns=list("ABC"))
        )
        sns.heatmap(
            M_,
            ax=ax,
            cmap="viridis",
            vmin=vmin,
            vmax=vmax,
            cbar=ax is axes[-1],
            annot=False,
            linewidths=0.4,
            linecolor="w",
            cbar_kws=dict(label="Macro AUROC", shrink=0.8),
        )
        for i, ts in enumerate(TRAIN_SETS):
            for j, T in enumerate("ABC"):
                v = M_.loc[ts, T]
                if np.isfinite(v):
                    ax.text(
                        j + 0.5,
                        i + 0.5,
                        f"{v:.3f}" + ("†" if is_seen(ts, T) else ""),
                        ha="center",
                        va="center",
                        fontsize=5.5,
                        color="w" if v < (vmin + vmax) / 2 else "k",
                    )
        ax.set(title=a, xlabel="Test site", ylabel="Training set" if ax is axes[0] else "")
        ax.set_xticklabels([SHORT[t] for t in "ABC"], rotation=30, ha="right", fontsize=6)
    fig.tight_layout()
    save_fig(
        fig,
        "fig05_transfer_matrices",
        "Transfer matrices (macro AUROC, mean over seeds) for three deep architectures "
        "and the classical baseline; † seen site.",
    )

    ext = MS[~MS.seen].copy()
    ext["direction"] = ext.train + "→" + ext.test
    cols = [c for c in METRIC_COLS if c in ext]
    extp = ext[ext.train.isin(POOLED)]  # pooled -> unseen directions: the only ones on which every method was run
    BENCH = extp.groupby(["direction", "arch", "method"])[cols].mean()
    BENCH_SD = extp.groupby(["direction", "arch", "method"])[cols].std(ddof=1).reindex(BENCH.index)
    BENCH_N = extp.groupby(["direction", "arch", "method"]).seed.nunique().reindex(BENCH.index)

    def _pm(m_, s_):
        return "n/a" if not np.isfinite(m_) else (f"{m_:.4f} ± {s_:.4f}" if np.isfinite(s_) else f"{m_:.4f}")

    BT = pd.DataFrame(
        {c: [_pm(BENCH.loc[i, c], BENCH_SD.loc[i, c]) for i in BENCH.index] for c in cols}, index=BENCH.index
    )
    BT.insert(0, "seeds", BENCH_N)
    order_ = [
        (d_, a, m_)
        for d_ in [dname(ts, UNSEEN[ts]) for ts in POOLED]
        for a in ARCH_ALL
        for m_ in METHODS
        if (d_, a, m_) in BENCH.index
    ]
    BT = BT.reindex(order_)
    save_tab(
        BT,
        "table12_external_benchmark_all_metrics",
        "Unseen-site performance of every model family on each pooled -> unseen direction (A+B→C, A+C→B, B+C→A; "
        "mean ± SD over seeds; the classical baseline exists for A+B→C only). Threshold metrics at source-selected "
        "max-F1 thresholds; Brier/RMSE/MAE/log-loss = probabilistic (regression-type) error; "
        "cal_intercept/cal_slope = logistic recalibration.",
    )
    fig, ax = plt.subplots(figsize=(COL2, 2.9))
    w = 0.8 / len(ARCH_ALL)
    for i, a in enumerate(ARCH_ALL):
        g = ext[(ext.arch == a) & (ext.method == NAT)]
        dirs = [dname(*d) for d in EXT_DIRS]
        mu = g.groupby("direction").auroc.mean().reindex(dirs)
        ax.bar(np.arange(len(dirs)) + (i - (len(ARCH_ALL) - 1) / 2) * w, mu, w, color=ARCH_COL[a], label=a)
        for k_, d_ in enumerate(dirs):
            v = g[g.direction == d_].auroc.values
            ax.scatter(np.full(len(v), k_ + (i - (len(ARCH_ALL) - 1) / 2) * w), v, s=3, color="k", zorder=3)
    ax.set_xticks(range(len(EXT_DIRS)))
    ax.set_xticklabels([dname(*d) for d in EXT_DIRS], rotation=35, ha="right")
    ax.set_ylim(max(0.45, ext.auroc.min() - 0.05), None)
    ax.set_ylabel("Macro AUROC, unseen site")
    ax.legend(ncol=4, fontsize=6)
    fig.tight_layout()
    save_fig(
        fig,
        "fig06_external_auroc_by_direction",
        "Unseen-site macro AUROC per transfer direction (bars = mean, dots = individual seeds).",
    )

In [ ]:

# 5.4  Performance deltas:
#      (a) generalisation gap of every external direction relative to the in-hospital reference model,
#      (b) architecture deltas relative to DenseNet121,
#      (c) DG-method deltas relative to ERM (natural) and ERM (balanced).
#      Seed-level means here are descriptive; their inferential versions (paired bootstrap, Holm) are in Section 6.
ALGO = pd.DataFrame(
    [
        dict(
            method=NAT,
            sampling="proportional to hospital size",
            extra_objective="none",
            extra_parameters="none",
            hyperparameters="-",
            reference="empirical risk minimisation",
        ),
        dict(
            method=BAL,
            sampling="equal images per hospital per step",
            extra_objective="none",
            extra_parameters="none",
            hyperparameters="-",
            reference="sampling control for every DG method",
        ),
        dict(
            method="DANN",
            sampling="balanced",
            extra_objective="+ lambda * domain cross-entropy through a gradient-reversal layer",
            extra_parameters="2-layer domain discriminator (feature dim x 256 + 256 x 2)",
            hyperparameters="lambda = 0.1, GRL ramp 2/(1+exp(-10p)) - 1",
            reference="Ganin et al. 2016",
        ),
        dict(
            method="CORAL",
            sampling="balanced",
            extra_objective="+ lambda * ||C_A - C_B||_F^2 / (4 d^2) on pre-logit features",
            extra_parameters="none",
            hyperparameters="lambda = 1.0",
            reference="Sun & Saenko 2016",
        ),
        dict(
            method="GroupDRO",
            sampling="balanced",
            extra_objective="q-weighted hospital losses, q <- q * exp(eta * loss)",
            extra_parameters="none (2 group weights)",
            hyperparameters="eta = 0.01, hardness = raw group loss (DenseNet121 A+C, B+C: loss / prevalence-only loss)",
            reference="Sagawa et al. 2020",
        ),
    ]
).set_index("method")
save_tab(
    ALGO,
    "table13_algorithmic_differences",
    "Algorithmic differences between the training methods (all other settings identical).",
)

with timed("5 deltas"):
    ref_in = {
        (a, T): MS[(MS.arch == a) & (MS.train == T) & (MS.test == T) & (MS.method == NAT)].auroc.mean()
        for a in ARCH_ALL
        for T in "AB"
    }
    rows = []
    for (a, ts, m_, T), g in MS[~MS.seen].groupby(["arch", "train", "method", "test"]):
        if T in "AB" and np.isfinite(ref_in.get((a, T), np.nan)):
            rows.append(
                dict(
                    arch=a,
                    method=m_,
                    direction=dname(ts, T),
                    in_hospital=ref_in[(a, T)],
                    external=g.auroc.mean(),
                    gap=ref_in[(a, T)] - g.auroc.mean(),
                )
            )
    GAPS = pd.DataFrame(rows)
    save_tab(
        GAPS[GAPS.method == NAT].pivot(index="direction", columns="arch", values="gap").round(4),
        "table14_generalisation_gaps",
        "Generalisation gap = AUROC of the target hospital's own model on its "
        "test set minus AUROC of the transferred model (ERM natural, seed means).",
    )
    base = MS[(MS.arch == "DenseNet121") & (MS.method == NAT)].groupby(["train", "test"]).auroc.mean()
    AD = (MS[MS.method == NAT].groupby(["arch", "train", "test"]).auroc.mean().unstack("arch").sub(base, axis=0)).drop(
        columns="DenseNet121", errors="ignore"
    )
    AD.index = [dname(ts, T) + (" (seen)" if is_seen(ts, T) else "") for ts, T in AD.index]
    save_tab(
        AD.round(4),
        "table15_architecture_deltas",
        "AUROC difference to DenseNet121 per direction (ERM natural, seed means).",
    )
    rows = []
    for (a, ts), g in MS[(~MS.seen) & MS.train.isin(POOLED)].groupby(["arch", "train"]):
        mu = g.groupby("method").auroc.mean()
        for m_ in mu.index:
            rows.append(
                dict(
                    arch=a,
                    direction=dname(ts, UNSEEN[ts]),
                    method=m_,
                    auroc=mu[m_],
                    delta_vs_natural=mu[m_] - mu.get(NAT, np.nan),
                    delta_vs_balanced=mu[m_] - mu.get(BAL, np.nan),
                )
            )
    DGD = pd.DataFrame(rows)
    fig, ax = plt.subplots(1, 2, figsize=(COL2, 3.8), gridspec_kw=dict(width_ratios=[1, 1.35]))
    deep_ = [c for c in AD.columns if c != CLASSICAL]  # colour scale from the deep models; classical cells saturate
    v = np.nanmax(np.abs(AD[deep_].values)) if AD.size and deep_ else 0.05
    sns.heatmap(
        AD,
        ax=ax[0],
        cmap="RdBu",
        center=0,
        vmin=-v,
        vmax=v,
        annot=True,
        fmt="+.3f",
        annot_kws=dict(fontsize=5.5),
        cbar=False,
        linewidths=0.3,
    )
    ax[0].set_title("Δ AUROC vs DenseNet121 (ERM natural)")
    ax[0].set(xlabel="", ylabel="")
    ax[0].tick_params(axis="y", labelsize=6)
    ax[0].tick_params(axis="x", labelsize=6, rotation=20)
    panel_label(ax[0], "a")
    piv = (
        DGD[DGD.method != NAT]
        .pivot_table(index=["arch", "direction"], columns="method", values="delta_vs_natural")
        .reindex(columns=METHODS[1:])
    )
    v2 = np.nanmax(np.abs(piv.values)) if piv.size else 0.05
    sns.heatmap(
        piv,
        ax=ax[1],
        cmap="RdBu",
        center=0,
        vmin=-v2,
        vmax=v2,
        annot=True,
        fmt="+.3f",
        annot_kws=dict(fontsize=5.5),
        cbar=False,
        linewidths=0.3,
    )
    ax[1].set_title("Δ unseen-site AUROC vs ERM (natural)")
    ax[1].set(xlabel="", ylabel="")
    ax[1].tick_params(axis="y", labelsize=6)
    panel_label(ax[1], "b")
    fig.tight_layout()
    save_fig(
        fig,
        "fig07_performance_deltas",
        "Performance deltas (seed means): (a) architectures relative to DenseNet121; "
        "(b) DG methods relative to ERM (natural).",
    )

In [ ]:
# 5.5  Computational complexity: model size, FLOPs, measured throughput/memory, recorded training time, and
#      accuracy-cost trade-off
REFERENCE_COST = {
    "DenseNet121": dict(params_M=6.96, gflops=5.74),
    "ConvNeXt-T": dict(params_M=27.8, gflops=8.94),
    "ViT-S": dict(params_M=21.7, gflops=9.21),
}  # fallback when PyTorch is absent (timm model cards, 224 px, 2 FLOPs per MAC)
with timed("5 complexity"):
    rows = []
    for a in ARCHS:
        r = dict(arch=a, **REFERENCE_COST[a], source="reference values")
        if HAS_TORCH:
            m = make_model(a, False).eval()
            r["params_M"] = sum(p_.numel() for p_ in m.parameters()) / 1e6
            r["source"] = "measured"
            try:
                from torch.utils.flop_counter import FlopCounterMode

                with FlopCounterMode(display=False) as fc, torch.inference_mode():
                    m(torch.zeros(1, 3, CFG.image_size, CFG.image_size, device=DEV))
                r["gflops"] = fc.get_total_flops() / 1e9
            except Exception as e:
                print(f"FLOP counter unavailable for {a}: {type(e).__name__}")
            if CFG.run_complexity_benchmark:
                bs_inf = 128 if DEV == "cuda" else 4
                x = torch.randn(bs_inf, 3, CFG.image_size, CFG.image_size, device=DEV)
                n_it = 20 if DEV == "cuda" else 2
                with torch.inference_mode(), autocast():
                    for _ in range(3 if DEV == "cuda" else 1):
                        m(x)
                    if DEV == "cuda":
                        torch.cuda.synchronize()
                    t0 = time.perf_counter()
                    for _ in range(n_it):
                        m(x)
                    if DEV == "cuda":
                        torch.cuda.synchronize()
                r["throughput_img_s"] = n_it * bs_inf / (time.perf_counter() - t0)
                r["latency_ms_per_img"] = 1000 / r["throughput_img_s"]
                bs_tr = CFG.batch_size if DEV == "cuda" else 4
                m.train()
                opt_ = torch.optim.AdamW(m.parameters(), 1e-4)
                if DEV == "cuda":
                    torch.cuda.reset_peak_memory_stats()
                xb = torch.randn(bs_tr, 3, CFG.image_size, CFG.image_size, device=DEV)
                t0 = time.perf_counter()
                with autocast():
                    loss_ = m(xb).float().mean()
                loss_.backward()
                opt_.step()
                if DEV == "cuda":
                    torch.cuda.synchronize()
                    r["train_peak_GB"] = torch.cuda.max_memory_allocated() / 1e9
                r["train_step_s"] = time.perf_counter() - t0
                r["device"] = DEV
                del opt_, xb, x
            del m
            gc.collect()
        rows.append(r)
    CX = pd.DataFrame(rows).set_index("arch")

    def _classify_history(p: Path):
        n = p.name.lower()
        s = str(p).lower()
        arch = (
            "ViT-S"
            if n.startswith("vit")
            else ("ConvNeXt-T" if ("convnext" in n or n.startswith("cnx") or "convnext" in s) else "DenseNet121")
        )
        if "stage3" in s:
            meth = next(
                (
                    m_
                    for m_, k in [("DANN", "dann_"), ("CORAL", "coral_"), ("GroupDRO", "groupdro"), (NAT, "erm_")]
                    if n.startswith(k)
                ),
                NAT,
            )
        else:
            meth = next((m_ for m_, t in METHOD_TAG.items() if f"_{t}" in n or n.startswith(t + "_")), NAT)
        return arch, meth, p.parent.name

    hrows = []
    for p in sorted(R.glob("**/*_history.csv")):
        if "selftest" in p.name or f"{CFG.out_name}_export" in str(p):  # trainer self-test runs are not study runs
            continue
        try:
            h = pd.read_csv(p)
        except Exception:
            continue
        col = next((c for c in ["sec", "minutes", "min"] if c in h.columns), None)
        if col is None or len(h) == 0:
            continue
        sec = h[col].astype(float).values * (60 if col != "sec" else 1)
        a, m_, fam = _classify_history(p)
        hrows.append(
            dict(
                file=p.name,
                folder=fam,
                arch=a,
                method=m_,
                epochs=len(h),
                total_h=sec.sum() / 3600,
                min_per_epoch=np.median(sec) / 60,
            )
        )
    HIST = pd.DataFrame(hrows)
    if len(HIST):
        COST = HIST.groupby(["arch", "method"]).agg(
            runs=("file", "count"),
            epochs=("epochs", "sum"),
            gpu_hours=("total_h", "sum"),
            median_min_per_epoch=("min_per_epoch", "median"),
        )
        save_tab(
            COST.round(2),
            "table16_training_cost",
            f"Recorded training cost from {len(HIST)} history files "
            "(wall-clock GPU hours on the training workstation).",
        )
        CX["recorded_gpu_hours"] = HIST.groupby("arch").total_h.sum().reindex(CX.index)
    ext_nat = MS[(~MS.seen) & (MS.method == NAT)].groupby("arch").auroc.mean()
    CX["external_auroc_ERM_natural"] = ext_nat.reindex(CX.index)
    save_tab(
        CX.round(3),
        "table17_model_complexity",
        "Model complexity (parameters, GFLOPs per 224-px image), measured throughput / memory on this machine "
        "and mean unseen-site AUROC over the 9 external directions.",
    )
    fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.6))
    for a in CX.index:
        r = CX.loc[a]
        ax[0].scatter(
            r.gflops,
            r.external_auroc_ERM_natural,
            s=20 + 6 * r.params_M,
            color=ARCH_COL[a],
            edgecolor="k",
            lw=0.4,
            label=a,
            zorder=3,
        )
        if "throughput_img_s" in CX and np.isfinite(r.get("throughput_img_s", np.nan)):
            ax[1].scatter(
                r.throughput_img_s,
                r.external_auroc_ERM_natural,
                s=20 + 6 * r.params_M,
                color=ARCH_COL[a],
                edgecolor="k",
                lw=0.4,
                zorder=3,
            )
    ax[0].set(xlabel="GFLOPs per image (224 px)", ylabel="Unseen-site AUROC (mean of 9 directions)")
    ax[0].legend(fontsize=6)
    panel_label(ax[0], "a")
    ax[1].set(xlabel=f"Inference throughput, images/s ({DEV})", ylabel="")
    panel_label(ax[1], "b")
    for a_ in ax:
        a_.grid(alpha=0.3)
    fig.tight_layout()
    save_fig(
        fig,
        "fig08_accuracy_vs_cost",
        "Accuracy-cost trade-off: unseen-site AUROC against compute (a) and "
        "measured throughput (b); marker area ~ parameters.",
    )

In [ ]:
# 6.1  Paired two-level cluster bootstrap of every model family on every test site it may be evaluated on
with timed("6 bootstrap (all model families)"):
    BOOT = ClusterBootstrap(GID, CFG.n_boot, CFG.boot_chunk)
    BMET = ("auroc", "auprc", "sens90") if CFG.secondary_boot_metrics else ("auroc",)
    BR = {}
    for k in tqdm(KEYS, desc="cluster bootstrap"):
        a, ts, m_ = k
        for T in sorted({T for s in SEEDS_OF[k] for T in PS[(*k, s)].test}):
            probs = {s: PS[(*k, s)].test[T] for s in SEEDS_OF[k] if T in PS[(*k, s)].test}
            BR[(k, T)] = bootstrap_cell(k, T, probs, metrics=BMET, allowed=model_findings(ts, T))

    # consistency: bootstrap point estimates must equal the metric table
    # (tolerance allows for float32 / implementation-level rounding differences)
    POINT_TOL = 1e-4
    chk = []
    for (k, T), r in list(BR.items())[:25]:
        a, ts, m_ = k
        ref = MS[(MS.arch == a) & (MS.train == ts) & (MS.method == m_) & (MS.test == T)].auroc.mean()
        chk.append(abs(macro_point(r) - ref))
    chk = np.asarray(chk, dtype=float)
    assert np.isfinite(chk).all(), "NaN in bootstrap-vs-table consistency check (missing metric-table rows?)"
    assert chk.max() < POINT_TOL, (
        f"bootstrap point estimates disagree with the metric table (max diff {chk.max():.2e})"
    )

    rows = []
    for (k, T), r in BR.items():
        for met in BMET:
            d = macro_draws(r, met)
            lo, hi = np.nanpercentile(d, [2.5, 97.5])
            rows.append(
                dict(
                    arch=k[0],
                    train=k[1],
                    method=k[2],
                    test=T,
                    seen=is_seen(k[1], T),
                    metric=met,
                    n_seeds=len(r.seeds),
                    n_findings=len(r.findings),
                    estimate=macro_point(r, met),
                    ci_low=lo,
                    ci_high=hi,
                    boot_se=np.nanstd(d, ddof=1),
                )
            )
    CI_ALL = pd.DataFrame(rows)
    CI_ALL.to_csv(TAB / "table18b_all_models_bootstrap_ci.csv", index=False)

    show = CI_ALL[(CI_ALL.metric == "auroc") & (CI_ALL.method == NAT)].copy()
    show["AUROC [95% CI]"] = show.apply(lambda r: fmt_ci(r.estimate, r.ci_low, r.ci_high), axis=1)
    save_tab(
        show.pivot_table(index=["arch", "train"], columns="test", values="AUROC [95% CI]", aggfunc="first")
        .reindex([(a, t) for a in ARCH_ALL for t in TRAIN_SETS])
        .dropna(how="all"),
        "table18_auroc_ci_erm_natural",
        f"Macro AUROC with 95% two-level cluster-bootstrap CI (B = {CFG.n_boot}; "
        "patients and seeds resampled), ERM (natural).",
    )

print(
    f"{len(BR)} (model family x site) cells bootstrapped; point estimates "
    f"agree with the metric table (max |diff| = {chk.max():.1e})"
)


In [ ]:
# 6.2  Pre-specified contrast families (paired bootstrap; Holm within family, BH across the whole study)
#   F1  transfer gaps and the NIH->CheXpert vs CheXpert->NIH ASYMMETRY, per architecture
#       (6 findings, both sites labelled)
#   F2  architecture differences per direction (ERM natural), all 15 directions with a clean test set
#   F3  does adding a second hospital help? pooled vs single-source on the unseen site and on the seen sites
HYP = []  # every pre-specified contrast of the study ends up here
B6 = list(range(L))
with timed("6 contrasts: transfer"):
    GAP_DRAWS = {}
    for a in ARCH_ALL:
        need = [((a, "A", NAT), "A"), ((a, "A", NAT), "B"), ((a, "B", NAT), "B"), ((a, "B", NAT), "A")]
        if not all(n in BR for n in need):
            continue
        aa, ab, bb, ba = (BR[n] for n in need)
        gab_d = macro_draws(aa, "auroc", B6) - macro_draws(ab, "auroc", B6)
        gba_d = macro_draws(bb, "auroc", B6) - macro_draws(ba, "auroc", B6)
        gab = macro_point(aa, "auroc", B6) - macro_point(ab, "auroc", B6)
        gba = macro_point(bb, "auroc", B6) - macro_point(ba, "auroc", B6)
        GAP_DRAWS[a] = dict(ab=gab_d, ba=gba_d, asym=gab_d - gba_d)
        for q_, est, d in [
            ("gap NIH→CheXpert (A→B)", gab, gab_d),
            ("gap CheXpert→NIH (B→A)", gba, gba_d),
            ("asymmetry (A→B gap − B→A gap)", gab - gba, gab_d - gba_d),
        ]:
            HYP.append(dict(family="F1 transfer gaps", contrast=q_, arch=a, direction="A↔B", **summarize_draws(est, d)))
    for a1, a2 in itertools.combinations([a for a in ARCH_ALL if a in GAP_DRAWS], 2):
        d = GAP_DRAWS[a1]["asym"] - GAP_DRAWS[a2]["asym"]
        e1 = [x for x in HYP if x["arch"] == a1 and x["contrast"].startswith("asym")][0]["estimate"]
        e2 = [x for x in HYP if x["arch"] == a2 and x["contrast"].startswith("asym")][0]["estimate"]
        HYP.append(
            dict(
                family="F1b asymmetry differs between models",
                contrast=f"asymmetry {a1} − {a2}",
                arch=f"{a1} vs {a2}",
                direction="A↔B",
                **summarize_draws(e1 - e2, d),
            )
        )
    for a1, a2 in itertools.combinations(ARCHS, 2):
        for ts, T in EXT_DIRS + SEEN_DIRS:
            k1, k2 = ((a1, ts, NAT), T), ((a2, ts, NAT), T)
            if k1 in BR and k2 in BR:
                HYP.append(
                    dict(
                        family="F2 architecture differences",
                        contrast=f"{a2} − {a1}",
                        arch=f"{a2} vs {a1}",
                        direction=dname(ts, T),
                        seen=is_seen(ts, T),
                        **contrast(BR[k2], BR[k1]),
                    )
                )
    if CLASSICAL in ARCH_ALL:
        for ts, T in EXT_DIRS + SEEN_DIRS:
            for a in ARCHS:
                k1, k2 = ((CLASSICAL, ts, NAT), T), ((a, ts, NAT), T)
                if k1 in BR and k2 in BR:
                    HYP.append(
                        dict(
                            family="F2b deep vs classical baseline",
                            contrast=f"{a} − {CLASSICAL}",
                            arch=a,
                            direction=dname(ts, T),
                            seen=is_seen(ts, T),
                            **contrast(BR[k2], BR[k1]),
                        )
                    )
    for a in ARCH_ALL:
        for T, (s1, s2) in SOURCES_FOR.items():
            pk = ((a, POOL_FOR[T], NAT), T)
            for s_ in (s1, s2):
                sk = ((a, s_, NAT), T)
                if pk in BR and sk in BR:
                    HYP.append(
                        dict(
                            family="F3 pooled vs single source (unseen site)",
                            contrast=f"{POOL_FOR[T]} − {s_}",
                            arch=a,
                            direction=f"→{T}",
                            **contrast(BR[pk], BR[sk]),
                        )
                    )
        for ts in POOLED:
            for T in ts.split("+"):
                if T in "AB" and ((a, ts, NAT), T) in BR and ((a, T, NAT), T) in BR:
                    HYP.append(
                        dict(
                            family="F3b pooled vs single source (seen site)",
                            contrast=f"{ts} − {T}",
                            arch=a,
                            direction=f"→{T}",
                            **contrast(BR[((a, ts, NAT), T)], BR[((a, T, NAT), T)]),
                        )
                    )
    H1 = finalize_family(HYP)
    save_tab(
        H1[H1.family.str.startswith("F1")][
            ["family", "contrast", "arch", "estimate", "ci_low", "ci_high", "p", "p_holm", "verdict"]
        ],
        "table19_gaps_asymmetry",
        "Transfer gaps and asymmetry (paired two-level cluster bootstrap; Holm within family).",
        index=False,
    )
    save_tab(
        H1[H1.family.str.startswith("F3")][
            ["family", "contrast", "arch", "direction", "estimate", "ci_low", "ci_high", "p_holm", "verdict"]
        ],
        "table20_pooled_vs_single",
        "Does pooling a second hospital help? (unseen and seen sites).",
        index=False,
    )

    fig, ax = plt.subplots(1, 2, figsize=(COL2, 3.6), gridspec_kw=dict(width_ratios=[1, 1.5]))
    f1 = H1[H1.family == "F1 transfer gaps"].reset_index(drop=True)
    for i, r in f1.iterrows():
        ax[0].errorbar(
            r.estimate,
            i,
            xerr=[[r.estimate - r.ci_low], [r.ci_high - r.estimate]],
            fmt=ARCH_MK.get(r.arch, "o"),
            color=ARCH_COL.get(r.arch, "k"),
            mfc=ARCH_COL.get(r.arch, "k") if r.significant_holm else "w",
            ms=4,
            capsize=2,
            lw=0.8,
        )
    ax[0].set_yticks(range(len(f1)))
    ax[0].set_yticklabels([f"{r.contrast.split(' (')[0]} | {r.arch}" for r in f1.itertuples()], fontsize=5.5)
    ax[0].axvline(0, color="k", lw=0.5)
    ax[0].invert_yaxis()
    ax[0].set_xlabel("Δ macro AUROC (95% CI)")
    panel_label(ax[0], "a")
    f2 = H1[H1.family == "F2 architecture differences"].reset_index(drop=True)
    dirs = [dname(*d) for d in EXT_DIRS + SEEN_DIRS]
    pairs = list(dict.fromkeys(f2.contrast))
    off = {p_: (i - (len(pairs) - 1) / 2) * 0.24 for i, p_ in enumerate(pairs)}
    mk = ["o", "s", "^"]
    for r in f2.itertuples():
        y_ = dirs.index(r.direction) + off[r.contrast]
        i_ = pairs.index(r.contrast)
        ax[1].errorbar(
            r.estimate,
            y_,
            xerr=[[r.estimate - r.ci_low], [r.ci_high - r.estimate]],
            fmt=mk[i_ % 3],
            color="k",
            mfc="k" if r.significant_holm else "w",
            ms=3,
            capsize=1.2,
            lw=0.6,
            label=r.contrast if r.direction == f2.direction.iloc[0] else None,
        )
    ax[1].axvline(0, color="k", lw=0.5)
    ax[1].axhline(len(EXT_DIRS) - 0.5, color="0.6", lw=0.5)
    ax[1].set_yticks(range(len(dirs)))
    ax[1].set_yticklabels([d + (" (seen)" if i >= len(EXT_DIRS) else "") for i, d in enumerate(dirs)], fontsize=6)
    ax[1].invert_yaxis()
    ax[1].set_xlabel("Δ macro AUROC (95% CI); filled = Holm-significant")
    ax[1].legend(
        fontsize=5.5, loc="upper center", bbox_to_anchor=(0.5, -0.17), ncol=3, handletextpad=0.3, columnspacing=0.8
    )
    panel_label(ax[1], "b")
    fig.tight_layout()
    save_fig(
        fig,
        "fig09_gaps_and_architectures",
        "(a) Transfer gaps and asymmetry per model; (b) pairwise architecture differences per direction "
        "(paired two-level cluster bootstrap; filled markers = significant after Holm correction).",
    )

In [ ]:
# 6.3  F4: domain-generalisation methods vs ERM on the UNSEEN hospital, against both baselines:
#      ERM (natural) = the deployed default; ERM (balanced) = the fair control (every DG method samples hospitals 1:1).
with timed("6 contrasts: DG methods"):
    for base, fam in [(NAT, "F4a DG vs ERM (natural)"), (BAL, "F4b DG vs ERM (balanced)")]:
        for a in ARCHS:
            for ts in POOLED:
                T = UNSEEN[ts]
                kb = ((a, ts, base), T)
                if kb not in BR:
                    continue
                for m_ in METHODS:
                    km = ((a, ts, m_), T)
                    if m_ == base or (base == BAL and m_ == NAT) or km not in BR:
                        continue
                    HYP.append(
                        dict(
                            family=fam,
                            contrast=f"{m_} − {base}",
                            arch=a,
                            direction=dname(ts, T),
                            method=m_,
                            **contrast(BR[km], BR[kb]),
                        )
                    )
    H1 = finalize_family(HYP)
    DGS = H1[H1.family.str.startswith("F4")]
    save_tab(
        DGS[
            [
                "family",
                "arch",
                "direction",
                "contrast",
                "estimate",
                "ci_low",
                "ci_high",
                "p",
                "p_holm",
                "q_bh",
                "prob_superiority",
                "verdict",
            ]
        ],
        "table21_dg_vs_erm",
        f"DG methods vs ERM on the unseen hospital (paired two-level cluster bootstrap, Holm within family; "
        f"equivalence margin ±{CFG.equivalence_margin}).",
        index=False,
    )
    display(DGS.groupby(["family", "verdict"]).size().unstack(fill_value=0))

    QW = []  # GroupDRO group weights at the end of training
    if HAS_TORCH:
        for a, folder, pat in [
            ("ViT-S", P["vit"], "vit_{t}_seed{s}_gdro_last.pt"),
            ("ConvNeXt-T", P["convnext_dg"], "cnx_{t}_seed{s}_gdro_last.pt"),
            ("DenseNet121", P["dg"], f"gdro_e{ARCHIVE_EPOCHS}_seed{{s}}_last.pt"),
        ]:
            for ts in (POOLED if a != "DenseNet121" else ["A+B"]):
                for s in SEEDS:
                    f = folder / pat.format(t=tsp(ts), s=s)
                    if f.exists():
                        q = torch.load(f, map_location="cpu", weights_only=False).get("q")
                        if q is not None:
                            QW.append(
                                dict(
                                    arch=a,
                                    train=ts,
                                    seed=s,
                                    hardness="raw group loss",
                                    **{
                                        f"q_{k}": round(float(v), 3)
                                        for k, v in zip(ts.split("+"), np.asarray(q).ravel())
                                    },
                                )
                            )
    for r in GDRO_DN_RUNS.itertuples():  # DenseNet121 A+C / B+C (section 4.5b): weights from the training history
        qs = {
            c: round(float(getattr(r, c)), 3)
            for c in GDRO_DN_RUNS.columns
            if c.startswith("q_") and pd.notna(getattr(r, c))
        }
        if qs:
            QW.append(
                dict(arch="DenseNet121", train=r.train, seed=r.seed, hardness="loss / prevalence-only loss", **qs)
            )
    if QW:
        QW = save_tab(
            pd.DataFrame(QW),
            "table22_groupdro_weights",
            "GroupDRO hospital weights q at the end of training (q ≈ 1 for one hospital = collapse); hardness = the "
            "quantity driving the weight update.",
            index=False,
        )
    fig, axes = plt.subplots(1, 2, figsize=(COL2, 3.3), sharey=True)
    for ax, fam in zip(axes, ["F4a DG vs ERM (natural)", "F4b DG vs ERM (balanced)"]):
        g = DGS[DGS.family == fam].reset_index(drop=True)
        g["row"] = g.direction + " | " + g.method
        rows_ = [f"{dname(ts, UNSEEN[ts])} | {m_}" for ts in POOLED for m_ in METHODS[1:]]
        yy = {r_: i for i, r_ in enumerate(rows_)}
        for r in g.itertuples():
            off_ = (ARCHS.index(r.arch) - 1) * 0.25
            ax.errorbar(
                r.estimate,
                yy[r.row] + off_,
                xerr=[[r.estimate - r.ci_low], [r.ci_high - r.estimate]],
                fmt=ARCH_MK[r.arch],
                color=ARCH_COL[r.arch],
                mfc=ARCH_COL[r.arch] if r.significant_holm else "w",
                ms=3.5,
                capsize=1.2,
                lw=0.7,
                label=r.arch if r.row == rows_[0] or (r.row == g.row.iloc[0]) else None,
            )
        ax.axvspan(-CFG.equivalence_margin, CFG.equivalence_margin, color="0.9", zorder=0)
        ax.axvline(0, color="k", lw=0.5)
        ax.set_yticks(range(len(rows_)))
        ax.set_yticklabels(rows_, fontsize=5.5)
        ax.invert_yaxis()
        ax.set_xlabel(f"Δ unseen-site AUROC vs {fam.split('vs ')[1]}")
        ax.set_title(fam.split(" ", 1)[1], fontsize=7)
    h_, l_ = axes[0].get_legend_handles_labels()
    uniq = dict(zip(l_, h_))
    fig.legend(uniq.values(), uniq.keys(), fontsize=6, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=len(uniq))
    fig.tight_layout()
    save_fig(
        fig,
        "fig10_dg_vs_erm",
        "DG methods vs ERM on the unseen hospital (95% CI; filled = Holm-significant; "
        "grey band = ± equivalence margin).",
    )

In [ ]:
# 6.4  Ensembles. F5 = the PRE-REGISTERED Stage-2 test (plan locked before analysis, hash ff8237ecf578bde4): does
#      averaging one model per source hospital (cross-source ensemble) beat pooled training, two pooled models (equal
#      inference cost) and same-hospital ensembles on the unseen hospital? Plus the diversity mechanism (H1, H2),
#      cross-architecture ensembles (F6) and seed ensembling (F7).
def logit_ensemble(members):
    """members: list of (probabilities (N, L), trained-on mask (L,)) -> probabilities of the logit average over members
    that know a finding."""
    Z = np.stack([logit(p) for p, _ in members])
    M = np.stack([np.broadcast_to(v, p.shape) for p, v in members]).astype(float)
    return sigmoid((Z * M).sum(0) / np.maximum(M.sum(0), 1)).astype(np.float32)


def _p(a, ts, s, T, m_=NAT):
    k = (a, ts, m_, s % len(SEEDS))
    return (PS[k].test[T], source_union_valid(ts)) if k in PS and T in PS[k].test else None


STAGE2 = dict(delta=0.01, alpha=0.05, locked_archs=["DenseNet121", "ConvNeXt-T"])
with timed("6 ensembles"):
    ENS = {}
    for a in ARCHS:
        for T, (s1, s2) in SOURCES_FOR.items():
            pool = POOL_FOR[T]
            fams = {}
            for s in SEEDS:
                parts = {
                    "cross": [_p(a, s1, s, T), _p(a, s2, s, T)],
                    "pooled x1": [_p(a, pool, s, T)],
                    "pooled x2": [_p(a, pool, s, T), _p(a, pool, s + 1, T)],
                    f"same {s1}+{s1}": [_p(a, s1, s, T), _p(a, s1, s + 1, T)],
                    f"same {s2}+{s2}": [_p(a, s2, s, T), _p(a, s2, s + 1, T)],
                }
                for nm, mem in parts.items():
                    if all(x is not None for x in mem):
                        fams.setdefault(nm, {})[s] = logit_ensemble(mem)
            for nm, probs in fams.items():
                if len(probs) == len(SEEDS):
                    allowed = (
                        model_findings(pool, T) if nm != f"same {s2}+{s2}" or s2 != "C" else model_findings("C", T)
                    )
                    ENS[(a, T, nm)] = bootstrap_cell(("ENS", a, T, nm), T, probs, metrics=("auroc",), allowed=allowed)
    rows = []
    for a in ARCHS:
        for T, (s1, s2) in SOURCES_FOR.items():
            if (a, T, "cross") not in ENS:
                continue
            for comp in ["pooled x1", "pooled x2", f"same {s1}+{s1}", f"same {s2}+{s2}"]:
                if (a, T, comp) in ENS:
                    # pre-registered endpoint: 5 findings
                    rows.append(
                        dict(
                            family="F5 Stage-2 cross-source ensemble",
                            contrast=f"cross-source − {comp}",
                            arch=a,
                            direction=f"{s1}+{s2} models → {T}",
                            comparator=comp,
                            target=T,
                            **contrast(ENS[(a, T, "cross")], ENS[(a, T, comp)], findings=IDX5),
                        )
                    )
    F5 = pd.DataFrame(rows)
    if len(F5):
        F5["p_holm"] = holm(F5.p.values)
        F5["p_equiv_holm"] = holm(F5.p_equiv.values)
        F5["verdict"] = [decide(e, p_, q_) for e, p_, q_ in zip(F5.estimate, F5.p_holm, F5.p_equiv_holm)]
        F5["beats"] = (F5.estimate > 0) & (F5.p_holm < STAGE2["alpha"])
        F5["loses"] = (F5.estimate < 0) & (F5.p_holm < STAGE2["alpha"])
        cells = []
        for (a, T), g in F5.groupby(["arch", "target"]):
            dp = (
                float(g[g.comparator == "pooled x1"].estimate.iloc[0])
                if (g.comparator == "pooled x1").any()
                else np.nan
            )
            cells.append(
                dict(
                    arch=a,
                    target=T,
                    locked=a in STAGE2["locked_archs"],
                    comparators_beaten=int(g.beats.sum()),
                    diff_vs_pooled=dp,
                    passes=bool(g.beats.all() and len(g) == 4 and dp >= STAGE2["delta"]),
                    pooled_wins=bool(g[g.comparator == "pooled x1"].loses.any()),
                )
            )
        CELLS = pd.DataFrame(cells)
        lk = CELLS[CELLS.locked]
        n_pass = int(lk.passes.sum())
        verdict = (
            "REVERSED"
            if lk.pooled_wins.sum() >= 4
            else (
                "PASS"
                if (
                    n_pass >= 4
                    and lk[lk.passes].arch.nunique() == len(STAGE2["locked_archs"])
                    and lk[lk.passes].target.nunique() >= 2
                )
                else "PARTIAL" if n_pass >= 1 else "FAIL"
            )
        )
        STAGE2_VERDICT = verdict
        save_tab(
            F5[["arch", "direction", "contrast", "estimate", "ci_low", "ci_high", "p", "p_holm", "verdict"]],
            "table23_stage2_ensemble_contrasts",
            "Pre-registered Stage-2 contrasts (Holm over all primary contrasts).",
            index=False,
        )
        save_tab(
            CELLS,
            "table24_stage2_cells",
            f"Stage-2 decision cells (locked architectures: {', '.join(STAGE2['locked_archs'])}; ViT-S = replication). "
            f"Verdict of the pre-registered ladder: {verdict}.",
            index=False,
        )
        print("Stage-2 verdict (pre-registered ladder PASS / PARTIAL / FAIL / REVERSED):", verdict)
        HYP.extend(F5.drop(columns=["p_holm", "p_equiv_holm", "beats", "loses"]).to_dict("records"))

    # mechanism H1/H2: are cross-hospital member pairs more diverse, and does the ensemble gain grow with diversity?
    mrows = []
    for a in ARCHS:
        for T, (s1, s2) in SOURCES_FOR.items():
            for kind, (ta, tb, off_) in {
                "cross-hospital": (s1, s2, 0),
                f"same hospital ({s1})": (s1, s1, 1),
                f"same hospital ({s2})": (s2, s2, 1),
            }.items():
                for s in SEEDS:
                    pa_, pb_ = _p(a, ta, s, T), _p(a, tb, s + off_, T)
                    if pa_ is None or pb_ is None:
                        continue
                    e = logit_ensemble([pa_, pb_])
                    for j in IDX5:
                        if j not in model_findings(ta, T) or j not in model_findings(tb, T):
                            continue
                        y = YT[T][:, j]
                        za, zb = logit(pa_[0][:, j]), logit(pb_[0][:, j])
                        mrows.append(
                            dict(
                                arch=a,
                                target=T,
                                pair=kind,
                                cross=kind == "cross-hospital",
                                seed=s,
                                finding=LABELS[j],
                                diversity=1 - np.corrcoef(za, zb)[0, 1],
                                gain=auroc(y, e[:, j]) - (auroc(y, pa_[0][:, j]) + auroc(y, pb_[0][:, j])) / 2,
                            )
                        )
    MECH = pd.DataFrame(mrows)
    if len(MECH):
        h1 = MECH.groupby(["arch", "target", "cross"]).diversity.mean().unstack("cross")
        h1["difference"] = h1[True] - h1[False]
        h1_test = paired_effects(h1[True].values, h1[False].values)
        agg = MECH.groupby(["arch", "target", "pair", "finding"])[["diversity", "gain"]].mean().reset_index()
        rho, p_rho = stats.spearmanr(agg.diversity, agg.gain)
        rb = []
        for b_ in range(1000):
            g_ = agg.iloc[rng_for("H2", b_).integers(0, len(agg), len(agg))]
            rb.append(stats.spearmanr(g_.diversity, g_.gain)[0])
        H2 = dict(
            spearman_rho=rho, ci_low=np.nanpercentile(rb, 2.5), ci_high=np.nanpercentile(rb, 97.5), n_points=len(agg)
        )
        save_tab(
            h1.rename(columns={True: "cross-hospital pairs", False: "same-hospital pairs"}).round(4),
            "table25_ensemble_diversity",
            f"Member diversity (1 − logit correlation). Cross > same in {int((h1.difference > 0).sum())}/{len(h1)} "
            "cells; paired Wilcoxon p = "
            f"{h1_test['p_wilcoxon']:.3g}, rank-biserial r = {h1_test['rank_biserial']:.2f}. "
            f"H2: Spearman(diversity, gain) = {rho:.2f} "
            f"[{H2['ci_low']:.2f}, {H2['ci_high']:.2f}] (points share models: descriptive).",
        )

    # F6 cross-architecture ensembles and F7 seed ensembling (ERM natural)
    XA = {}
    combos = {"D+C+V": ARCHS, "D+C": ARCHS[:2], "D+V": [ARCHS[0], ARCHS[2]], "C+V": ARCHS[1:]}
    for ts, T in EXT_DIRS:
        for cname, members in combos.items():
            probs = {
                s: logit_ensemble([_p(a, ts, s, T) for a in members])
                for s in SEEDS
                if all(_p(a, ts, s, T) is not None for a in members)
            }
            if len(probs) == len(SEEDS):
                XA[(ts, T, cname)] = bootstrap_cell(
                    ("XARCH", ts, T, cname), T, probs, metrics=("auroc",), allowed=model_findings(ts, T)
                )
        for a in ARCHS:
            k = ((a, ts, NAT), T)
            if (ts, T, "D+C+V") in XA and k in BR:
                HYP.append(
                    dict(
                        family="F6 3-architecture ensemble vs single architecture",
                        contrast=f"D+C+V − {a}",
                        arch=a,
                        direction=dname(ts, T),
                        **contrast(XA[(ts, T, "D+C+V")], BR[k]),
                    )
                )
            if k in BR and len(SEEDS_OF[(a, ts, NAT)]) == len(SEEDS):
                se = {0: logit_ensemble([_p(a, ts, s, T) for s in SEEDS])}
                r_se = bootstrap_cell(("SEEDENS", a, ts, T), T, se, metrics=("auroc",), allowed=model_findings(ts, T))
                HYP.append(
                    dict(
                        family="F7 seed ensemble vs single model",
                        contrast="3-seed ensemble − single seed",
                        arch=a,
                        direction=dname(ts, T),
                        **contrast(r_se, BR[k]),
                    )
                )
    H1 = finalize_family(HYP)
    save_tab(
        H1[H1.family.str.startswith(("F6", "F7"))]
        .groupby(["family", "arch"])[["estimate"]]
        .agg(["mean", "min", "max"])
        .round(4),
        "table26_ensemble_gains",
        "Gain of 3-architecture and 3-seed ensembles over single models (unseen-site directions).",
    )
    if len(F5):
        fig, ax = plt.subplots(1, 2, figsize=(COL2, 4.6), gridspec_kw=dict(width_ratios=[1.4, 1]))
        g = F5.reset_index(drop=True)
        lab = [f"{r.arch[:8]} → {r.target} | {r.comparator}" for r in g.itertuples()]
        for i, r in g.iterrows():
            c_ = "C3" if r.beats else ("C0" if r.loses else "0.5")
            ax[0].errorbar(
                r.estimate,
                i,
                xerr=[[r.estimate - r.ci_low], [r.ci_high - r.estimate]],
                fmt="o",
                color=c_,
                ms=3,
                capsize=1.2,
                lw=0.7,
            )
        ax[0].axvline(0, color="k", lw=0.5)
        ax[0].axvline(STAGE2["delta"], color="C1", ls=":", lw=0.8)
        ax[0].set_yticks(range(len(g)))
        ax[0].set_yticklabels(lab, fontsize=4.8)
        ax[0].invert_yaxis()
        ax[0].set_xlabel("Cross-source ensemble − comparator (AUROC)")
        ax[0].set_title(f"Stage-2 verdict: {STAGE2_VERDICT} (red = beats, blue = loses, Holm)", fontsize=7)
        panel_label(ax[0], "a")
        if len(MECH):
            for (cross_, a), gg in agg.assign(cross=agg.pair == "cross-hospital").groupby(["cross", "arch"]):
                ax[1].scatter(
                    gg.diversity,
                    gg.gain,
                    s=9,
                    marker=ARCH_MK[a],
                    color="C3" if cross_ else "0.55",
                    alpha=0.8,
                    label=f"{a}, {'cross' if cross_ else 'same'}-hospital",
                )
            ax[1].axhline(0, color="k", lw=0.5)
            ax[1].set(xlabel="Member diversity (1 − logit correlation)", ylabel="Ensemble gain (AUROC)")
            ax[1].set_title(
                f"H2: Spearman ρ = {H2['spearman_rho']:.2f} [{H2['ci_low']:.2f}, {H2['ci_high']:.2f}]", fontsize=7
            )
            ax[1].legend(fontsize=5, loc="upper left", bbox_to_anchor=(0.0, -0.16), ncol=2)
            panel_label(ax[1], "b")
        fig.tight_layout()
        save_fig(
            fig,
            "fig11_ensembles",
            "(a) Pre-registered Stage-2 contrasts per architecture and target; "
            "(b) ensemble gain against member diversity.",
        )

In [ ]:
# 6.5  Training-set SIZE and COMPOSITION controls (DenseNet121; che44-nih77 runs)
#   F8  size control: is NIH's larger external gap explained by NIH having fewer training images (77k vs 152k)? Eight
#       random 77,353-image CheXpert subsets (patient-level) are compared with the full CheXpert model and the NIH
#       model. original asymmetry = size effect + size-controlled asymmetry (identity on the bootstrap draws).
#   F9  composition: fixed 77,353-image budget, 0/25/50/75/100% CheXpert; contrasts and a linear trend per +25 points.
CTRL = load_controls()
SIZE, COMP = None, None
with timed("6 size and composition"):
    cx77 = {s: d for (k, s), d in CTRL.items() if k == "CX77"}
    if len(cx77) >= 2 and all(((("DenseNet121", t, NAT), T) in BR) for t in "AB" for T in "AB"):
        R77 = {T: bootstrap_cell(("CX77",), T, {s: d[T] for s, d in cx77.items()}, metrics=("auroc",)) for T in "AB"}

        def _gap(rin, rout):
            return macro_point(rin, "auroc", B6) - macro_point(rout, "auroc", B6), macro_draws(
                rin, "auroc", B6
            ) - macro_draws(rout, "auroc", B6)

        gN = _gap(BR[(("DenseNet121", "A", NAT), "A")], BR[(("DenseNet121", "A", NAT), "B")])
        gC = _gap(BR[(("DenseNet121", "B", NAT), "B")], BR[(("DenseNet121", "B", NAT), "A")])
        g7 = _gap(R77["B"], R77["A"])
        q = {
            "gap NIH-77k": gN,
            "gap CheXpert-152k": gC,
            "gap CheXpert-77k (8 subsets)": g7,
            "original asymmetry (NIH-77k − CheXpert-152k)": (gN[0] - gC[0], gN[1] - gC[1]),
            "size effect (CheXpert-77k − CheXpert-152k)": (g7[0] - gC[0], g7[1] - gC[1]),
            "size-controlled asymmetry (NIH-77k − CheXpert-77k)": (gN[0] - g7[0], gN[1] - g7[1]),
        }
        # Holm over the six size quantities
        SIZE = finalize_family(
            [
                dict(
                    family="F8 training-set size control",
                    contrast=k_,
                    arch="DenseNet121",
                    direction="A↔B",
                    **summarize_draws(e, d),
                )
                for k_, (e, d) in q.items()
            ]
        )
        share = (
            q["size effect (CheXpert-77k − CheXpert-152k)"][1] / q["original asymmetry (NIH-77k − CheXpert-152k)"][1]
        )
        ok_ = np.abs(q["original asymmetry (NIH-77k − CheXpert-152k)"][1]) > 0.01
        SIZE_SHARE = (
            q["size effect (CheXpert-77k − CheXpert-152k)"][0] / q["original asymmetry (NIH-77k − CheXpert-152k)"][0],
            *np.percentile(share[ok_], [2.5, 97.5]),
        )
        HYP.extend(SIZE[SIZE.contrast.str.contains("asymmetry|size effect")].to_dict("records"))
        save_tab(
            SIZE,
            "table27_size_control",
            f"Training-set size control ({len(cx77)} random CheXpert subsets; subsets resampled inside the bootstrap). "
            "Share of the asymmetry explained by size: "
            f"{SIZE_SHARE[0]:.2f} [{SIZE_SHARE[1]:.2f}, {SIZE_SHARE[2]:.2f}].",
            index=False,
        )
    else:
        print("size-control runs not found in", P["controls"])
    FR = {0: None, 25: "MIX25", 50: "MIX50", 75: "MIX75", 100: "CX77"}
    have = all(any(k == v for (k, s) in CTRL) for v in FR.values() if v) and ("DenseNet121", "A", NAT) in [
        k for k in KEYS
    ]
    if have:
        CR = {}
        for f_, nm in FR.items():
            for T in "ABC":
                probs = (
                    {s: PS[("DenseNet121", "A", NAT, s)].test[T] for s in SEEDS_OF[("DenseNet121", "A", NAT)]}
                    if nm is None
                    else {s: d[T] for (k, s), d in CTRL.items() if k == nm and s in SEEDS and T in d}
                )
                if len(probs) >= 2:
                    CR[(f_, T)] = bootstrap_cell(
                        ("COMP", f_), T, probs, metrics=("auroc",), allowed=IDX5 if T == "C" else None
                    )
        rows = []
        for T in "ABC":
            for f_ in FR:
                if (f_, T) in CR:
                    d = macro_draws(CR[(f_, T)])
                    rows.append(
                        dict(
                            test=T,
                            pct_chexpert=f_,
                            auroc=macro_point(CR[(f_, T)]),
                            ci_low=np.percentile(d, 2.5),
                            ci_high=np.percentile(d, 97.5),
                        )
                    )
        COMP = pd.DataFrame(rows)
        comp_h = []
        for T, ref in [("A", 0), ("B", 100), ("C", 100)]:
            for f_ in FR:
                if f_ != ref and (f_, T) in CR and (ref, T) in CR:
                    comp_h.append(
                        dict(
                            family=f"F9 composition, test {SHORT[T]}",
                            contrast=f"{f_}% − {ref}% CheXpert",
                            arch="DenseNet121",
                            direction=f"→{T}",
                            **contrast(CR[(f_, T)], CR[(ref, T)]),
                        )
                    )
        x = np.array(list(FR))
        xc = (x - x.mean()) / 100
        for T in "ABC":
            if all((f_, T) in CR for f_ in FR):
                D = np.stack([macro_draws(CR[(f_, T)]) for f_ in FR], 1)
                pt = np.array([macro_point(CR[(f_, T)]) for f_ in FR])
                comp_h.append(
                    dict(
                        family="F9 composition trend",
                        contrast="AUROC change per +25 points CheXpert",
                        arch="DenseNet121",
                        direction=f"→{T}",
                        **summarize_draws(
                            float((pt * xc).sum() / (xc**2).sum() * 0.25), (D * xc).sum(1) / (xc**2).sum() * 0.25
                        ),
                    )
                )
        HYP.extend(comp_h)
        save_tab(
            finalize_family(comp_h)[
                ["family", "contrast", "direction", "estimate", "ci_low", "ci_high", "p", "p_holm", "verdict"]
            ],
            "table28_composition",
            "Source composition at a fixed budget of 77,353 images (DenseNet121); Holm within each family.",
            index=False,
        )
        fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.8))
        for T in "ABC":
            g = COMP[COMP.test == T]
            ax[0].plot(
                g.pct_chexpert,
                g.auroc,
                marker="o",
                color=SITE_COL[T],
                label=f"test {SHORT[T]}" + (" (unseen)" if T == "C" else ""),
            )
            ax[0].fill_between(g.pct_chexpert, g.ci_low, g.ci_high, color=SITE_COL[T], alpha=0.2, lw=0)
        ax[0].set(xlabel="% of training images from CheXpert (fixed budget)", ylabel="Macro AUROC (95% CI)")
        ax[0].legend()
        panel_label(ax[0], "a")
        if SIZE is not None:
            s_ = SIZE.reset_index(drop=True)
            ax[1].barh(
                range(len(s_)),
                s_.estimate,
                xerr=[s_.estimate - s_.ci_low, s_.ci_high - s_.estimate],
                color=["0.3"] * 3 + ["C3", "0.6", "C0"],
                error_kw=dict(lw=0.6, capsize=2),
            )
            ax[1].set_yticks(range(len(s_)))
            ax[1].set_yticklabels(s_.contrast, fontsize=5.5)
            ax[1].invert_yaxis()
            ax[1].axvline(0, color="k", lw=0.5)
            ax[1].set_xlabel("Δ AUROC (95% CI)")
            panel_label(ax[1], "b")
        fig.tight_layout()
        save_fig(
            fig,
            "fig12_size_composition",
            "(a) Source composition at a fixed training budget; (b) decomposition "
            "of the asymmetry into a size effect and a size-controlled part.",
        )
    else:
        print("composition runs not found in", P["controls"])
H1 = finalize_family(HYP)

In [ ]:
# 6.6  Case mix and subgroups.
#   F10 view-matched transfer gaps: the asymmetry recomputed inside PA-only and AP-only images (findings with
#       >= 30 positives and negatives in that view at both sites). If it survives in both views, the PA/AP mix is
#       not the explanation.
#   F11 subgroup robustness (sex, age band, view) for the four single-hospital directions;
#       contrasts = subgroup AUROC gaps.
def _masks(T):
    m = META[T]
    out = {"PA view": (m["view"] == "PA").values, "AP view": (m["view"] == "AP").values}
    if m["sex"].notna().any():
        out.update({"male": (m["sex"] == "Male").values, "female": (m["sex"] == "Female").values})
    if m["age"].notna().any():
        out.update(
            {
                "age < 50": (m["age"] < 50).values,
                "age 50-65": m["age"].between(50, 65).values,
                "age > 65": (m["age"] > 65).values,
            }
        )
    return out


SUBMASK = {T: _masks(T) for T in "AB"}
with timed("6 view and subgroups"):
    VB_ = {}
    for a in ARCH_ALL:
        for src, T in [("A", "A"), ("A", "B"), ("B", "B"), ("B", "A")]:
            k = (a, src, NAT)
            if k not in KEYS:
                continue
            probs = {s: PS[(*k, s)].test[T] for s in SEEDS_OF[k]}
            for g, mk in SUBMASK[T].items():
                VB_[(a, src, T, g)] = bootstrap_cell(
                    ("SUB", a, src, T, g),
                    T,
                    probs,
                    metrics=("auroc",),
                    mask=mk,
                    min_events=30,
                    allowed=model_findings(src, T),
                )
    rows = []
    for a in ARCH_ALL:
        for v in ["PA view", "AP view"]:
            ks = [(a, "A", "A", v), (a, "A", "B", v), (a, "B", "B", v), (a, "B", "A", v)]
            if not all(k in VB_ for k in ks):
                continue
            f = sorted(set.intersection(*[set(VB_[k].findings) for k in ks]))
            d = [macro_draws(VB_[k], "auroc", f) for k in ks]
            p = [macro_point(VB_[k], "auroc", f) for k in ks]
            for nm, est, dd in [
                ("gap A→B", p[0] - p[1], d[0] - d[1]),
                ("gap B→A", p[2] - p[3], d[2] - d[3]),
                ("asymmetry", (p[0] - p[1]) - (p[2] - p[3]), (d[0] - d[1]) - (d[2] - d[3])),
            ]:
                rows.append(
                    dict(
                        family="F10 view-matched gaps",
                        contrast=f"{nm} ({v})",
                        arch=a,
                        direction="A↔B",
                        view=v,
                        n_findings=len(f),
                        **summarize_draws(est, dd),
                    )
                )
    VIEWG = pd.DataFrame(rows)
    HYP.extend(rows)
    srows = []
    for (a, src, T, g), r in VB_.items():
        d = macro_draws(r)
        srows.append(
            dict(
                arch=a,
                direction=dname(src, T),
                subgroup=g,
                n_images=int(SUBMASK[T][g].sum()),
                n_findings=len(r.findings),
                auroc=macro_point(r),
                ci_low=np.nanpercentile(d, 2.5),
                ci_high=np.nanpercentile(d, 97.5),
            )
        )
    SUBG = pd.DataFrame(srows)
    for a in ARCH_ALL:
        for src, T in [("A", "A"), ("A", "B"), ("B", "B"), ("B", "A")]:
            for g1, g2 in [("female", "male"), ("AP view", "PA view"), ("age > 65", "age < 50")]:
                k1, k2 = (a, src, T, g1), (a, src, T, g2)
                if k1 in VB_ and k2 in VB_:
                    HYP.append(
                        dict(
                            family="F11 subgroup gaps",
                            contrast=f"{g1} − {g2}",
                            arch=a,
                            direction=dname(src, T),
                            **contrast(VB_[k1], VB_[k2]),
                        )
                    )
    H1 = finalize_family(HYP)
    save_tab(
        H1[H1.family == "F10 view-matched gaps"][
            ["contrast", "arch", "n_findings", "estimate", "ci_low", "ci_high", "p_holm", "verdict"]
        ],
        "table29_view_matched_gaps",
        "Transfer gaps and asymmetry within PA-only and AP-only images.",
        index=False,
    )
    save_tab(
        H1[H1.family == "F11 subgroup gaps"][
            ["arch", "direction", "contrast", "estimate", "ci_low", "ci_high", "p_holm", "q_bh", "verdict"]
        ],
        "table30_subgroup_gaps",
        "Subgroup AUROC gaps (fairness audit; Holm within family, BH across the study).",
        index=False,
    )
    fig, ax = plt.subplots(1, 2, figsize=(COL2, 3.4), gridspec_kw=dict(width_ratios=[1, 1.3]))
    vg = VIEWG[VIEWG.contrast.str.startswith("asym")]
    allv = H1[(H1.family == "F1 transfer gaps") & H1.contrast.str.startswith("asym")]
    labs, i = [], 0
    for a in ARCH_ALL:
        for nm, sub in [
            ("all images", allv[allv.arch == a]),
            ("PA only", vg[(vg.arch == a) & (vg.view == "PA view")]),
            ("AP only", vg[(vg.arch == a) & (vg.view == "AP view")]),
        ]:
            if len(sub):
                r = sub.iloc[0]
                ax[0].errorbar(
                    r.estimate,
                    i,
                    xerr=[[r.estimate - r.ci_low], [r.ci_high - r.estimate]],
                    fmt=ARCH_MK[a],
                    color=ARCH_COL[a],
                    ms=4,
                    capsize=2,
                    lw=0.8,
                )
                labs.append(f"{a} | {nm}")
                i += 1
    ax[0].axvline(0, color="k", lw=0.5)
    ax[0].set_yticks(range(len(labs)))
    ax[0].set_yticklabels(labs, fontsize=5.5)
    ax[0].invert_yaxis()
    ax[0].set_xlabel("Asymmetry (A→B gap − B→A gap), 95% CI")
    panel_label(ax[0], "a")
    sg = SUBG[SUBG.arch == ARCHS[0]].copy()
    order_ = list(dict.fromkeys(sg.subgroup))
    dirs_ = list(dict.fromkeys(sg.direction))
    for k_, dd in enumerate(dirs_):
        g = sg[sg.direction == dd].set_index("subgroup").reindex(order_)
        ax[1].errorbar(
            np.arange(len(order_)) + (k_ - 1.5) * 0.18,
            g.auroc,
            yerr=[g.auroc - g.ci_low, g.ci_high - g.auroc],
            fmt="o",
            ms=3,
            capsize=1.5,
            lw=0.7,
            label=dd,
        )
    ax[1].set_xticks(range(len(order_)))
    ax[1].set_xticklabels(order_, rotation=30, ha="right")
    ax[1].set_ylabel(f"Macro AUROC ({ARCHS[0]}, 95% CI)")
    ax[1].legend(fontsize=6, ncol=2)
    panel_label(ax[1], "b")
    fig.tight_layout()
    save_fig(
        fig,
        "fig13_case_mix_subgroups",
        "(a) Asymmetry on all images and within PA-only / AP-only images; (b) "
        "subgroup AUROC for the four single-hospital directions.",
    )

In [ ]:
# 6.7  Analytic cross-check: DeLong (independent images) vs Obuchowski (patient-clustered) vs cluster bootstrap, per
#      finding, for every architecture contrast on an unseen site (seed-averaged predictions, one finding at a time).
#      Ignoring that NIH/CheXpert patients contribute several images makes image-level DeLong intervals too narrow.
with timed("6 DeLong cross-check"):
    rows = []
    for a1, a2 in itertools.combinations(ARCHS, 2):
        for ts, T in EXT_DIRS:
            k1, k2 = (a1, ts, NAT), (a2, ts, NAT)
            if k1 not in KEYS or k2 not in KEYS:
                continue
            p1 = np.mean([PS[(*k1, s)].test[T] for s in SEEDS_OF[k1]], 0)
            p2 = np.mean([PS[(*k2, s)].test[T] for s in SEEDS_OF[k2]], 0)
            fs = sorted(set(model_findings(ts, T)))
            r1 = bootstrap_cell(("SEEDAVG",) + k1, T, {0: p1}, metrics=("auroc",), allowed=fs)
            r2 = bootstrap_cell(("SEEDAVG",) + k2, T, {0: p2}, metrics=("auroc",), allowed=fs)
            for j in sorted(set(r1.findings) & set(r2.findings)):
                y = YT[T][:, j]
                dl = delong_test(y, p2[:, j], p1[:, j])
                ob = delong_test(y, p2[:, j], p1[:, j], clusters=None if T == "C" else GID[T])
                db = r2.boot["auroc"][:, 0, j] - r1.boot["auroc"][:, 0, j]
                sb = summarize_draws(float(r2.point["auroc"][0, j] - r1.point["auroc"][0, j]), db)
                rows.append(
                    dict(
                        contrast=f"{a2} − {a1}",
                        direction=dname(ts, T),
                        finding=LABELS[j],
                        diff=dl["diff"],
                        se_delong=dl["se"],
                        se_obuchowski=ob["se"],
                        se_bootstrap=sb["se"],
                        p_delong=dl["p"],
                        p_obuchowski=ob["p"],
                        p_bootstrap=sb["p"],
                        clustered=T != "C",
                    )
                )
    DL = pd.DataFrame(rows)
    if len(DL):
        DL["se_ratio_naive_vs_cluster"] = DL.se_delong / DL.se_bootstrap
        DL["se_ratio_obuchowski_vs_cluster"] = DL.se_obuchowski / DL.se_bootstrap
        DL.to_csv(TAB / "table31_delong_vs_bootstrap.csv", index=False)
        agree = (
            DL.groupby("clustered")[["se_ratio_naive_vs_cluster", "se_ratio_obuchowski_vs_cluster"]].median().round(3)
        )
        agree["spearman(p_obuchowski, p_bootstrap)"] = (
            DL.groupby("clustered")[["p_obuchowski", "p_bootstrap"]]
            .apply(lambda g: stats.spearmanr(g.p_obuchowski, g.p_bootstrap)[0])
            .round(3)
        )
        agree["decisions agree at alpha (Obuchowski vs bootstrap)"] = (
            DL.groupby("clustered")[["p_obuchowski", "p_bootstrap"]]
            .apply(lambda g: ((g.p_obuchowski < CFG.alpha) == (g.p_bootstrap < CFG.alpha)).mean())
            .round(3)
        )
        save_tab(
            agree,
            "table31b_delong_vs_bootstrap_summary",
            "Median SE ratios of analytic tests to the cluster bootstrap (per finding). Clustered = NIH/CheXpert "
            "(several images per patient); VinBigData images are independent.",
        )
        fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.7))
        for cl_, mk_ in [(True, "o"), (False, "s")]:
            g = DL[DL.clustered == cl_]
            ax[0].scatter(
                -np.log10(g.p_bootstrap.clip(1e-12)),
                -np.log10(g.p_obuchowski.clip(1e-12)),
                s=8,
                marker=mk_,
                alpha=0.7,
                label="NIH/CheXpert (clustered)" if cl_ else "VinBigData",
            )
        lim = [0, max(4.0, ax[0].get_xlim()[1], ax[0].get_ylim()[1])]
        ax[0].plot(lim, lim, "k:", lw=0.6)
        ax[0].set(xlabel="−log10 p, cluster bootstrap", ylabel="−log10 p, DeLong/Obuchowski")
        ax[0].legend(fontsize=6)
        panel_label(ax[0], "a")
        sns.histplot(
            DL[DL.clustered],
            x="se_ratio_naive_vs_cluster",
            ax=ax[1],
            color="C3",
            label="image-level DeLong",
            element="step",
            stat="density",
            bins=25,
        )
        sns.histplot(
            DL[DL.clustered],
            x="se_ratio_obuchowski_vs_cluster",
            ax=ax[1],
            color="C0",
            label="Obuchowski (patient clusters)",
            element="step",
            stat="density",
            bins=25,
        )
        ax[1].axvline(1, color="k", lw=0.5)
        ax[1].set(xlabel="SE ratio: analytic / cluster bootstrap", ylabel="Density")
        ax[1].legend(fontsize=6)
        panel_label(ax[1], "b")
        fig.tight_layout()
        save_fig(
            fig,
            "figS02_delong_vs_bootstrap",
            "Analytic AUROC tests vs the cluster bootstrap: (a) p-value agreement; "
            "(b) standard-error ratios (image-level DeLong "
            "ignores patient clustering and is too narrow).",
        )

In [ ]:
# 6.8  Comparisons across many transfer tasks (Demšar 2006): a task = (unseen-site direction x finding), score =
#      seed-mean AUROC. Friedman + Iman-Davenport F with Kendall's W, Nemenyi critical difference (CD diagram),
#      pairwise Wilcoxon signed-rank (rank-biserial r) and paired t (Cohen's d_z, Hedges' g_av) with Holm, and
#      repeated-measures ANOVA (partial eta^2). Tasks share test images and training sets, so they are not independent:
#      these results are SECONDARY and descriptive; the paired bootstrap contrasts above carry the inference (Benavoli
#      et al. 2016).
def task_matrix(df: pd.DataFrame, by: str, keep, index_cols) -> pd.DataFrame:
    g = df[df[by].isin(keep)].groupby(index_cols + [by]).auroc.mean().unstack(by)
    return g.reindex(columns=[k for k in keep if k in g.columns]).dropna()


MTX = {}
with timed("6 multi-task tests"):
    ext_mt = MT[~MT.seen]
    MTX["architectures (ERM natural)"] = task_matrix(
        ext_mt[ext_mt.method == NAT], "arch", ARCHS, ["train", "test", "finding"]
    )
    if CLASSICAL in ARCH_ALL:
        MTX["architectures + classical baseline"] = task_matrix(
            ext_mt[ext_mt.method == NAT], "arch", ARCHS + [CLASSICAL], ["train", "test", "finding"]
        )
    pool_mt = ext_mt[ext_mt.train.isin(POOLED)]
    MTX["training methods (4, all architectures)"] = task_matrix(
        pool_mt, "method", METHODS[:4], ["arch", "train", "finding"]
    )
    MTX["training methods (5, complete blocks)"] = task_matrix(pool_mt, "method", METHODS, ["arch", "train", "finding"])
    MT_RES, MT_PAIRS = {}, {}
    for nm, X_ in MTX.items():
        if X_.shape[0] < 5 or X_.shape[1] < 2:
            continue
        fr = friedman_nemenyi(X_)
        pw = pairwise_wilcoxon_holm(X_)
        rm = rm_anova(X_)
        MT_RES[nm] = dict(
            tasks=fr["N"],
            models=fr["k"],
            chi2=fr["chi2"],
            p_friedman=fr["p_friedman"],
            F_ID=fr["F_iman_davenport"],
            p_ID=fr["p_iman_davenport"],
            kendalls_W=fr["kendalls_W"],
            CD=fr["cd_nemenyi"],
            rmANOVA_F=rm.get("F"),
            rmANOVA_p=rm.get("p"),
            partial_eta2=rm.get("partial_eta2"),
            mean_ranks="; ".join(f"{k} {v:.2f}" for k, v in fr["mean_ranks"].items()),
        )
        MT_PAIRS[nm] = pw.assign(analysis=nm)
        MT_RES[nm]["_ranks"] = fr["mean_ranks"]
    MTR = pd.DataFrame({k: {kk: vv for kk, vv in v.items() if kk != "_ranks"} for k, v in MT_RES.items()}).T
    save_tab(
        MTR,
        "table32_friedman_nemenyi",
        "Omnibus tests across transfer tasks (Friedman, Iman-Davenport, Kendall's "
        "W, Nemenyi CD, repeated-measures ANOVA).",
    )
    PW = pd.concat(MT_PAIRS.values(), ignore_index=True) if MT_PAIRS else pd.DataFrame()
    if len(PW):
        save_tab(
            PW[
                [
                    "analysis",
                    "A",
                    "B",
                    "n_tasks",
                    "mean_diff",
                    "wins",
                    "losses",
                    "p_wilcoxon",
                    "p_wilcoxon_holm",
                    "rank_biserial",
                    "p_ttest",
                    "p_ttest_holm",
                    "cohens_dz",
                    "hedges_gav",
                    "shapiro_p_diff",
                ]
            ],
            "table33_pairwise_tests",
            "Pairwise Wilcoxon signed-rank and paired t-tests across tasks "
            "(Holm within analysis) with effect sizes (rank-biserial r; Cohen's "
            "d_z; Hedges' g_av). Shapiro-Wilk p of the differences guides "
            "which test to read.",
            index=False,
        )
    show_ = [k for k in ["architectures (ERM natural)", "training methods (4, all architectures)"] if k in MT_RES]
    if show_:
        fig, ax = plt.subplots(1, len(show_), figsize=(COL2, 1.7))
        for a_, k in zip(np.atleast_1d(ax), show_):
            cd_diagram(
                a_,
                MT_RES[k]["_ranks"],
                MT_RES[k]["CD"],
                f"{k}\nN = {MT_RES[k]['tasks']} tasks, Friedman p = {MT_RES[k]['p_friedman']:.2g}",
            )
        fig.tight_layout()
        save_fig(
            fig,
            "fig14_cd_diagrams",
            "Critical-difference diagrams (mean rank, 1 = best; bars join models "
            "whose ranks differ by less than the Nemenyi CD).",
        )

In [ ]:
# 6.9  Which design factor explains unseen-site performance? Factorial ANOVA (Type II, unbalanced) on per-seed,
#      per-finding AUROC, with partial eta^2 and omega^2 as effect sizes, plus the share of variance that is pure seed
#      (training) noise.
with timed("6 factorial ANOVA"):
    ANOVA = {}
    if HAS_SM:
        # five findings labelled at every site: a complete direction x finding layout keeps the design full-rank
        # (with Edema, the VinBigData cells are empty and Type II sums of squares are not uniquely defined)
        dfa = MT[(~MT.seen) & MT.train.isin(POOLED) & MT.arch.isin(ARCHS) & MT.finding.isin(FIND5)].copy()
        dfa["direction"] = dfa.train + "→" + dfa.test
        if dfa.method.nunique() > 1 and dfa.arch.nunique() > 1:
            fit = smf.ols(
                "auroc ~ C(arch) * C(method) + C(direction) + C(finding) + C(direction):C(finding)", data=dfa
            ).fit()
            ANOVA["DG experiment (pooled -> unseen)"] = anova_effects(anova_lm(fit, typ=2))
        dfn = MT[(~MT.seen) & (MT.method == NAT) & MT.arch.isin(ARCHS) & MT.finding.isin(FIND5)].copy()
        dfn["direction"] = dfn.train + "→" + dfn.test
        if dfn.arch.nunique() > 1:
            fit2 = smf.ols(
                "auroc ~ C(arch) + C(direction) + C(finding) + C(arch):C(direction) + C(direction):C(finding)", data=dfn
            ).fit()
            ANOVA["transfer experiment (ERM natural, 9 directions)"] = anova_effects(anova_lm(fit2, typ=2))
        for nm, t in ANOVA.items():
            save_tab(
                t.round(5),
                f"table34_anova_{'dg' if nm.startswith('DG') else 'transfer'}",
                f"Type II ANOVA, {nm}, five findings labelled at every site: partial eta^2 and omega^2.",
            )
    else:
        print("statsmodels missing: ANOVA skipped")
    cellcols = ["arch", "train", "method", "test", "finding"]
    vd = MT[~MT.seen].groupby(cellcols).auroc.agg(["var", "count"]).dropna()
    seed_var = float((vd["var"] * (vd["count"] - 1)).sum() / (vd["count"] - 1).sum())
    tot_var = float(MT[~MT.seen].auroc.var())
    VARDEC = pd.Series(
        {
            "pooled within-configuration (seed) variance": seed_var,
            "total variance of unseen-site AUROC": tot_var,
            "share due to seeds": seed_var / tot_var,
            "seed SD (AUROC units)": np.sqrt(seed_var),
        }
    )
    display(VARDEC.to_frame("value"))
    if ANOVA:
        fig, ax = plt.subplots(1, len(ANOVA), figsize=(COL2, 2.6))
        for a_, (nm, t) in zip(np.atleast_1d(ax), ANOVA.items()):
            e = t.drop(index="Residual")["partial_eta2"].sort_values()
            a_.barh(range(len(e)), e.values, color="0.45")
            a_.set_yticks(range(len(e)))
            a_.set_yticklabels([i.replace("C(", "").replace(")", "") for i in e.index], fontsize=6)
            a_.set(xlabel="partial η²", title=nm)
        fig.tight_layout()
        save_fig(
            fig,
            "fig15_factor_importance_anova",
            "Design-factor importance: partial eta^2 from Type II factorial ANOVA of unseen-site AUROC.",
        )

In [ ]:
# 6.10 Per-image error distributions. Error of an image = mean squared probability error over its labelled findings
#      (Brier contribution). Between unseen sites (independent images): Kruskal-Wallis + epsilon^2 and Dunn-Holm
#      post-hoc with Cliff's delta. Between architectures on the same images: Friedman across images with Kendall's W
#      (with ~10^4 images every p-value is tiny, so the effect sizes, not the p-values, are the result).
def image_error(P, T, findings):
    return np.mean((P[:, findings] - YT[T][:, findings]) ** 2, 1)


with timed("6 error-distribution tests"):
    ERR, krows, frows, DUNN = {}, [], [], None
    for a in ARCH_ALL:
        for ts, T in EXT_DIRS + SEEN_DIRS:
            k = (a, ts, NAT)
            if k in KEYS and all(T in PS[(*k, s)].test for s in SEEDS_OF[k]):
                ERR[(a, ts, T)] = image_error(
                    np.mean([PS[(*k, s)].test[T] for s in SEEDS_OF[k]], 0), T, model_findings(ts, T)
                )
    for a in ARCH_ALL:
        groups = {
            f"{dname(ts, T)}": ERR[(a, ts, T)] for ts, T in EXT_DIRS if (a, ts, T) in ERR and ts in ("A", "B", "A+B")
        }
        if len(groups) >= 3:
            kw = kruskal_effect(groups)
            krows.append(dict(arch=a, groups=", ".join(groups), **kw))
            if a == ARCHS[0]:
                DUNN = dunn_holm(groups)
    KW = pd.DataFrame(krows)
    if len(KW):
        save_tab(
            KW,
            "table35_kruskal_error_by_site",
            "Kruskal-Wallis test of per-image Brier error across unseen-site directions (epsilon^2 = effect size).",
            index=False,
        )
    if DUNN is not None:
        save_tab(
            DUNN, "table36_dunn_posthoc", f"Dunn post-hoc test (Holm) with Cliff's delta, {ARCHS[0]}.", index=False
        )
    for ts, T in EXT_DIRS:
        ks = [a for a in ARCHS if (a, ts, T) in ERR]
        if len(ks) == len(ARCHS):
            M_ = np.column_stack([ERR[(a, ts, T)] for a in ks])
            chi2, p = stats.friedmanchisquare(*M_.T)
            N_ = len(M_)
            frows.append(
                dict(
                    direction=dname(ts, T),
                    n_images=N_,
                    chi2=chi2,
                    p=p,
                    kendalls_W=chi2 / (N_ * (len(ks) - 1)),
                    **{f"median error {a}": float(np.median(ERR[(a, ts, T)])) for a in ks},
                )
            )
    if frows:
        save_tab(
            pd.DataFrame(frows),
            "table37_friedman_error_by_architecture",
            "Friedman test of per-image error across architectures on the same images "
            "(Kendall's W = effect size; W < 0.1 = negligible).",
            index=False,
        )

In [ ]:
# 6.11 Few-label adaptation evaluated with repeated patient-grouped K-fold CV ON THE TARGET SITE.
#      In every fold, k labelled target images are drawn from the fitting folds, each recalibration model is fitted on
#      them, and calibration (ECE, Brier, calibration-in-the-large) and F1 are measured on the held-out patients.
#      Intervals and paired tests use the Nadeau-Bengio correction. AUROC is unchanged by construction (monotone maps),
#      so it is not reported here.
RECAL_DIRS = [("A", "B"), ("B", "A"), ("A+B", "C"), ("A+C", "B"), ("B+C", "A")]
with timed("6 recalibration CV"):
    rrows = []
    cells_ = [(a, ts, T) for a in ARCH_ALL for ts, T in RECAL_DIRS if (a, ts, NAT) in KEYS]
    for a, ts, T in tqdm(cells_, desc="recalibration CV"):
        Y = YT[T]
        js = model_findings(ts, T)
        ystrat = np.minimum(Y[:, js].sum(1), 2)
        folds = list(repeated_group_kfold(ystrat, GID[T], CFG.cv_k, CFG.recal_cv_r, f"recal-{T}"))
        prior_src = np.concatenate([SRC[k]["Y"][SRC[k]["tr"]] for k in ts.split("+")]).mean(0)
        for s in SEEDS_OF[(a, ts, NAT)]:
            ps = PS[(a, ts, NAT, s)]
            Pt = ps.test[T]
            Z = logit(Pt)
            thr, _ = THR[(a, ts, NAT, s)]
            em = {j: saerens_em(Pt[:, j], float(prior_src[j]))[0] for j in js}
            for rep, fold, fit_idx, ho in folds:
                n_fit = len(fit_idx)
                rng_ = rng_for("recal-draw", a, ts, T, s, rep, fold)
                for k in [*CFG.recal_budgets, "all"]:
                    if k != "all" and k > n_fit:
                        continue
                    sel = fit_idx if k == "all" else rng_.choice(fit_idx, k, replace=False)
                    for meth in RECAL_METHODS + ["EM (label-free)"]:
                        if meth in ("none (source threshold)", "EM (label-free)") and k != CFG.recal_budgets[0]:
                            continue  # label-free methods do not depend on k
                        e_, b_, f_, c_, fitted = [], [], [], [], 0
                        for j in js:
                            if meth == "EM (label-free)":
                                pe, t_, ok_ = em[j][ho], (thr[j] if np.isfinite(thr[j]) else 0.5), True
                            else:
                                pe, t_, ok_ = recalibrate(
                                    meth, Z[sel, j], Y[sel, j], Z[ho, j], thr[j] if np.isfinite(thr[j]) else 0.5
                                )
                            yh = Y[ho, j]
                            e_.append(ece_score(yh, pe))
                            b_.append(np.mean((pe - yh) ** 2))
                            c_.append(pe.mean() - yh.mean())
                            f_.append(binary_report(yh, pe, t_)["f1"])
                            fitted += ok_
                        rrows.append(
                            dict(
                                arch=a,
                                direction=dname(ts, T),
                                group="single source" if "+" not in ts else "pooled → unseen",
                                seed=s,
                                rep=rep,
                                fold=fold,
                                k=0 if meth in ("none (source threshold)", "EM (label-free)") else k,
                                method=meth,
                                ece=np.mean(e_),
                                brier=np.mean(b_),
                                citl=np.mean(c_),
                                f1=np.mean(f_),
                                fitted_share=fitted / len(js),
                                n_fit=n_fit,
                                n_test=len(ho),
                            )
                        )
    RC = pd.DataFrame(rrows)
    RC.to_csv(TAB / "table38_recalibration_cv_raw.csv", index=False)

    def kk(v):
        """Sort key for label budgets ('all' last)."""
        return 10**9 if v == "all" else v

    summ = []
    for (grp, meth, k), g in RC.groupby(["group", "method", "k"]):
        fm = g.groupby(["rep", "fold"])[["ece", "f1", "brier", "citl", "fitted_share"]].mean()
        for met in ["ece", "f1", "brier"]:
            summ.append(
                dict(
                    group=grp,
                    method=meth,
                    k=k,
                    metric=met,
                    fitted_share=fm.fitted_share.mean(),
                    **corrected_t_ci(fm[met].values, g.n_fit.mean(), g.n_test.mean()),
                )
            )
    RCS = pd.DataFrame(summ)
    RCS["k_sort"] = RCS.k.map(kk)
    RCS = RCS.sort_values(["group", "metric", "method", "k_sort"]).drop(columns="k_sort")
    save_tab(
        RCS[RCS.k.isin([0, 100, 500, "all"])].round(4),
        "table39_recalibration_cv",
        f"Few-label recalibration, {CFG.recal_cv_r} x {CFG.cv_k}-fold patient-grouped CV "
        "on the target site (mean over architectures and seeds; Nadeau-Bengio corrected 95% CI).",
        index=False,
    )
    k_ref = 500 if 500 in CFG.recal_budgets else CFG.recal_budgets[-1]
    prow = []
    for grp, g in RC.groupby("group"):
        base0 = g[g.method == "none (source threshold)"].groupby(["rep", "fold"])[["ece", "f1"]].mean()
        gk = g[g.k == k_ref].groupby(["method", "rep", "fold"])[["ece", "f1"]].mean()
        for m1, m2 in [
            ("Platt", None),
            ("bias", None),
            ("isotonic", None),
            ("temperature", None),
            ("threshold only", None),
            ("Platt", "bias"),
            ("Platt", "isotonic"),
            ("Platt", "temperature"),
        ]:
            if m1 not in gk.index.get_level_values(0):
                continue
            ref = base0 if m2 is None else gk.loc[m2]
            for met in ["ece", "f1"]:
                d = (gk.loc[m1][met] - ref[met]).dropna()
                prow.append(
                    dict(
                        group=grp,
                        k=k_ref,
                        comparison=f"{m1} − {m2 or 'no adaptation'}",
                        metric=met,
                        **corrected_paired_t(d.values, g.n_fit.mean(), g.n_test.mean()),
                    )
                )
    RCP = pd.DataFrame(prow)
    if len(RCP):
        RCP["p_holm"] = holm(RCP.p.values)
        save_tab(
            RCP.round(5),
            "table40_recalibration_tests",
            f"Corrected resampled paired t-tests between adaptation methods at k = {k_ref} labelled target images.",
            index=False,
        )
    fig, ax = plt.subplots(2, 2, figsize=(COL2, 4.8), sharex=True)
    cols_m = dict(zip(RECAL_METHODS + ["EM (label-free)"], sns.color_palette("colorblind", 7)))
    for c_, met in enumerate(["ece", "f1"]):
        for r_, grp in enumerate(["single source", "pooled → unseen"]):
            a_ = ax[r_, c_]
            g = RCS[(RCS.group == grp) & (RCS.metric == met)]
            for meth, gg in g.groupby("method"):
                gg = gg[gg.k != "all"]
                if (gg.k == 0).all():
                    a_.axhline(gg["mean"].iloc[0], color=cols_m[meth], ls="--", lw=0.8, label=meth)
                    continue
                x = gg.k.astype(float)
                a_.plot(x, gg["mean"], marker="o", ms=2.5, color=cols_m[meth], label=meth)
                a_.fill_between(x, gg.ci_low, gg.ci_high, color=cols_m[meth], alpha=0.15, lw=0)
            a_.set_xscale("log")
            a_.set_xticks([k_ for k_ in CFG.recal_budgets])
            a_.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
            a_.xaxis.set_minor_formatter(mpl.ticker.NullFormatter())
            a_.tick_params(axis="x", which="minor", length=0)
            a_.set(
                title=f"{grp}: {'ECE (lower is better)' if met == 'ece' else 'macro-F1'}",
                xlabel="k labelled target images" if r_ else "",
            )
    h_, l_ = ax[0, 0].get_legend_handles_labels()
    fig.tight_layout()
    fig.legend(h_, l_, fontsize=6, ncol=4, loc="lower center", bbox_to_anchor=(0.5, 1.0))
    save_fig(
        fig,
        "fig16_recalibration_cv",
        "Few-label adaptation on the target site: ECE and F1 against the number of labelled target images "
        "(repeated patient-grouped CV; bands = corrected 95% CI; dashed = label-free methods).",
    )

In [ ]:
# 6.12 Precision of the study: minimal detectable effects, bootstrap convergence, and where the uncertainty comes from
#      (training seeds vs test-set sampling; Bouthillier et al. 2021).
with timed("6 power and variance sources"):
    H1 = finalize_family(HYP)
    MDE = H1.groupby("family").agg(
        contrasts=("estimate", "size"),
        median_se=("se", "median"),
        median_mde_80=("mde_80", "median"),
        significant_holm=("significant_holm", "sum"),
        equivalent=("verdict", lambda v: int(v.str.startswith("equivalent").sum())),
    )
    save_tab(
        MDE.round(4),
        "table41_power_mde",
        "Per family: median bootstrap SE and minimal detectable AUROC difference at 80% power (2.8 x SE), "
        "number of Holm-significant and equivalent contrasts.",
    )
    conv = []
    for nm, (ka, kb, f) in {
        "asymmetry, DenseNet121": (None, None, None),
        "ConvNeXt-T − DenseNet121, A+B→C": (
            (("ConvNeXt-T", "A+B", NAT), "C"),
            (("DenseNet121", "A+B", NAT), "C"),
            None,
        ),
        "DANN − ERM (natural), DenseNet121, A+B→C": (
            (("DenseNet121", "A+B", "DANN"), "C"),
            (("DenseNet121", "A+B", NAT), "C"),
            None,
        ),
    }.items():
        if ka is None:
            if "DenseNet121" not in GAP_DRAWS:
                continue
            d_full = GAP_DRAWS["DenseNet121"]["asym"]
        elif ka in BR and kb in BR:
            d_full = macro_draws(BR[ka]) - macro_draws(BR[kb])
        else:
            continue
        for b_ in [n for n in (100, 250, 500, 1000, 2000, 5000, 10000) if n <= len(d_full)]:
            d = d_full[:b_]
            lo, hi = np.percentile(d, [2.5, 97.5])
            conv.append(dict(contrast=nm, B=b_, ci_width=hi - lo, se=d.std(ddof=1)))
    CONV = pd.DataFrame(conv)
    vrows = []
    for (k, T), r in BR.items():
        if len(r.seeds) < 2 or is_seen(k[1], T):
            continue
        per_seed = np.nanmean(r.point["auroc"][:, r.findings], 1)
        boot_seed0 = np.nanmean(r.boot["auroc"][:, 0, r.findings], 1)
        vrows.append(
            dict(
                arch=k[0],
                method=k[2],
                direction=dname(k[1], T),
                seed_sd=per_seed.std(ddof=1),
                test_sampling_sd=np.nanstd(boot_seed0, ddof=1),
            )
        )
    VARS = pd.DataFrame(vrows)
    if len(VARS):
        VARS["seed_share_of_variance"] = VARS.seed_sd**2 / (VARS.seed_sd**2 + VARS.test_sampling_sd**2)
        save_tab(
            VARS.groupby("arch")[["seed_sd", "test_sampling_sd", "seed_share_of_variance"]].median().round(4),
            "table42_variance_sources",
            "Median SD of the unseen-site macro AUROC due to training seeds vs test-set (patient) sampling.",
        )
    fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.6))
    for nm, g in CONV.groupby("contrast"):
        ax[0].plot(g.B, g.ci_width, marker="o", ms=3, label=nm)
    ax[0].set(xscale="log", xlabel="Bootstrap replicates B", ylabel="95% CI width (AUROC)")
    ax[0].legend(fontsize=5.5)
    panel_label(ax[0], "a")
    if len(VARS):
        ax[1].scatter(VARS.test_sampling_sd, VARS.seed_sd, s=10, c=[ARCH_COL.get(a, "k") for a in VARS.arch], alpha=0.8)
        m_ = float(max(VARS.test_sampling_sd.max(), VARS.seed_sd.max()))
        ax[1].plot([0, m_], [0, m_], "k:", lw=0.6)
        ax[1].set(xlabel="SD from test-set sampling (one seed)", ylabel="SD across training seeds")
        ax[1].legend(
            handles=[mpl.patches.Patch(color=ARCH_COL[a], label=a) for a in VARS.arch.unique() if a in ARCH_COL],
            fontsize=5.5,
        )
        panel_label(ax[1], "b")
    fig.tight_layout()
    save_fig(
        fig,
        "figS03_bootstrap_convergence_variance",
        "(a) Convergence of bootstrap CI width with B; (b) training-seed vs "
        "test-sampling variability per unseen-site cell.",
    )

In [ ]:
# 7.1  Hyperparameter impact (Stage 3, equal tuning budget: 6 trials per method, 4 epochs, DenseNet121, NIH + CheXpert
#      -> VinBigData unseen) How much do learning rate, weight decay, sampler and the DG strength move unseen-site
#      AUROC, and can SOURCE validation pick a good configuration (Gulrajani & Lopez-Paz 2021)? Collapsed trials
#      (source-validation AUROC < 0.6) are flagged and analysed separately.
TR3, LG3 = load_stage3()
S3 = None
with timed("7 stage 3 hyperparameters"):
    if TR3 is not None and len(LG3):
        hist_rows = []
        for r in TR3.itertuples():
            f = P["stage3"] / "runs" / f"{r.trial_id}_seed0_history.csv"
            if f.exists():
                h = pd.read_csv(f)
                i = int(h.val_mean.idxmax())
                hist_rows.append(dict(trial_id=r.trial_id, best_epoch=int(h.epoch[i]), val_mean=float(h.val_mean[i])))
        tv = pd.DataFrame(hist_rows)
        if not len(tv) and (P["stage3"] / "trials_source_validation.csv").exists():
            tv = pd.read_csv(P["stage3"] / "trials_source_validation.csv")[["trial_id", "best_epoch", "val_mean"]]
        S3 = TR3.merge(tv, on="trial_id", how="inner")
        for T in "ABC":
            S3[f"auroc_{T}"] = [
                np.mean([auroc(YT[T][:, j], LG3[(t, 0)][T][:, j]) for j in IDX5]) if (t, 0) in LG3 else np.nan
                for t in S3.trial_id
            ]
        S3["collapsed"] = S3.val_mean < 0.6
        core = S3[~S3.diagnostic.astype(bool)]

        def fx(v):
            return np.log10(v.astype(float))

        X3 = pd.DataFrame(
            {
                "log10 lr": fx(core.lr),
                "log10 wd": fx(core.wd),
                "balanced sampler": (core.sampler == "balanced").astype(float),
                "log10 DG strength": np.where(core.method == "ERM", np.nan, fx(core.lam.fillna(core.eta))),
            }
        )
        ok = ~core.collapsed.values
        rows = []
        for c_ in ["log10 lr", "log10 wd", "balanced sampler", "log10 DG strength"]:
            m_ = ok & np.isfinite(X3[c_].values)
            if m_.sum() >= 5:
                rho = stats.spearmanr(X3[c_].values[m_], core.auroc_C.values[m_])[0]
                bs = []
                for b_ in range(2000):
                    ii = rng_for("hp-boot", c_, b_).integers(0, m_.sum(), m_.sum())
                    bs.append(stats.spearmanr(X3[c_].values[m_][ii], core.auroc_C.values[m_][ii])[0])
                rows.append(
                    dict(
                        hyperparameter=c_,
                        n_trials=int(m_.sum()),
                        spearman_with_unseen_auroc=rho,
                        ci_low=np.nanpercentile(bs, 2.5),
                        ci_high=np.nanpercentile(bs, 97.5),
                    )
                )
        HPI = save_tab(
            pd.DataFrame(rows),
            "table44_hyperparameter_impact",
            "Spearman correlation of each hyperparameter with VinBigData AUROC across "
            "non-collapsed tuning trials (trial-bootstrap 95% CI; n is small: descriptive).",
            index=False,
        )
        Xd = pd.concat(
            [
                X3.drop(columns="log10 DG strength"),
                pd.get_dummies(core.method, prefix="method", drop_first=True).astype(float).set_index(X3.index),
            ],
            axis=1,
        )[ok]
        yd = core.auroc_C.values[ok]
        if len(yd) >= 8:
            rf = RandomForestRegressor(n_estimators=500, min_samples_leaf=2, random_state=0).fit(Xd, yd)
            from sklearn.inspection import permutation_importance

            pi = permutation_importance(rf, Xd, yd, n_repeats=50, random_state=0)
            RFI = pd.DataFrame(
                dict(feature=Xd.columns, importance_mean=pi.importances_mean, importance_sd=pi.importances_std)
            ).sort_values("importance_mean", ascending=False)
            save_tab(
                RFI,
                "table45_hyperparameter_rf_importance",
                "Random-forest permutation importance of hyperparameters for unseen-site "
                "AUROC (in-sample, descriptive).",
                index=False,
            )
        sel = {
            m_: g.sort_values("val_mean", ascending=False).iloc[0] for m_, g in core[~core.collapsed].groupby("method")
        }
        rows = []
        for m_, g in core.groupby("method"):
            d0 = g[g.trial == 0].iloc[0]
            o = g.sort_values("auroc_C").iloc[-1]
            s_ = sel.get(m_, d0)
            rho_m = stats.spearmanr(g.val_mean, g.auroc_C)[0] if len(g) > 2 else np.nan
            rows.append(
                dict(
                    method=m_,
                    trials=len(g),
                    collapsed=int(g.collapsed.sum()),
                    default_C=d0.auroc_C,
                    selected_trial=s_.trial_id,
                    selected_C=s_.auroc_C,
                    oracle_trial=o.trial_id,
                    oracle_C=o.auroc_C,
                    regret=o.auroc_C - s_.auroc_C,
                    spearman_val_vs_C=rho_m,
                )
            )
        REG3 = save_tab(
            pd.DataFrame(rows),
            "table46_source_selection_regret",
            "Model selection on SOURCE validation vs the target oracle (seed 0; oracle leaks the target).",
            index=False,
        )
        rho_all = stats.spearmanr(core[~core.collapsed].val_mean, core[~core.collapsed].auroc_C)[0]
        print(f"Spearman(source-validation AUROC, VinBigData AUROC) over non-collapsed trials: {rho_all:.3f}")
        TUN = {}
        for m_, r in sel.items():
            probs = {s: LG3[(r.trial_id, s)]["C"] for s in SEEDS if (r.trial_id, s) in LG3}
            if len(probs) == len(SEEDS):
                TUN[m_] = bootstrap_cell(("STAGE3", r.trial_id), "C", probs, metrics=("auroc",), allowed=IDX5)
        if "ERM" in TUN:
            for m_ in [x for x in TUN if x != "ERM"]:
                HYP.append(
                    dict(
                        family="F12 Stage-3 tuned DG vs tuned ERM (unseen VinBigData)",
                        contrast=f"{m_} (tuned) − ERM (tuned)",
                        arch="DenseNet121",
                        direction="A+B→C",
                        **contrast(TUN[m_], TUN["ERM"], findings=IDX5),
                    )
                )
        fig, ax = plt.subplots(1, 3, figsize=(COL2, 2.6))
        mcol = {
            "ERM": METHOD_COL[NAT],
            "DANN": METHOD_COL["DANN"],
            "CORAL": METHOD_COL["CORAL"],
            "GroupDRO": METHOD_COL["GroupDRO"],
        }
        for m_, g in core.groupby("method"):
            ax[0].scatter(
                g.val_mean, g.auroc_C, s=14, color=mcol.get(m_, "k"), marker="x" if g.collapsed.any() else "o", label=m_
            )
            ax[1].scatter(fx(g.lr), g.auroc_C, s=14, color=mcol.get(m_, "k"))
            ax[2].scatter(fx(g.wd), g.auroc_C, s=14, color=mcol.get(m_, "k"))
        ax[0].set(xlabel="Source-validation AUROC (selection signal)", ylabel="Unseen VinBigData AUROC")
        ax[0].legend(fontsize=6)
        panel_label(ax[0], "a")
        ax[1].set(xlabel="log10 learning rate")
        ax[2].set(xlabel="log10 weight decay")
        panel_label(ax[1], "b")
        panel_label(ax[2], "c")
        fig.tight_layout()
        save_fig(
            fig,
            "fig17_hyperparameter_impact",
            "Stage 3 equal-budget tuning: (a) source-validation signal vs unseen-site AUROC (x = collapsed trials); "
            "(b, c) unseen-site AUROC against learning rate and weight decay.",
        )
    else:
        print("Stage 3 artifacts not found in", P["stage3"])
H1 = finalize_family(HYP)

In [ ]:
# 7.2  Sensitivity to analysis choices that are not part of the model:
#   (a) VinBigData reference standard: >= 2 of 3 radiologists (main) vs any radiologist vs images with exactly
#       one marking removed;
#   (b) operating threshold: source max-F1 (main), source Youden, fixed 0.5, label-free prevalence matching,
#       target-oracle max-F1 (upper bound);
#   (c) calibration estimator: equal-width ECE with 10/15/20 bins vs equal-mass ACE; rankings compared with
#       Kendall's tau.
def _source_pairs(ps, ts, j):
    ys, pp = [], []
    for k in ts.split("+"):
        if SRC[k]["valid"][j]:
            if k in ps.val:
                ys.append(VALY[k][:, j])
                pp.append(ps.val[k][:, j])
            elif k in ps.test and k in "AB":
                ys.append(YT[k][:, j])
                pp.append(ps.test[k][:, j])
    return (np.concatenate(ys), np.concatenate(pp)) if ys else (None, None)


with timed("7 sensitivity: labels, thresholds, ECE"):
    rows = []
    Yany = np.zeros_like(Yvb)
    Ncnt = np.zeros_like(Yvb, dtype=np.int16)
    for f_ in FIND5:
        j = LABELS.index(f_)
        if f"{f_}_any" in vb:
            Yany[:, j] = vb[f"{f_}_any"].values
            Ncnt[:, j] = vb[f"{f_}_n"].values
    for (k, T), r in BR.items():
        if T != "C":
            continue
        Pm = {s: PS[(*k, s)].test["C"] for s in SEEDS_OF[k]}
        for rule, Yr, excl in [
            ("≥2 of 3 radiologists (main)", Yvb, None),
            ("any radiologist", Yany, None),
            ("exactly-one-rater images removed", Yvb, Ncnt == 1),
        ]:
            v = []
            for j in IDX5:
                keep = np.ones(NVB, bool) if excl is None else ~excl[:, j]
                if Yr[keep, j].sum() >= 5:
                    v.append(np.mean([auroc(Yr[keep, j], Pm[s][keep, j]) for s in Pm]))
            rows.append(dict(arch=k[0], train=k[1], method=k[2], rule=rule, auroc=np.mean(v) if v else np.nan))
    LR_ = pd.DataFrame(rows)
    if len(LR_) and LR_.rule.nunique() > 1:
        piv = LR_.pivot_table(index=["arch", "train", "method"], columns="rule", values="auroc")
        tau = {c_: stats.kendalltau(piv.iloc[:, 0], piv[c_])[0] for c_ in piv.columns[1:]}
        save_tab(
            piv.round(4),
            "table47_label_rule_sensitivity",
            "VinBigData macro AUROC (5 findings) under three reference-standard rules. Kendall's tau of model "
            "rankings vs the main rule: " + "; ".join(f"{k_}: {v:.2f}" for k_, v in tau.items()),
        )
    trows = []
    for (a, ts, m_, s), ps in PS.items():
        for T in ps.test:
            if is_seen(ts, T):
                continue
            for j in model_findings(ts, T):
                y, p = YT[T][:, j], ps.test[T][:, j]
                ysrc, psrc = _source_pairs(ps, ts, j)
                if ysrc is None:
                    continue
                prev_src = ysrc.mean()
                rules = {
                    "source max-F1 (main)": best_f1_threshold(ysrc, psrc),
                    "source Youden": youden_threshold(ysrc, psrc),
                    "fixed 0.5": 0.5,
                    "label-free prevalence matching": float(np.quantile(p, 1 - prev_src)),
                    "target-oracle max-F1 (upper bound)": best_f1_threshold(y, p),
                }
                for rn, t_ in rules.items():
                    br_ = binary_report(y, p, t_)
                    trows.append(
                        dict(
                            arch=a,
                            method=m_,
                            direction=dname(ts, T),
                            rule=rn,
                            f1=br_["f1"],
                            precision=br_["precision"],
                            recall=br_["recall"],
                            specificity=br_["specificity"],
                            mcc=br_["mcc"],
                        )
                    )
    THRS = pd.DataFrame(trows)
    if len(THRS):
        ttab = THRS.groupby(["rule", "arch"])[["f1", "precision", "recall", "specificity", "mcc"]].mean().round(4)
        save_tab(
            ttab,
            "table48_threshold_rule_sensitivity",
            "Unseen-site threshold metrics under five operating-point rules (macro over findings, mean over models).",
        )
    erows = []
    for (a, ts, m_, s), ps in PS.items():
        for T in ps.test:
            if not is_seen(ts, T):
                js = model_findings(ts, T)
                Y_, P_ = YT[T][:, js], ps.test[T][:, js]
                erows.append(
                    dict(
                        model=f"{a} | {ts} | {m_}",
                        seed=s,
                        direction=dname(ts, T),
                        **{
                            nm: np.mean([ece_score(Y_[:, i], P_[:, i], nb, st) for i in range(len(js))])
                            for nm, nb, st in [
                                ("ECE-10", 10, "uniform"),
                                ("ECE-15", 15, "uniform"),
                                ("ECE-20", 20, "uniform"),
                                ("ACE-10", 10, "quantile"),
                                ("ACE-15", 15, "quantile"),
                            ]
                        },
                    )
                )
    ECS = pd.DataFrame(erows)
    if len(ECS):
        em_ = ECS.groupby(["model", "direction"])[["ECE-10", "ECE-15", "ECE-20", "ACE-10", "ACE-15"]].mean()
        kt = pd.DataFrame({a_: {b_: stats.kendalltau(em_[a_], em_[b_])[0] for b_ in em_.columns} for a_ in em_.columns})
        save_tab(
            kt.round(3),
            "table49_ece_estimator_agreement",
            "Kendall's tau between model rankings by different calibration-error estimators (unseen sites).",
        )
    if len(THRS):
        fig, ax = plt.subplots(figsize=(COL2, 2.6))
        tt = THRS.groupby(["rule", "arch"]).f1.mean().unstack("arch").reindex(columns=ARCH_ALL)
        tt.plot.bar(ax=ax, color=[ARCH_COL[a] for a in tt.columns], width=0.8, edgecolor="k", lw=0.3)
        ax.set(xlabel="", ylabel="Macro-F1 on the unseen site")
        ax.tick_params(axis="x", rotation=15)
        ax.legend(fontsize=6, ncol=4)
        fig.tight_layout()
        save_fig(
            fig,
            "fig18_threshold_rules",
            "Unseen-site macro-F1 under five operating-point rules (the oracle "
            "rule uses target labels: upper bound only).",
        )

In [ ]:
# 7.3  Robustness of the headline contrasts to (i) individual seeds, (ii) leaving one finding out, (iii) the ensemble
#      aggregation rule, and (iv) the bootstrap unit (patient vs image): ignoring patient clustering makes intervals
#      too narrow.
with timed("7 robustness"):
    rob = []
    heads = [(f"asymmetry {a}", "asym", a) for a in ARCH_ALL if a in GAP_DRAWS]
    for a in ARCHS:
        for m_ in METHODS[1:]:
            if ((a, "A+B", m_), "C") in BR and ((a, "A+B", NAT), "C") in BR:
                heads.append((f"{m_} − ERM (natural), {a}, A+B→C", "dg", (a, m_)))
    for nm, kind, arg in heads:
        if kind == "asym":
            aa, ab, bb, ba = (BR[((arg, s_, NAT), t_)] for s_, t_ in [("A", "A"), ("A", "B"), ("B", "B"), ("B", "A")])
            per_seed = [
                (np.nanmean(aa.point["auroc"][i, B6]) - np.nanmean(ab.point["auroc"][i, B6]))
                - (np.nanmean(bb.point["auroc"][i, B6]) - np.nanmean(ba.point["auroc"][i, B6]))
                for i in range(min(len(aa.seeds), len(bb.seeds)))
            ]
            lofo = [
                (
                    (macro_point(aa, "auroc", f) - macro_point(ab, "auroc", f))
                    - (macro_point(bb, "auroc", f) - macro_point(ba, "auroc", f))
                )
                for f in [[x for x in B6 if x != j] for j in B6]
            ]
        else:
            a, m_ = arg
            r1, r0 = BR[((a, "A+B", m_), "C")], BR[((a, "A+B", NAT), "C")]
            per_seed = [
                np.nanmean(r1.point["auroc"][i, IDX5]) - np.nanmean(r0.point["auroc"][i, IDX5])
                for i in range(min(len(r1.seeds), len(r0.seeds)))
            ]
            lofo = [
                macro_point(r1, "auroc", f) - macro_point(r0, "auroc", f)
                for f in [[x for x in IDX5 if x != j] for j in IDX5]
            ]
        rob.append(
            dict(
                contrast=nm,
                per_seed=" / ".join(f"{x:+.4f}" for x in per_seed),
                seeds_same_sign=f"{int(np.sum(np.sign(per_seed) == np.sign(np.mean(per_seed))))}/{len(per_seed)}",
                lofo_min=min(lofo),
                lofo_max=max(lofo),
                lofo_sign_stable=bool(np.all(np.sign(lofo) == np.sign(np.mean(lofo)))),
            )
        )
    ROB = save_tab(
        pd.DataFrame(rob),
        "table50_robustness_seeds_lofo",
        "Headline contrasts per seed and with each finding left out (LOFO).",
        index=False,
    )

    agg_rows = []

    def _agg(members, how):
        Ps_ = [p for p, _ in members]
        if how == "logit":
            return logit_ensemble(members)
        if how == "probability":
            return np.mean(Ps_, 0)
        return np.mean([np.apply_along_axis(stats.rankdata, 0, p) / len(p) for p in Ps_], 0)

    for a in ARCHS:
        for T, (s1, s2) in SOURCES_FOR.items():
            for how in ["logit", "probability", "rank"]:
                cr = [
                    _agg([_p(a, s1, s, T), _p(a, s2, s, T)], how)
                    for s in SEEDS
                    if _p(a, s1, s, T) is not None and _p(a, s2, s, T) is not None
                ]
                p2 = [
                    _agg([_p(a, POOL_FOR[T], s, T), _p(a, POOL_FOR[T], s + 1, T)], how)
                    for s in SEEDS
                    if _p(a, POOL_FOR[T], s, T) is not None
                ]
                if cr and p2:
                    agg_rows.append(
                        dict(
                            arch=a,
                            target=T,
                            aggregation=how,
                            cross_minus_pooled_x2=np.mean(
                                [np.mean([auroc(YT[T][:, j], c[:, j]) for j in IDX5]) for c in cr]
                            )
                            - np.mean([np.mean([auroc(YT[T][:, j], c[:, j]) for j in IDX5]) for c in p2]),
                        )
                    )
    if agg_rows:
        save_tab(
            pd.DataFrame(agg_rows)
            .pivot_table(index=["arch", "target"], columns="aggregation", values="cross_minus_pooled_x2")
            .round(4),
            "table51_aggregation_rule",
            "Cross-source ensemble minus two pooled models (equal cost) under three aggregation rules (5 findings).",
        )

    if "DenseNet121" in GAP_DRAWS:  # bootstrap unit: image-level resampling ignores patient clustering
        BOOT_PAT = BOOT
        BOOT = ClusterBootstrap({T: np.arange(len(YT[T])) for T in "AB"}, CFG.n_boot, CFG.boot_chunk)
        try:
            img = {
                (s_, t_): bootstrap_cell(
                    ("IMG", "DenseNet121", s_),
                    t_,
                    {s: PS[("DenseNet121", s_, NAT, s)].test[t_] for s in SEEDS_OF[("DenseNet121", s_, NAT)]},
                    metrics=("auroc",),
                    tag="image-level",
                )
                for s_, t_ in [("A", "A"), ("A", "B"), ("B", "B"), ("B", "A")]
            }
            d_img = (macro_draws(img[("A", "A")], "auroc", B6) - macro_draws(img[("A", "B")], "auroc", B6)) - (
                macro_draws(img[("B", "B")], "auroc", B6) - macro_draws(img[("B", "A")], "auroc", B6)
            )
        finally:
            BOOT = BOOT_PAT
        d_pat = GAP_DRAWS["DenseNet121"]["asym"]
        UNIT = pd.DataFrame(
            [
                dict(unit=u, se=np.std(d, ddof=1), ci_low=np.percentile(d, 2.5), ci_high=np.percentile(d, 97.5))
                for u, d in [
                    ("patient (cluster) bootstrap - used", d_pat),
                    ("image bootstrap - ignores clustering", d_img),
                ]
            ]
        )
        UNIT["se_ratio_vs_patient"] = UNIT.se / UNIT.se.iloc[0]
        save_tab(
            UNIT.round(5),
            "table52_bootstrap_unit",
            "DenseNet121 asymmetry: standard error under patient-level vs image-level resampling.",
            index=False,
        )
H1 = finalize_family(HYP)

In [ ]:
# 7.9  The complete register of pre-specified contrasts: effect, 95% CI, Wald p from the bootstrap SE, percentile p,
#      Holm within family, TOST equivalence p (margin +/- CFG.equivalence_margin, Holm within family),
#      Benjamini-Hochberg across the study, probability of superiority, MDE and the Holm-based verdict.
H1 = finalize_family(HYP)
cols_ = [
    "family",
    "contrast",
    "arch",
    "direction",
    "n_findings",
    "estimate",
    "ci_low",
    "ci_high",
    "se",
    "z",
    "p",
    "p_boot",
    "p_holm",
    "q_bh",
    "p_equiv",
    "p_equiv_holm",
    "significant_holm",
    "prob_superiority",
    "mde_80",
    "verdict",
]
REGISTER = H1[[c for c in cols_ if c in H1.columns]]
REGISTER.to_csv(TAB / "table43_hypothesis_register.csv", index=False)
summary_ = REGISTER.groupby("family").agg(
    contrasts=("estimate", "size"),
    holm_significant=("significant_holm", "sum"),
    positive=("verdict", lambda v: int(v.str.startswith("positive").sum())),
    negative=("verdict", lambda v: int(v.str.startswith("negative").sum())),
    equivalent=("verdict", lambda v: int(v.str.startswith("equivalent").sum())),
    inconclusive=("verdict", lambda v: int((v == "inconclusive").sum())),
    median_effect=("estimate", "median"),
)
REG_SUMMARY = save_tab(
    summary_.round(4),
    "table43_hypothesis_register_summary",
    f"{len(REGISTER)} pre-specified contrasts in {REGISTER.family.nunique()} families "
    "(full register: table43_hypothesis_register.csv). Positive/negative "
    "= Holm-significant difference (X - Y > 0 / < 0); "
    f"equivalent = Holm-adjusted TOST within ±{CFG.equivalence_margin} AUROC.",
)

In [ ]:
# 8.1  ROC and precision-recall curves with pointwise 95% cluster-bootstrap bands (seed-averaged predictions, ERM
#      natural)
FPR_GRID = np.linspace(0, 1, 101)
REC_GRID = np.linspace(0, 1, 101)


def curve_bands(y, s, T, n_rep: int = 200):
    """Weighted ROC/PR curves for the first n_rep shared bootstrap replicates -> (point ROC, ROC band, point PR, PR
    band)."""
    ix = ScoreIndex(y, s)
    out_roc, out_pr = [], []
    W = np.vstack([np.ones((1, len(y)), np.float32), BOOT.W[T][:n_rep].astype(np.float32)])
    pos, neg = ix.group_sums(W)
    P_ = pos.sum(1, keepdims=True)
    N_ = neg.sum(1, keepdims=True)
    tp = P_ - (np.cumsum(pos, 1) - pos)
    fp = N_ - (np.cumsum(neg, 1) - neg)  # predicted positive at each group threshold (descending score)
    for b_ in range(len(W)):
        fpr = np.r_[0, (fp[b_] / N_[b_])[::-1], 1]
        tpr = np.r_[0, (tp[b_] / P_[b_])[::-1], 1]
        out_roc.append(np.interp(FPR_GRID, fpr, tpr))
        rec = (tp[b_] / P_[b_])[::-1]
        prec = (tp[b_] / np.maximum(tp[b_] + fp[b_], 1e-12))[::-1]
        o = np.argsort(rec, kind="mergesort")
        out_pr.append(np.interp(REC_GRID, rec[o], np.maximum.accumulate(prec[o][::-1])[::-1]))
    R_, Q_ = np.array(out_roc), np.array(out_pr)
    return R_[0], np.percentile(R_[1:], [2.5, 97.5], 0), Q_[0], np.percentile(Q_[1:], [2.5, 97.5], 0)


with timed("8 ROC/PR figures"):
    for ts, T in [("A+B", "C"), ("A", "B"), ("B", "A")]:
        js = [j for j in model_findings(ts, T)]
        nc = 3
        nr = int(np.ceil(len(js) / nc))
        for kind in ["ROC", "PR"]:
            fig, axes = plt.subplots(nr, nc, figsize=(COL2, 2.25 * nr), squeeze=False)
            for a_i, j in enumerate(js):
                ax = axes.ravel()[a_i]
                y = YT[T][:, j]
                for a in ARCHS:
                    k = (a, ts, NAT)
                    if k not in KEYS:
                        continue
                    s_ = np.mean([PS[(*k, s)].test[T][:, j] for s in SEEDS_OF[k]], 0)
                    r_pt, r_b, q_pt, q_b = curve_bands(y, s_, T)
                    if kind == "ROC":
                        ax.plot(FPR_GRID, r_pt, color=ARCH_COL[a], lw=0.9, label=f"{a} ({auroc(y, s_):.3f})")
                        ax.fill_between(FPR_GRID, *r_b, color=ARCH_COL[a], alpha=0.15, lw=0)
                    else:
                        ax.plot(
                            REC_GRID,
                            q_pt,
                            color=ARCH_COL[a],
                            lw=0.9,
                            label=f"{a} ({average_precision_score(y, s_):.3f})",
                        )
                        ax.fill_between(REC_GRID, *q_b, color=ARCH_COL[a], alpha=0.15, lw=0)
                if kind == "ROC":
                    ax.plot([0, 1], [0, 1], "k:", lw=0.5)
                    ax.set(xlabel="1 − specificity", ylabel="Sensitivity")
                else:
                    ax.axhline(y.mean(), color="k", ls=":", lw=0.5)
                    ax.set(xlabel="Recall", ylabel="Precision")
                ax.set_title(f"{LABELS[j]} (n+ = {int(y.sum())})", fontsize=7)
                ax.legend(fontsize=5, loc="lower right" if kind == "ROC" else "upper right")
            for ax in axes.ravel()[len(js) :]:
                ax.axis("off")
            fig.suptitle(
                f"{kind} curves, {dname(ts, T)} ({SHORT[T]} unseen), ERM (natural); bands = 95% cluster-bootstrap",
                fontsize=7,
            )
            fig.tight_layout()
            save_fig(
                fig,
                f"fig19_{kind.lower()}_{tsp(ts)}_to_{T}",
                f"{kind} curves per finding for {dname(ts, T)} with pointwise 95% bootstrap bands (AUC/AP in legend).",
            )

In [ ]:
# 8.2  Confusion matrices at the SOURCE-selected operating point (counts summed over seeds; rows normalised by the true
#      class)
with timed("8 confusion matrices"):
    for ts, T in [("A+B", "C"), ("A", "B")]:
        js = model_findings(ts, T)
        archs_ = [a for a in ARCH_ALL if (a, ts, NAT) in KEYS]
        fig, axes = plt.subplots(len(archs_), len(js), figsize=(COL2, 1.25 * len(archs_) + 0.4), squeeze=False)
        crow = []
        for r_, a in enumerate(archs_):
            k = (a, ts, NAT)
            for c_, j in enumerate(js):
                cm = np.zeros((2, 2), int)
                for s in SEEDS_OF[k]:
                    thr_j = THR[(*k, s)][0][j]
                    thr_j = thr_j if np.isfinite(thr_j) else 0.5
                    cm += confusion_matrix(YT[T][:, j], (PS[(*k, s)].test[T][:, j] >= thr_j).astype(int), labels=[0, 1])
                cmn = cm / cm.sum(1, keepdims=True)
                ax = axes[r_, c_]
                sns.heatmap(
                    cmn,
                    ax=ax,
                    vmin=0,
                    vmax=1,
                    cmap="Blues",
                    cbar=False,
                    annot=np.array([[f"{cmn[i, q]:.2f}\n({cm[i, q]})" for q in range(2)] for i in range(2)]),
                    fmt="",
                    annot_kws=dict(fontsize=4.8),
                    square=True,
                    linewidths=0.3,
                )
                ax.set_xticklabels(["pred −", "pred +"], fontsize=5)
                ax.set_yticklabels(["true −", "true +"], fontsize=5, rotation=0)
                if r_ == 0:
                    ax.set_title(LABELS[j], fontsize=6.5)
                if c_ == 0:
                    ax.set_ylabel(a, fontsize=6.5)
                tn, fp, fn, tp = cm.ravel()
                crow.append(
                    dict(
                        arch=a,
                        direction=dname(ts, T),
                        finding=LABELS[j],
                        TP=tp,
                        FP=fp,
                        FN=fn,
                        TN=tn,
                        sensitivity=tp / max(tp + fn, 1),
                        specificity=tn / max(tn + fp, 1),
                        PPV=tp / max(tp + fp, 1),
                    )
                )
        fig.tight_layout()
        save_fig(
            fig,
            f"fig20_confusion_{tsp(ts)}_to_{T}",
            f"Confusion matrices for {dname(ts, T)} at source-selected max-F1 "
            "thresholds (row-normalised; counts summed over seeds).",
        )
        save_tab(
            pd.DataFrame(crow),
            f"table53_confusion_{tsp(ts)}_to_{T}",
            f"Confusion-matrix counts and derived rates for {dname(ts, T)}.",
            index=False,
            show=False,
        )

In [ ]:
# 8.3  Calibration diagnostics: reliability curves (equal-mass bins, bootstrap bands), binned residual plots (Gelman &
#      Hill: mean residual y - p per bin with +/- 2 SE), and calibration intercept/slope (Van Calster et al. 2019) for
#      every unseen-site direction.
def reliability_bins(y, p, n_bins: int = 15):
    q = np.unique(np.quantile(p, np.linspace(0, 1, n_bins + 1)))
    b = np.clip(np.searchsorted(q, p, side="right") - 1, 0, len(q) - 2)
    n = np.bincount(b, minlength=len(q) - 1)
    mp = np.bincount(b, weights=p, minlength=len(q) - 1) / np.maximum(n, 1)
    my = np.bincount(b, weights=y, minlength=len(q) - 1) / np.maximum(n, 1)
    return mp[n > 0], my[n > 0], n[n > 0]


with timed("8 calibration figures"):
    cases = [("A", "B"), ("B", "A"), ("A+B", "C")]
    fig, axes = plt.subplots(2, 3, figsize=(COL2, 4.6))
    for c_, (ts, T) in enumerate(cases):
        for a in ARCHS:  # deep models; the classical baseline's calibration is in Table 54
            k = (a, ts, NAT)
            if k not in KEYS:
                continue
            js = model_findings(ts, T)
            P_ = np.mean([PS[(*k, s)].test[T] for s in SEEDS_OF[k]], 0)[:, js].ravel()
            Y_ = YT[T][:, js].ravel().astype(float)
            mp, my, n = reliability_bins(Y_, P_)
            axes[0, c_].plot(mp, np.maximum(my, 1e-4), marker="o", ms=2.5, color=ARCH_COL[a], label=a)
            res = my - mp
            se = np.sqrt(np.maximum(mp * (1 - mp), 1e-9) / n)
            axes[1, c_].plot(mp, res, marker="o", ms=2.5, color=ARCH_COL[a])
            axes[1, c_].fill_between(mp, -2 * se, 2 * se, color=ARCH_COL[a], alpha=0.08, lw=0)
        axes[0, c_].plot(np.logspace(-4, 0, 50), np.logspace(-4, 0, 50), "k:", lw=0.6)
        axes[0, c_].set_xlim(left=1e-3)
        axes[1, c_].set_xlim(left=1e-3)
        axes[0, c_].set(
            xscale="log",
            yscale="log",
            title=f"{dname(ts, T)}",
            xlabel="Mean predicted probability",
            ylabel="Observed frequency" if c_ == 0 else "",
        )
        axes[1, c_].axhline(0, color="k", lw=0.6)
        axes[1, c_].set(
            xscale="log", xlabel="Mean predicted probability", ylabel="Binned residual (y − p)" if c_ == 0 else ""
        )
    axes[0, 0].legend(fontsize=6)
    panel_label(axes[0, 0], "a")
    panel_label(axes[1, 0], "b")
    fig.tight_layout()
    save_fig(
        fig,
        "fig21_reliability_residuals",
        "(a) Reliability curves (15 equal-mass bins, all findings pooled, log "
        "axes) and (b) binned residual plots with ±2 SE bands.",
    )
    cal = (
        MS[~MS.seen & (MS.method == NAT)]
        .groupby(["arch", "train", "test"])[["cal_intercept", "cal_slope", "ece", "brier"]]
        .agg(["mean", "std"])
    )
    save_tab(
        cal.round(4),
        "table54_calibration_external",
        "Calibration intercept (calibration-in-the-large; 0 = ideal), slope "
        "(1 = ideal), ECE and Brier score on unseen sites.",
    )
    cc = (
        MS[~MS.seen & (MS.method == NAT)]
        .groupby(["arch", "train", "test"])[["cal_intercept", "cal_slope"]]
        .mean()
        .reset_index()
    )
    fig, ax = plt.subplots(figsize=(COL1 * 1.3, 2.9))
    cc = cc[cc.arch.isin(ARCHS)]  # classical baseline: Table 54
    for (a, T), g in cc.groupby(["arch", "test"]):
        ax.scatter(
            g.cal_intercept,
            g.cal_slope,
            s=20,
            marker=ARCH_MK.get(a, "o"),
            color=SITE_COL[T],
            edgecolor="k",
            lw=0.3,
        )
    handles_ = [
        mpl.lines.Line2D([], [], ls="", marker=ARCH_MK[a], color="0.6", markeredgecolor="k", label=a) for a in ARCHS
    ] + [mpl.patches.Patch(color=SITE_COL[T], label=f"test {SHORT[T]}") for T in "ABC"]
    ax.axvline(0, color="k", lw=0.5)
    ax.axhline(1, color="k", lw=0.5)
    ax.set(xlabel="Calibration intercept", ylabel="Calibration slope")
    ax.legend(handles=handles_, fontsize=5.5, ncol=2)
    fig.tight_layout()
    save_fig(
        fig,
        "fig22_calibration_intercept_slope",
        "Calibration intercept vs slope of the deep models for every unseen-site direction "
        "(marker = architecture, colour = test site; ideal = (0, 1)).",
    )

In [ ]:
# 8.4  Error distributions: per-image probability error by direction (ECDF), per-architecture error on one unseen site,
#      the bootstrap distributions of the headline effects, and seed-to-seed deviations of the unseen-site AUROC.
with timed("8 error-distribution figures"):
    fig, ax = plt.subplots(2, 2, figsize=(COL2, 4.8))
    a0 = ARCHS[0]
    for ts, T in EXT_DIRS:
        if (a0, ts, T) in ERR:
            e = np.sort(np.sqrt(ERR[(a0, ts, T)]))
            ax[0, 0].plot(e, np.arange(1, len(e) + 1) / len(e), lw=0.8, label=dname(ts, T))
    ax[0, 0].set(
        xscale="log",
        xlabel="Per-image RMS probability error",
        ylabel="Cumulative share of images",
        title=f"{a0}, unseen sites",
    )
    ax[0, 0].legend(fontsize=5, ncol=2)
    panel_label(ax[0, 0], "a")
    vd = [
        pd.DataFrame(dict(arch=a, log10_error=np.log10(np.sqrt(v) + 1e-6)))
        for (a, ts, T), v in ERR.items()
        if (ts, T) == ("A+B", "C")
    ]
    if vd:
        dfv = pd.concat(vd)
        sns.violinplot(
            data=dfv, x="arch", y="log10_error", ax=ax[0, 1], color="0.8", cut=0, inner="quartile", linewidth=0.5
        )
        ax[0, 1].set(xlabel="", ylabel="log10 per-image RMS error", title="A+B→C (VinBigData unseen)")
        ax[0, 1].tick_params(axis="x", rotation=20, labelsize=6)
        plt.setp(ax[0, 1].get_xticklabels(), ha="right")
    panel_label(ax[0, 1], "b")
    dd = []
    for a, g in GAP_DRAWS.items():
        dd.append(pd.DataFrame(dict(effect=f"asymmetry | {a}", draw=g["asym"])))
    for a in ARCHS:
        for m_ in ["DANN", "CORAL", "GroupDRO"]:
            k1, k0 = ((a, "A+B", m_), "C"), ((a, "A+B", NAT), "C")
            if k1 in BR and k0 in BR and m_ == "DANN":
                dd.append(
                    pd.DataFrame(
                        dict(effect=f"DANN − ERM | {a} | A+B→C", draw=macro_draws(BR[k1]) - macro_draws(BR[k0]))
                    )
                )
    if dd:
        dfd = pd.concat(dd)
        sns.violinplot(data=dfd, y="effect", x="draw", ax=ax[1, 0], color="0.8", cut=0, inner="quartile", linewidth=0.5)
        ax[1, 0].axvline(0, color="r", lw=0.6, ls="--")
        ax[1, 0].set(xlabel="Bootstrap distribution of Δ AUROC", ylabel="")
        ax[1, 0].tick_params(axis="y", labelsize=5.5)
    panel_label(ax[1, 0], "c")
    sd_ = MS[~MS.seen & (MS.method == NAT)].copy()
    sd_["dev"] = sd_.auroc - sd_.groupby(["arch", "train", "test"]).auroc.transform("mean")
    sns.stripplot(data=sd_, x="arch", y="dev", ax=ax[1, 1], color="0.3", size=2.5, jitter=0.25)
    ax[1, 1].axhline(0, color="k", lw=0.5)
    ax[1, 1].set(xlabel="", ylabel="Seed deviation (Δ AUROC)")
    ax[1, 1].tick_params(axis="x", rotation=20, labelsize=6)
    plt.setp(ax[1, 1].get_xticklabels(), ha="right")
    panel_label(ax[1, 1], "d")
    fig.tight_layout()
    save_fig(
        fig,
        "fig23_error_distributions",
        "(a) ECDF of per-image error on unseen sites; (b) per-image error by "
        "architecture (A+B→C); (c) bootstrap distributions "
        "of headline effects; (d) seed-to-seed deviations of unseen-site AUROC.",
    )

In [ ]:
# 8.5  Feature-importance rankings in one figure: what identifies the hospital, what the shallow disease model relies
#      on, which hyperparameters matter, and which design factors explain unseen-site AUROC.
with timed("8 importance figure"):
    fig, ax = plt.subplots(2, 2, figsize=(COL2, 4.6))
    g = IMPS[IMPS.kind == "group"].sort_values("mean")
    ax[0, 0].barh(
        g.name,
        g["mean"],
        xerr=[g["mean"] - g.ci_low, g.ci_high - g["mean"]],
        color="0.5",
        error_kw=dict(lw=0.6, capsize=2),
    )
    ax[0, 0].set(xlabel="Drop in balanced accuracy (site probe)", title="Acquisition descriptors → hospital")
    panel_label(ax[0, 0], "a")
    if len(CL_IMP):
        ci_ = CL_IMP.groupby(["group", "train"]).importance.mean().unstack("train")
        ci_.sort_values(ci_.columns[0]).plot.barh(
            ax=ax[0, 1], color=[SITE_COL.get(c, "0.5") for c in ci_.columns], width=0.75
        )
        ax[0, 1].set(
            xlabel="Drop in macro AUROC (held-out folds)",
            ylabel="",
            title="Classical disease model: grouped permutation",
        )
        ax[0, 1].legend(title="trained on", fontsize=6)
    panel_label(ax[0, 1], "b")
    if "RFI" in globals() and len(RFI):
        r_ = RFI.sort_values("importance_mean")
        ax[1, 0].barh(
            r_.feature, r_.importance_mean, xerr=r_.importance_sd, color="0.5", error_kw=dict(lw=0.6, capsize=2)
        )
        ax[1, 0].set(xlabel="Permutation importance (Δ R²)", title="Hyperparameters → unseen AUROC (Stage 3)")
    else:
        ax[1, 0].axis("off")
    panel_label(ax[1, 0], "c")
    if ANOVA:
        nm, t = next(iter(ANOVA.items()))
        e = t.drop(index="Residual")["partial_eta2"].sort_values()
        ax[1, 1].barh([i.replace("C(", "").replace(")", "") for i in e.index], e.values, color="0.5")
        ax[1, 1].set(xlabel="partial η²", title=f"Design factors ({nm.split(' (')[0]})")
        ax[1, 1].tick_params(axis="y", labelsize=6)
    else:
        ax[1, 1].axis("off")
    panel_label(ax[1, 1], "d")
    fig.tight_layout()
    save_fig(
        fig,
        "fig24_importance_rankings",
        "Feature-importance rankings: (a) acquisition-descriptor groups for site identification (corrected 95% CI); "
        "(b) grouped permutation importance of the classical disease model; "
        "(c) Stage 3 hyperparameters; (d) design factors (partial eta^2).",
    )

In [ ]:
# 8.6  (GPU) Representations and saliency. Runs only with PyTorch and the archived checkpoints (CFG.run_gpu_analyses).
#   (a) site probe on penultimate embeddings of the A+B models (repeated grouped K-fold CV) + t-SNE;
#   (b) Grad-CAM audit on CheXpert positives of the NIH-trained models: share of saliency in the outer 10% border for
#       correct vs missed cases, agreement between architectures, and a weight-randomisation sanity check (Adebayo et
#       al. 2018). Grad-CAM is descriptive;
#   (c) border-masking sensitivity: AUROC on the unseen hospital when the outer 5/10/15% of the image is replaced by
#   the image median.
def find_ckpt(arch: str, ts: str, s: int = 0):
    cands = {
        ("DenseNet121", "A+B"): [P["dg"] / f"erm_nat_e{ARCHIVE_EPOCHS}_seed{s}.pt"],
        ("ConvNeXt-T", "A+B"): [P["c_source"] / f"convnext_ApB_erm_nat_e{ARCHIVE_EPOCHS}_seed{s}.pt"],
        ("ViT-S", "A+B"): [P["vit"] / f"vit_ApB_seed{s}.pt"],
        ("DenseNet121", "A"): [P["nih_train"] / f"densenet121_nih_e{ARCHIVE_EPOCHS}_seed{s}.pt"],
        ("ConvNeXt-T", "A"): [P["part34"] / f"convnext_tiny_nih_e{ARCHIVE_EPOCHS}_seed{s}.pt"],
        ("ViT-S", "A"): [P["vit"] / f"vit_A_seed{s}.pt"],
        ("DenseNet121", "B"): [R / f"densenet121_e{ARCHIVE_EPOCHS}_seed{s}.pt"],
        ("ConvNeXt-T", "B"): [P["part34"] / f"convnext_tiny_cx_e{ARCHIVE_EPOCHS}_seed{s}.pt"],
        ("ViT-S", "B"): [P["vit"] / f"vit_B_seed{s}.pt"],
    }.get((arch, ts), [])
    cands = cands + [Path(str(TrainSpec(arch, ts, NAT, s).stem()) + ".pt")]
    if arch == "ConvNeXt-T" and ts == "A+B":
        cands += sorted(R.glob(f"**/convnext*ApB*erm_nat*seed{s}.pt"))
    return next((c for c in cands if Path(c).exists()), None)


if HAS_TORCH:

    class ViTGradCAM:
        """Grad-CAM on the patch tokens entering the last ViT block (norm1 output), class token excluded."""

        def __init__(self, m):
            self.m = m
            self.np_ = m.num_prefix_tokens
            self.h = m.blocks[-1].norm1.register_forward_hook(self._fwd)

        def _fwd(self, mod, i, out):
            self.act = out
            if out.requires_grad:
                out.register_hook(lambda g: setattr(self, "grad", g))

        def __call__(self, x, j: int):
            self.m.zero_grad(set_to_none=True)
            with torch.enable_grad():
                self.m(x)[:, j].sum().backward()
            a_, g_ = self.act[:, self.np_ :].detach(), self.grad[:, self.np_ :].detach()
            cam = torch.relu((g_.mean(1, keepdim=True) * a_).sum(-1))
            n_ = int(cam.shape[1] ** 0.5)
            return (
                F.interpolate(cam.view(-1, 1, n_, n_), size=CFG.image_size, mode="bilinear", align_corners=False)[:, 0]
                .cpu()
                .numpy()
            )

    def gradcam(m, arch, x, j: int):
        if arch == "ViT-S":
            cam = ViTGradCAM(m)
            out = cam(x, j)
            cam.h.remove()
            return out
        with torch.enable_grad():
            fm = m.forward_features(x)
            out = m.forward_head(fm)
            g_ = torch.autograd.grad(out[:, j].sum(), fm)[0]
        cam = torch.relu((g_.mean((2, 3), keepdim=True) * fm).sum(1, keepdim=True))
        return (
            F.interpolate(cam, size=CFG.image_size, mode="bilinear", align_corners=False)[:, 0].detach().cpu().numpy()
        )

    def border_share(c, frac: float = 0.10):
        k = int(c.shape[-1] * frac)
        mk = np.ones(c.shape[-2:], bool)
        mk[k:-k, k:-k] = False
        return c[:, mk].sum(1) / (c.reshape(len(c), -1).sum(1) + 1e-8)


with timed("8 GPU interpretability"):
    GPU_OK = HAS_TORCH and CFG.run_gpu_analyses
    if not GPU_OK:
        print("skipped: PyTorch missing or CFG.run_gpu_analyses = False")
    else:
        # images per analysis
        N_EMB, N_CAM, N_MASK = {"full": (600, 150, None), "fast": (300, 60, 1000)}.get(PROFILE, (60, 12, 64))
        n_emb = min(N_EMB, *[len(YT[T]) for T in "ABC"])
        EMBP = {}
        sel_e = {T: np.sort(rng_for("embed", T).choice(len(YT[T]), n_emb, replace=False)) for T in "ABC"}
        ysite_e = np.repeat(np.arange(3), n_emb)
        gsite_e = np.concatenate([[f"{T}{g}" for g in GID[T][sel_e[T]]] for T in "ABC"])
        EMB = {}
        for a in ARCHS:
            ck = find_ckpt(a, "A+B")
            if ck is None:
                print("no A+B checkpoint for", a)
                continue
            m = load_checkpoint(a, ck)
            EMB[a] = np.concatenate([embed(m, TEST[T][0], TEST[T][1][sel_e[T]], a) for T in "ABC"])
            del m
            sc_ = [
                balanced_accuracy_score(
                    ysite_e[b],
                    make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
                    .fit(EMB[a][a_], ysite_e[a_])
                    .predict(EMB[a][b]),
                )
                for _, _, a_, b in repeated_group_kfold(ysite_e, gsite_e, CFG.cv_k, CFG.cv_r, "embed-probe")
            ]
            EMBP[a] = corrected_t_ci(sc_, n_emb * 3 * (CFG.cv_k - 1) / CFG.cv_k, n_emb * 3 / CFG.cv_k)
        if EMBP:
            save_tab(
                pd.DataFrame(EMBP).T.round(4),
                "table55_embedding_site_probe",
                "Site identification from penultimate embeddings of the A+B models "
                "(balanced accuracy, repeated grouped CV, corrected 95% CI; chance = 0.333).",
            )
            from sklearn.manifold import TSNE

            fig, axes = plt.subplots(1, len(EMB), figsize=(COL2, 2.5), squeeze=False)
            for ax, (a, E_) in zip(axes[0], EMB.items()):
                Z2 = TSNE(2, perplexity=min(30, max(5, n_emb // 4)), init="pca", random_state=0).fit_transform(
                    PCA(min(50, E_.shape[1], len(E_) - 1), random_state=0).fit_transform(
                        StandardScaler().fit_transform(E_)
                    )
                )
                for i, T in enumerate("ABC"):
                    ax.scatter(*Z2[ysite_e == i].T, s=2, alpha=0.6, color=SITE_COL[T], label=SHORT[T])
                ax.set(title=f"{a}: probe {EMBP[a]['mean']:.2f}", xticks=[], yticks=[])
            axes[0, 0].legend(markerscale=3, fontsize=6)
            fig.tight_layout()
            save_fig(
                fig,
                "fig25_embedding_tsne",
                "t-SNE of penultimate embeddings of the A+B models, coloured by hospital (probe = balanced accuracy).",
            )
        # (b) Grad-CAM audit, NIH-trained models on CheXpert positives
        j_lab = LABELS.index("Pleural Effusion")
        CAMS, BS_rows, n_cam = {}, [], min(N_CAM, int(YT["B"][:, j_lab].sum()))
        for a in ARCHS:
            ck = find_ckpt(a, "A")
            if ck is None:
                continue
            m = load_checkpoint(a, ck)
            k = (a, "A", NAT)
            p_ = np.mean([PS[(*k, s)].test["B"][:, j_lab] for s in SEEDS_OF[k]], 0)
            thr_ = np.nanmean([THR[(*k, s)][0][j_lab] for s in SEEDS_OF[k]])
            pos = np.flatnonzero(YT["B"][:, j_lab] == 1)
            for grp, ids in [("correct (TP)", pos[p_[pos] >= thr_]), ("missed (FN)", pos[p_[pos] < thr_])]:
                ids = (
                    np.sort(rng_for("cam", a, grp).choice(ids, min(n_cam, len(ids)), replace=False))
                    if len(ids)
                    else ids
                )
                if len(ids) == 0:
                    continue
                cams = np.concatenate(
                    [gradcam(m, a, to_tensor(CX_T, ids[i : i + 25], a), j_lab) for i in range(0, len(ids), 25)]
                )
                CAMS[(a, grp)] = (ids, cams / (cams.reshape(len(cams), -1).max(1)[:, None, None] + 1e-8))
                BS_rows += [dict(arch=a, group=grp, border_share=v) for v in border_share(cams)]
            mr = make_model(a, False).eval()
            ids0 = CAMS.get((a, "correct (TP)"), (pos[:25], None))[0][:25]  # sanity check: random weights
            c_tr = gradcam(m, a, to_tensor(CX_T, ids0, a), j_lab)
            c_rd = gradcam(mr, a, to_tensor(CX_T, ids0, a), j_lab)
            BS_rows += [
                dict(
                    arch=a,
                    group="sanity: trained vs random-weight CAM (Spearman)",
                    border_share=stats.spearmanr(x.ravel(), y.ravel())[0],
                )
                for x, y in zip(c_tr, c_rd)
            ]
            del m, mr
        if BS_rows:
            BSH = pd.DataFrame(BS_rows)
            cam_tab = BSH.groupby(["arch", "group"]).border_share.agg(["mean", "std", "count"]).round(3)
            save_tab(
                cam_tab,
                "table56_gradcam_border_share",
                "Grad-CAM saliency share in the outer 10% border (uniform map = 0.36) for correct and missed CheXpert "
                "Pleural Effusion positives of NIH-trained models; sanity rows "
                "= Spearman similarity between trained and random-weight maps (should be low).",
            )
            tests_ = []
            for a in ARCHS:
                x_ = BSH[(BSH.arch == a) & (BSH.group == "correct (TP)")].border_share
                y_ = BSH[(BSH.arch == a) & (BSH.group == "missed (FN)")].border_share
                if len(x_) > 2 and len(y_) > 2:
                    tests_.append(
                        dict(
                            arch=a,
                            test="TP vs FN (Mann-Whitney)",
                            p=stats.mannwhitneyu(x_, y_).pvalue,
                            cliffs_delta=cliffs_delta(y_, x_),
                        )
                    )
            gk = {
                a: BSH[(BSH.arch == a) & (BSH.group == "correct (TP)")].border_share.values
                for a in ARCHS
                if ((BSH.arch == a) & (BSH.group == "correct (TP)")).sum() > 2
            }
            if len(gk) >= 2:
                tests_.append(
                    dict(
                        arch="all",
                        test="architectures, TP maps (Kruskal-Wallis)",
                        **{k_: v for k_, v in kruskal_effect(gk).items() if k_ in ("p", "epsilon2")},
                    )
                )
            if tests_:
                save_tab(
                    pd.DataFrame(tests_),
                    "table57_gradcam_tests",
                    "Grad-CAM border-share tests (Holm not applied: descriptive).",
                    index=False,
                )
        # (c) border masking
        mrows = []
        for a in ARCHS:
            for src, T in [("A", "B"), ("B", "A")]:
                ck = find_ckpt(a, src)
                if ck is None:
                    continue
                m = load_checkpoint(a, ck)
                A_, idx_ = TEST[T]
                out = {}
                sub = (
                    np.arange(len(idx_))
                    if N_MASK is None or N_MASK >= len(idx_)
                    else np.sort(rng_for("mask-subset", T).choice(len(idx_), N_MASK, replace=False))
                )
                msk = None if len(sub) == len(idx_) else np.isin(np.arange(len(idx_)), sub)
                for frac in (0.0, 0.05, 0.10, 0.15):
                    preds = []
                    for i in range(0, len(sub), 128):
                        x = torch.from_numpy(np.ascontiguousarray(A_[idx_[sub[i : i + 128]]])).to(DEV).float()
                        mg = int(CFG.image_size * frac)
                        if mg:
                            fill = x.flatten(1).median(1).values.view(-1, 1, 1)
                            x[:, :mg] = fill
                            x[:, -mg:] = fill
                            x[:, :, :mg] = fill
                            x[:, :, -mg:] = fill
                        mu, sd = arch_norm(a)
                        x = (x.div(255).unsqueeze(1).repeat(1, 3, 1, 1) - mu) / sd
                        with torch.inference_mode(), autocast():
                            preds.append(torch.sigmoid(m(x).float()).cpu().numpy())
                    full_ = np.zeros((len(idx_), L), np.float32)
                    full_[sub] = np.concatenate(preds)  # rows outside `sub` get weight 0
                    out[frac] = bootstrap_cell(
                        ("MASK", a, src, T, frac),
                        T,
                        {0: full_},
                        metrics=("auroc",),
                        mask=msk,
                        allowed=model_findings(src, T),
                    )
                for frac in (0.05, 0.10, 0.15):
                    mrows.append(
                        dict(
                            arch=a,
                            direction=dname(src, T),
                            masked=f"{int(frac * 100)}%",
                            **contrast(out[frac], out[0.0]),
                        )
                    )
                del m
        if mrows:
            MASKT = pd.DataFrame(mrows)
            save_tab(
                MASKT[["arch", "direction", "masked", "estimate", "ci_low", "ci_high", "p"]].round(4),
                "table58_border_masking",
                "Change in external AUROC when the image border is masked (seed-0 checkpoints; paired cluster bootstrap"
                + ("" if N_MASK is None else f"; random subset of {N_MASK} test images per site")
                + ").",
                index=False,
            )
        if DEV == "cuda":
            torch.cuda.empty_cache()

In [ ]:
# 8.6b  Make find_ckpt aware of A+C and B+C (and any other training set) + diagnostic
if "_find_ckpt_orig" not in globals():
    _find_ckpt_orig = find_ckpt  # keep the original once (safe to re-run this cell)

# Optional: set exact paths by hand if you know them, e.g.
# MANUAL_CKPT = {("DenseNet121", "A+C"): Path("G:/runs/.../densenet121_ApC_erm_nat_e7_seed0.pt"), ...}
MANUAL_CKPT = {}

_ARCH_KEYS = {"DenseNet121": ("densenet", "dn_"), "ConvNeXt-T": ("convnext", "cnx"), "ViT-S": ("vit",)}
_EXCLUDE = ("last", "dann", "coral", "gdro", "bal", "selftest", "optim")  # ERM (natural) best-epoch weights only


def find_ckpt(arch: str, ts: str, s: int = 0):
    if (arch, ts) in MANUAL_CKPT and Path(MANUAL_CKPT[(arch, ts)]).exists():
        return Path(MANUAL_CKPT[(arch, ts)])
    c = _find_ckpt_orig(arch, ts, s)
    if c is not None or "+" not in ts:
        return c
    tag = tsp(ts).lower()  # 'apc', 'bpc', 'apb'
    hits = []
    for f in R.rglob("*.pt"):
        n, full = f.name.lower(), str(f).lower()
        if tag in n and f"seed{s}" in n and not any(x in n for x in _EXCLUDE):
            if any(k in n or k in full for k in _ARCH_KEYS[arch]):
                hits.append(f)
    return sorted(hits)[0] if hits else None


# ---- diagnostic: what is found, and is the model in the prediction registry? ----
rows = []
for ts, T in EXT_DIRS:
    for a in ARCHS:
        ck = find_ckpt(a, ts, 0)
        rows.append(dict(direction=dname(ts, T), arch=a, checkpoint=str(ck) if ck else "NOT FOUND",
                         predictions_in_registry=(a, ts, NAT) in KEYS))
DIAG = pd.DataFrame(rows)
display(DIAG)

In [ ]:
# 8.7  Grad-CAM for ALL findings and ALL unseen-site directions that have a checkpoint (visual companion to 8.6)
#      Paste as a NEW cell after 8.6. Re-uses find_ckpt, load_checkpoint, gradcam, to_tensor, PS, THR, YT, TEST, save_fig.
#      For every direction (training set -> unseen site), every finding labelled at that site and trained on, and every
#      architecture it computes Grad-CAM for correctly detected positives (TP) and missed positives (FN) at the
#      SOURCE-selected threshold, and writes:
#        * one overlay gallery per (direction, finding): rows = architecture x {TP, FN}, columns = n_show examples,
#        * one mean-CAM figure per direction: rows = findings, columns = architecture x {TP, FN},
#        * table59: border share (outer 10%) of the saliency for every direction / finding / architecture / group.
#      Directions without a checkpoint (find_ckpt returns None) are skipped and reported.
from matplotlib import cm as _cm

GC_DIRECTIONS = EXT_DIRS  # all 9 unseen-site directions; e.g. [("A", "B"), ("B", "A"), ("A+B", "C")] to restrict
GC_SEED = 0
GC_N_SHOW, GC_N_MEAN = {"full": (4, 60), "fast": (4, 30)}.get(PROFILE, (3, 6))
GC_MIN_POS = 5  # a finding needs at least this many TP or FN images in a group to be drawn


def _gc_collect(ts, T):
    """Grad-CAM for every finding/architecture of one direction. Returns {(arch, finding_idx, group): dict}."""
    A_, idx_ = TEST[T]
    res = {}
    for a in ARCHS:
        ck, k = find_ckpt(a, ts, GC_SEED), (a, ts, NAT)
        if ck is None or k not in KEYS or not all(T in PS[(*k, s)].test for s in SEEDS_OF[k]):
            print(f"  skip {a} {dname(ts, T)}: no checkpoint / predictions")
            continue
        m = load_checkpoint(a, ck)  # loaded once, reused for every finding
        for j in model_findings(ts, T):
            y = YT[T][:, j]
            p = np.mean([PS[(*k, s)].test[T][:, j] for s in SEEDS_OF[k]], 0)
            thr = np.nanmean([THR[(*k, s)][0][j] for s in SEEDS_OF[k]])
            thr = thr if np.isfinite(thr) else 0.5
            pos = np.flatnonzero(y == 1)
            for g, ids in {"TP": pos[p[pos] >= thr], "FN": pos[p[pos] < thr]}.items():
                if len(ids) < GC_MIN_POS:
                    continue
                ids = np.sort(rng_for("gradcam-all", a, ts, T, j, g).choice(ids, min(GC_N_MEAN, len(ids)), replace=False))
                cams = np.concatenate(
                    [gradcam(m, a, to_tensor(A_, idx_[ids[i : i + 25]], a), j) for i in range(0, len(ids), 25)]
                )
                cams = cams / (cams.reshape(len(cams), -1).max(1)[:, None, None] + 1e-8)  # per-image max = 1
                res[(a, j, g)] = dict(
                    ids=ids[:GC_N_SHOW],
                    p=p[ids[:GC_N_SHOW]],
                    show=cams[:GC_N_SHOW].astype(np.float16),
                    mean=cams.mean(0),
                    border=border_share(cams),
                    n=len(ids),
                )
        del m
        if DEV == "cuda":
            torch.cuda.empty_cache()
    return res


def _gc_gallery(res, ts, T, j):
    A_, idx_ = TEST[T]
    archs_ = [a for a in ARCHS if any((a, j, g) in res for g in ("TP", "FN"))]
    if not archs_:
        return
    fig, axes = plt.subplots(2 * len(archs_), GC_N_SHOW, figsize=(COL2, 1.75 * 2 * len(archs_)), squeeze=False)
    for r_, a in enumerate(archs_):
        for g_i, g in enumerate(("TP", "FN")):
            for c_ in range(GC_N_SHOW):
                ax = axes[2 * r_ + g_i, c_]
                ax.axis("off")
                d = res.get((a, j, g))
                if d is None or c_ >= len(d["ids"]):
                    continue
                ax.imshow(np.asarray(A_[idx_[d["ids"][c_]]]), cmap="gray", vmin=0, vmax=255)
                ax.imshow(d["show"][c_].astype(np.float32), cmap="jet", alpha=0.45, vmin=0, vmax=1)
                ax.set_title(f"p = {d['p'][c_]:.2f}", fontsize=5.5, pad=1)
                if c_ == 0:
                    ax.text(-0.04, 0.5, f"{a}\n{g}", transform=ax.transAxes, rotation=90, ha="right", va="center", fontsize=6)
    sm = _cm.ScalarMappable(cmap="jet")
    sm.set_array([0, 1])
    fig.colorbar(sm, ax=axes, shrink=0.5, pad=0.01, label="Grad-CAM (normalised)")
    fig.suptitle(f"Grad-CAM, {dname(ts, T)}, {LABELS[j]} (TP = detected, FN = missed at the source threshold)", fontsize=7)
    save_fig(
        fig,
        f"fig26_gradcam_{tsp(ts)}_to_{T}_{LABELS[j].replace(' ', '_')}",
        f"Grad-CAM overlays of detected (TP) and missed (FN) {LABELS[j]} positives, {dname(ts, T)}.",
    )


def _gc_mean_figure(res, ts, T):
    js = sorted({j for (_, j, _) in res})
    cols = [(a, g) for a in ARCHS for g in ("TP", "FN") if any((a, j, g) in res for j in js)]
    if not js or not cols:
        return
    fig, axes = plt.subplots(len(js), len(cols), figsize=(COL2, 1.25 * len(js) + 0.5), squeeze=False)
    for r_, j in enumerate(js):
        for c_, (a, g) in enumerate(cols):
            ax = axes[r_, c_]
            ax.axis("off")
            d = res.get((a, j, g))
            if d is not None:
                im = ax.imshow(d["mean"], cmap="jet", vmin=0, vmax=1)
                ax.set_title(f"{a[:8]} {g} (n={d['n']})" if r_ == 0 else f"n={d['n']}", fontsize=5, pad=1)
            if c_ == 0:
                ax.text(-0.04, 0.5, LABELS[j], transform=ax.transAxes, rotation=90, ha="right", va="center", fontsize=5.5)
    fig.suptitle(f"Mean Grad-CAM, {dname(ts, T)} (rows = findings; TP vs FN per architecture)", fontsize=7)
    fig.tight_layout()
    save_fig(
        fig,
        f"fig27_gradcam_mean_{tsp(ts)}_to_{T}",
        f"Mean Grad-CAM maps of TP vs FN positives for every finding, {dname(ts, T)}.",
    )


with timed("8 Grad-CAM all findings"):
    GC_ROWS = []
    if HAS_TORCH and CFG.run_gpu_analyses:
        for ts, T in GC_DIRECTIONS:
            print(f"Grad-CAM {dname(ts, T)}")
            res = _gc_collect(ts, T)
            if not res:
                continue
            for j in sorted({j for (_, j, _) in res}):
                _gc_gallery(res, ts, T, j)
            _gc_mean_figure(res, ts, T)
            GC_ROWS += [
                dict(direction=dname(ts, T), arch=a, finding=LABELS[j], group=g, n=d["n"], border_share=float(d["border"].mean()))
                for (a, j, g), d in res.items()
            ]
            del res
            gc.collect()
        if GC_ROWS:
            GCT = save_tab(
                pd.DataFrame(GC_ROWS).round(3),
                "table59_gradcam_border_share_all",
                "Mean Grad-CAM saliency share in the outer 10% border (uniform map = 0.36) for detected (TP) and missed "
                "(FN) positives, every finding and unseen-site direction with a checkpoint.",
                index=False,
                show=False,
            )
            display(GCT.pivot_table(index=["direction", "finding"], columns=["arch", "group"], values="border_share"))
    else:
        print("skipped: PyTorch missing or CFG.run_gpu_analyses = False")


In [ ]:
# 8.8  Save and export every Grad-CAM output (figures, tables, raw maps) -> one folder, one zip, optional copy elsewhere
import shutil, zipfile

DEST = None  # e.g. Path("D:/my_results") to also copy the zip there; None = keep inside runs/master_q1

EXPORT = OUT / "gradcam_export"
if EXPORT.exists():
    shutil.rmtree(EXPORT)
for sub in ("figures", "tables", "raw_maps"):
    (EXPORT / sub).mkdir(parents=True)

figs = sorted(list(FIG.glob("fig26_gradcam_*")) + list(FIG.glob("fig27_gradcam_*")))
tabs = sorted(TAB.glob("table59_gradcam_border_share_all.*"))
raws = sorted(CACHE.glob("gradcam_*.npz"))
for files, sub in [(figs, "figures"), (tabs, "tables"), (raws, "raw_maps")]:
    for f in files:
        shutil.copy2(f, EXPORT / sub / f.name)

# caption index (filled during cell 8.7 in this session)
idx = pd.DataFrame([r for r in FIGURE_INDEX if r["figure"].startswith(("fig26_gradcam", "fig27_gradcam"))])
idx.to_csv(EXPORT / "figure_index.csv", index=False)

zip_path = OUT / "gradcam_export.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in EXPORT.rglob("*"):
        if f.is_file():
            zf.write(f, f.relative_to(EXPORT))

if DEST is not None:
    DEST = Path(DEST)
    DEST.mkdir(parents=True, exist_ok=True)
    shutil.copy2(zip_path, DEST / zip_path.name)
    print("copied to:", DEST / zip_path.name)

print(f"exported {len(figs)} figure files, {len(tabs)} table files, {len(raws)} raw-map files")
print("folder:", EXPORT)
print("zip   :", zip_path, f"({zip_path.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# 9.1  Automatic results summary (every number and every qualitative statement below is computed in this cell from
#      the current run) and a reproduction check against the archived runs.
# Per-seed mean macro AUROC of ERM (natural) models as printed by the archived notebooks
# (MASTER_CXR_Generalization, che44-nih77, REFER2, refer).
ARCHIVED = {
    ("DenseNet121", "A", "A"): 0.813,
    ("DenseNet121", "A", "B"): 0.713,
    ("DenseNet121", "A", "C"): 0.866,
    ("DenseNet121", "B", "B"): 0.804,
    ("DenseNet121", "B", "A"): 0.785,
    ("DenseNet121", "B", "C"): 0.852,
    ("DenseNet121", "A+B", "C"): 0.878,
    ("DenseNet121", "A+C", "B"): 0.712,
    ("DenseNet121", "B+C", "A"): 0.776,
    ("ConvNeXt-T", "A", "A"): 0.818,
    ("ConvNeXt-T", "A", "B"): 0.730,
    ("ConvNeXt-T", "A", "C"): 0.880,
    ("ConvNeXt-T", "B", "B"): 0.811,
    ("ConvNeXt-T", "B", "A"): 0.800,
    ("ConvNeXt-T", "B", "C"): 0.875,
    ("ConvNeXt-T", "A+B", "C"): 0.893,
    ("ConvNeXt-T", "A+C", "B"): 0.726,
    ("ConvNeXt-T", "B+C", "A"): 0.787,
    ("ViT-S", "A", "A"): 0.811,
    ("ViT-S", "A", "B"): 0.731,
    ("ViT-S", "A", "C"): 0.844,
    ("ViT-S", "B", "B"): 0.803,
    ("ViT-S", "B", "A"): 0.796,
    ("ViT-S", "B", "C"): 0.873,
    ("ViT-S", "A+B", "C"): 0.888,
    ("ViT-S", "A+C", "B"): 0.722,
    ("ViT-S", "B+C", "A"): 0.780,
}
REPRO_TOL = 0.0015  # archived values are printed to 3 decimals


def _ci_excludes_zero(r) -> bool:
    return bool(r.ci_low > 0 or r.ci_high < 0)


with timed("9 summary"):
    rows = []
    for (a, ts, T), v in ARCHIVED.items():
        g = MS[(MS.arch == a) & (MS.train == ts) & (MS.test == T) & (MS.method == NAT)]
        if len(g):
            rows.append(
                dict(
                    arch=a,
                    method=NAT,
                    direction=dname(ts, T),
                    seed="mean",
                    archived=v,
                    regenerated=round(g.auroc.mean(), 4),
                    difference=round(g.auroc.mean() - v, 4),
                )
            )
    for r in GDRO_DN_RUNS.itertuples():  # DenseNet121 GroupDRO A+C / B+C: per seed, values printed by that notebook
        g = MS[
            (MS.arch == "DenseNet121")
            & (MS.method == "GroupDRO")
            & (MS.train == r.train)
            & (MS.test == r.test)
            & (MS.seed == r.seed)
        ]
        if len(g) and pd.notna(getattr(r, "mean_AUROC", np.nan)):
            rows.append(
                dict(
                    arch="DenseNet121",
                    method="GroupDRO",
                    direction=dname(r.train, r.test),
                    seed=str(r.seed),
                    archived=round(float(r.mean_AUROC), 4),
                    regenerated=round(float(g.auroc.iloc[0]), 4),
                    difference=round(float(g.auroc.iloc[0]) - float(r.mean_AUROC), 4),
                )
            )
    REPRO = pd.DataFrame(rows)
    if len(REPRO):
        REPRO[f"reproduced (|diff| <= {REPRO_TOL})"] = REPRO.difference.abs() <= REPRO_TOL
        save_tab(
            REPRO,
            "table59_reproduction_check",
            "Regenerated vs archived macro AUROC (ERM natural: seed means; DenseNet121 GroupDRO A+C/B+C: per seed). "
            "A difference larger than the rounding tolerance means "
            "the prediction files read by this run are not the ones the archived notebooks evaluated.",
            index=False,
        )
    S = {}
    reg = finalize_family(HYP)

    def _row(fam, contrast_, arch=None, direction=None):
        g = reg[(reg.family == fam) & (reg.contrast == contrast_)]
        g = g[g.arch == arch] if arch else g
        g = g[g.direction == direction] if direction else g
        return g.iloc[0] if len(g) else None

    lines = [f"### Key results regenerated by this run (profile `{PROFILE}`, B = {CFG.n_boot})", ""]
    if len(REPRO):
        n_ok = int(REPRO.iloc[:, -1].sum())
        lines.append(
            f"- Reproduction check: {n_ok} of {len(REPRO)} archived transfer AUROCs reproduced within ±{REPRO_TOL} "
            f"(largest |difference| {REPRO.difference.abs().max():.4f})."
        )
    for a in ARCH_ALL:
        r = _row("F1 transfer gaps", "asymmetry (A→B gap − B→A gap)", a)
        if r is not None:
            S[f"asymmetry {a}"] = dict(
                estimate=r.estimate, ci=(r.ci_low, r.ci_high), p_holm=r.p_holm, verdict=r.verdict
            )
            lines.append(
                f"- **{a}**: NIH→CheXpert gap minus CheXpert→NIH gap = {fmt_ci(r.estimate, r.ci_low, r.ci_high)} "
                f"(Holm p = {r.p_holm:.2g}; {r.verdict})."
            )
    if SIZE is not None:
        r = SIZE[SIZE.contrast.str.startswith("size-controlled")].iloc[0]
        r2 = SIZE[SIZE.contrast.str.startswith("size effect")].iloc[0]
        S["size control"] = dict(size_effect=r2.estimate, size_controlled_asymmetry=r.estimate)
        reading = (
            "training-set size does not explain the asymmetry"
            if (not _ci_excludes_zero(r2)) and _ci_excludes_zero(r)
            else "size and site effects cannot be separated cleanly"
        )
        lines.append(
            f"- Size control (DenseNet121): size effect {fmt_ci(r2.estimate, r2.ci_low, r2.ci_high)}, size-controlled "
            f"asymmetry {fmt_ci(r.estimate, r.ci_low, r.ci_high)}: {reading}."
        )
    f2 = reg[reg.family == "F2 architecture differences"]
    if len(f2):
        ext2 = f2[~f2["seen"].fillna(False).astype(bool)] if "seen" in f2 else f2
        lines.append(
            f"- Architecture differences: {int(ext2.significant_holm.sum())} of {len(ext2)} unseen-site comparisons "
            f"are Holm-significant; median |Δ| = {ext2.estimate.abs().median():.3f} AUROC."
        )
    for fam in ["F4a DG vs ERM (natural)", "F4b DG vs ERM (balanced)"]:
        g = reg[reg.family == fam]
        if len(g):
            vc = g.verdict.value_counts().to_dict()
            S[fam] = vc
            lines.append(
                f"- {fam}: "
                + ", ".join(f"{k} {v}" for k, v in vc.items())
                + f" (of {len(g)} contrasts; equivalence margin ±{CFG.equivalence_margin} AUROC)."
            )
    if "STAGE2_VERDICT" in globals():
        S["Stage-2 verdict"] = STAGE2_VERDICT
        lines.append(
            f"- Pre-registered Stage-2 claim (cross-source ensemble beats pooled training): **{STAGE2_VERDICT}** "
            "(archived notebook: REVERSED)."
        )
    if S3 is not None and "rho_all" in globals():
        col_ = S3[S3.collapsed].groupby("method").size().to_dict()
        S["Stage 3"] = dict(spearman=rho_all, collapsed=col_)
        lines.append(
            "- Stage 3: Spearman correlation between source-validation and VinBigData AUROC over non-collapsed "
            f"trials = {rho_all:.2f}; collapsed trials (validation AUROC < 0.6) by method: {col_ if col_ else 'none'}."
        )
    if "RCS" in globals() and len(RCS):
        g = RCS[(RCS.metric == "ece") & (RCS.method == "Platt")]
        g0 = RCS[(RCS.metric == "ece") & (RCS.method == "none (source threshold)")]
        k_ref = 500 if 500 in CFG.recal_budgets else CFG.recal_budgets[-1]
        for grp in g.group.unique():
            kk_ = g[(g.group == grp) & (g.k == k_ref)]
            if len(kk_) and len(g0[g0.group == grp]):
                lines.append(
                    f"- Few-label recalibration ({grp}): ECE {g0[g0.group == grp]['mean'].iloc[0]:.3f} "
                    "without adaptation, "
                    f"{kk_['mean'].iloc[0]:.3f} with Platt scaling on {k_ref} labelled target images "
                    f"(corrected 95% CI {kk_.ci_low.iloc[0]:.3f} to {kk_.ci_high.iloc[0]:.3f})."
                )
    if "EMBP" in globals() and EMBP:
        above = [a for a, v in EMBP.items() if v["ci_low"] > 1 / 3]
        lines.append(
            "- Site probe on penultimate embeddings (balanced accuracy, chance 0.33): "
            + ", ".join(f"{a} {v['mean']:.2f}" for a, v in EMBP.items())
            + f"; above chance (corrected CI) for {len(above)} of {len(EMBP)} architectures."
        )
    acq_ = PROBE_SUM[(PROBE_SUM.model == "LR (acquisition)") & (PROBE_SUM.metric == "balanced_accuracy")]
    if len(acq_):
        lines.append(
            f"- Acquisition descriptors alone identify the hospital with balanced accuracy {acq_['mean'].iloc[0]:.3f} "
            f"(corrected 95% CI {acq_.ci_low.iloc[0]:.3f} to {acq_.ci_high.iloc[0]:.3f}; chance 0.333)."
        )
    display(Markdown("\n".join(lines)))
    write_json(
        {k: (v if not isinstance(v, tuple) else list(v)) for k, v in S.items()},
        OUT / "RESULTS_SUMMARY.json",
        default=float,
    )

In [ ]:
# 9.3  Export: figure/table index, analysis run time, environment, and one zip bundle (cache and model runs excluded)
FIGT = pd.DataFrame(FIGURE_INDEX).drop_duplicates("figure", keep="last")
TABT = pd.DataFrame(TABLE_INDEX).drop_duplicates("table", keep="last")
FIGT.to_csv(OUT / "figure_index.csv", index=False)
TABT.to_csv(OUT / "table_index.csv", index=False)
RUNTIME = pd.Series(TIMER, name="seconds").round(1).to_frame()
RUNTIME["minutes"] = (RUNTIME.seconds / 60).round(2)
save_tab(
    RUNTIME,
    "table60_analysis_runtime",
    f"Wall-clock time of every analysis block of this notebook (total {RUNTIME.seconds.sum() / 60:.1f} "
    f"min, profile = {PROFILE}).",
)
with open(LOGS / "run_record.json", "w") as fh:
    json.dump(
        dict(
            environment=ENV,
            config={k: str(v) for k, v in asdict(CFG).items()},
            n_figures=len(FIGT),
            n_tables=len(TABT),
            runtime_minutes=float(RUNTIME.seconds.sum() / 60),
        ),
        fh,
        indent=2,
    )
_tmp = CFG.runs / f"{CFG.out_name}_export"
shutil.rmtree(_tmp, ignore_errors=True)
shutil.copytree(OUT, _tmp, ignore=shutil.ignore_patterns("cache", "runs"))
bundle = shutil.make_archive(str(CFG.runs / f"{CFG.out_name}_bundle"), "zip", _tmp)
shutil.rmtree(_tmp, ignore_errors=True)
print(f"{len(FIGT)} figures, {len(TABT)} tables -> {OUT}\nbundle: {bundle} ({os.path.getsize(bundle) / 1e6:.1f} MB)")
display(FIGT)